# ML-1 V4.4 · Миграционная привлекательность муниципалитетов России

**Полнофункциональный исследовательский Colab**. Независимая загрузка реальных источников Этапа I, верификация кодов, временных срезов и административных ключей, H1–H5, три горизонта (1/3/5), четыре протокола holdout, шесть моделей, Optuna/TPE, прогнозные интервалы и классификация, 50 итоговых seeds стохастических моделей, SHAP/ALE/взаимодействия, Holm/FDR, блоковый bootstrap, проверка и публикация.

**Выход V4 строго ограничен:** `ML1_V4_RESULTS.xlsx` (сводная наука), `ML1_V4_PREDICTIONS.xlsx` (полные прогнозы), `ML1_V4_DATA_QA.xlsx` (данные/источники/гипотезы), `ML1_V4_MODEL_QA.xlsx` (Optuna/сходимость/XAI/диагностика), `ML1_V4_FIGURES.zip` (не более 10 рисунков). В рабочем каталоге присутствуют скрытые checkpoint для восстановления, но они не публикуются. GitHub: один исходный ноутбук, один научный отчёт и компактный манифест; пять Release-артефактов.

Исходная база `data/processed` и `handoff/07_v2` не меняется; исходные Parquet в релиз не входят. `NOT_FEASIBLE` не подменяется подтверждением гипотезы.

## 0. Конфигурация и правила запуска

`GITHUB_TOKEN` хранится в **Colab → Secrets** (Contents: Read and write), но не в файлах. `MODE='FULL'` заново рассчитывает все модели; `AUTO` может восстановить только верифицированный V4-checkpoint того же плана. В V4 сохранена двухступенчатая оптимизация: экономный Optuna на исторической подвыборке FIT/DEV, обязательная проверка полного DEV, полное обучение FINAL и закрытый TEST. CPU/GPU выбираются после отдельного теста, скорость около двух часов не гарантируется.

In [ ]:
# V4.4 | Установка и независимая проверка библиотек ДО научных тестов.
# Исправление: XlsxWriter обязательно указан, а Excel может быть записан через openpyxl.
import sys, subprocess, importlib, importlib.util
REQUIREMENTS = ['numpy>=1.24','pandas>=2.1','scikit-learn>=1.3','pyarrow>=17',
                'duckdb>=1.1','optuna>=4.0','catboost>=1.2','lightgbm>=4.0',
                'shap>=0.45','openpyxl>=3.1','XlsxWriter>=3.2',
                'matplotlib>=3.8','tqdm>=4.66','requests>=2.31','nbformat>=5.10']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
REQUIRED_IMPORTS = ('numpy','pandas','sklearn','pyarrow','duckdb','optuna',
                    'catboost','lightgbm','shap','openpyxl','matplotlib','tqdm',
                    'requests','nbformat')
missing = [name for name in REQUIRED_IMPORTS if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('V4.2 DEPENDENCY GATE: отсутствуют библиотеки: '+repr(missing))
# Excel engines: XlsxWriter preferred; openpyxl streaming is the supported fallback.
try:
    import xlsxwriter
    XLSX_ENGINE = 'xlsxwriter'
except (ImportError, OSError):
    import openpyxl
    XLSX_ENGINE = 'openpyxl_write_only'
from openpyxl import load_workbook
print('V4.4 DEPENDENCY GATE: PASS | XLSX engine:', XLSX_ENGINE,
      '| modules:', len(REQUIRED_IMPORTS), '| Google Drive не используется')

In [ ]:
# V4 | новый эксперимент, отдельные каталоги и независимый GitHub Release
MODE='FULL'                       # FULL / AUTO / RESUME / SMOKE / VERIFY
RECOVERY_BACKUP_ZIP=None           # резервный архив только V4; V3 не совместим
COMPUTE_PROFILE='balanced'; RESULT_DIR='/content/ml1_results_v4'; CACHE_DIR='/content/ml1_data_cache'
ENABLE_SPATIAL=True; INDICATOR_OVERRIDES={}; PUBLISH_GITHUB=True
PUBLISH_BRANCH='stage2-ml1-v4'; RELEASE_TAG_PREFIX='stage2-ml1-v4.0.0'
RETRY_PUBLISH=False; AUTO_DOWNLOAD=True
CATBOOST_DEVICE='CPU'; CATBOOST_GPU_POLICY='AUTO'; CPU_MODEL_THREADS=4
OPTUNA_FIT_MAX_ROWS=2800; OPTUNA_DEV_MAX_ROWS=700; OPTUNA_GLOBAL_BUDGET_S=4200
OPTUNA_MIN_TRIALS_PER_STUDY=1; OPTUNA_PRUNE_FAST_FIT=True
assert MODE in {'FULL','AUTO','RESUME','SMOKE','VERIFY'}
assert CATBOOST_GPU_POLICY in {'AUTO','CPU','GPU'}


## 1. Цели и критерии научной приёмки

**H1:** экономика добавляет точность относительно авторегрессии; **H2:** жильё/инфраструктура относительно экономики; **H3:** географические факторы и соседи относительно непространственного набора; **H4:** неоднородность влияния по типам муниципалитетов; **H5:** воспроизводимость выигрыша между протоколами. H1–H3 проверяются парно на идентичных TEST-строках, Holm; исследовательские анализы — FDR. Отсутствие достоверной исторической географии → `NOT_FEASIBLE`, а не фиктивная H3.

Важные инварианты: `target_year=origin_year+h`; максимальный год входного признака ≤ `information_cutoff`; нельзя подменять мультигодовый DEV однолетним без явной пометки. Закрытый TEST не участвует в выборе гиперпараметров. Финальные стохастические модели — 50 seeds только при научной и вычислительной допустимости.

## 2. Профессиональная загрузка исходных файлов, инварианты и SHA-256

In [ ]:

# ------- CELL 5 -------
# ML-1 V4 autonomous Colab research engine (embedded in notebook, no local module dependencies).
import os, sys, io, re, json, math, time, hashlib, zipfile, gzip, traceback, warnings, logging, shutil, platform, itertools
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO = 'AntonDozhdikov/Demograf_sovetnik'
GITHUB_API = f'https://api.github.com/repos/{REPO}'
INDEX_FILE = 'handoff/07_v2/dataset_index.json'
LOADER_FILE = 'handoff/07_v2/stage2_loader.py'
SOURCE_LAG_YEARS = 2  # conservative assumption, not a verified reporting date
HORIZONS = (1, 3, 5)
DEV_SEEDS = (11, 23, 37)
FINAL_SEEDS = tuple(range(1001, 1051))
PRIMARY_METRIC = 'MAE'
VERSION = 'ML-1_COLAB_V3.3'




def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

def hash_obj(o): return hashlib.sha256(json.dumps(o,ensure_ascii=False,sort_keys=True,default=str).encode()).hexdigest()

def utc(): return datetime.now(timezone.utc).isoformat(timespec='seconds')

def write_json(path, obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8');tmp.replace(path)

def token_from_colab():
    tok=os.getenv('GITHUB_TOKEN','').strip()
    if not tok:
        try:
            from google.colab import userdata
            tok=(userdata.get('GITHUB_TOKEN') or '').strip()
        except Exception: pass
    if not tok: raise PermissionError('Нет GITHUB_TOKEN в Colab Secrets / environment. Нужен доступ к PRIVATE-репозиторию.')
    return tok

def github_get(session, url, **kwargs):
    r=session.get(url,timeout=(30,180),**kwargs)
    if r.status_code in (403,404): raise PermissionError(f'GitHub {r.status_code}: доступ к {url.split("?")[0]} запрещён; это НЕ отсутствие датасета')
    r.raise_for_status();return r

def github_session(token):
    import requests
    s=requests.Session();s.headers.update({'Authorization':'Bearer '+token,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
    return s

def github_text(session, path, ref='main'):
    import base64
    o=github_get(session,f'{GITHUB_API}/contents/{path}',params={'ref':ref}).json()
    if not o.get('content'): raise ValueError('GitHub file must be base64 inline for control '+path)
    raw=base64.b64decode(o['content']);return raw,{'path':path,'git_blob_sha':o['sha'],'sha256':hashlib.sha256(raw).hexdigest()}

def read_handoff(session):
    raw,meta=github_text(session,INDEX_FILE)
    idx=json.loads(raw)
    if idx.get('repo')!=REPO or str(idx.get('version'))!='7.2':
        raise ValueError('Handoff repository/version mismatch')
    a=idx.get('assets',[])
    if len(a)!=52 or len(set(x['asset_id'] for x in a))!=len(a):
        raise ValueError('Handoff inventory mismatch; inspect altered contract before proceeding')
    target=[x for x in a if x.get('segment')=='01' and x.get('source_id')=='D01']
    if len(target)!=1 or not {'oktmo','year','migration_rate','migration','population','region'}.issubset(target[0]['columns_manifest']):
        raise ValueError('Required D01 fields missing')
    # Verify the actual source loader is unchanged in design; code is self-contained and never imports it.
    loader_bytes, loader_meta=github_text(session,LOADER_FILE)
    if b'def fetch_asset' not in loader_bytes or b'def load_index' not in loader_bytes:
        raise ValueError('stage2_loader contract changed')
    return idx,target[0],{'index':meta,'loader':loader_meta}

def _stream_response_to_path(r, dst):
    tmp=dst.with_suffix('.partial')
    with tmp.open('wb') as f:
        for block in r.iter_content(4*1024*1024):
            if block:f.write(block)
    r.close();tmp.replace(dst)

def fetch_asset(session, asset, cache):
    """Fetch exactly indexed bytes from Git blob or Release; always verify indexed SHA-256."""
    import base64, requests
    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)
    dst=cache/(hashlib.sha256(asset['asset_id'].encode()).hexdigest()[:24]+'.parquet')
    exp=asset['sha256']
    if dst.exists() and sha256_file(dst)==exp:return dst
    dst.unlink(missing_ok=True);loc=asset['locator']
    if loc['type']=='git':
        r=github_get(session,GITHUB_API+'/contents/'+loc['path'])
        obj=r.json()
        if obj.get('content'): dst.write_bytes(base64.b64decode(obj['content']))
        else:
            rr=session.get(GITHUB_API+'/contents/'+loc['path'],headers={'Accept':'application/vnd.github.raw+json'},stream=True,timeout=(30,900))
            rr.raise_for_status();_stream_response_to_path(rr,dst)
    elif loc['type']=='release':
        url=GITHUB_API+'/releases/assets/'+str(loc['asset_id'])
        rr=session.get(url,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
        if rr.status_code>=400:rr.raise_for_status()
        if 'json' in rr.headers.get('content-type','').lower():
            rr.close();j=github_get(session,url).json()
            rr=session.get(j['browser_download_url'],headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
            rr.raise_for_status()
            if 'json' in rr.headers.get('content-type','').lower():
                raise RuntimeError('GitHub Release binary fetch returned JSON; access requires refresh')
        _stream_response_to_path(rr,dst)
    else:raise ValueError('Unsupported locator '+str(loc.get('type')))
    if sha256_file(dst)!=exp:
        dst.unlink(missing_ok=True);raise ValueError('Asset SHA-256 mismatch '+asset['asset_id'])
    return dst

def oktmo_str(s, strict=True):
    if pd.api.types.is_numeric_dtype(s):raise ValueError('ОКТМО хранится числом — начальные нули могли быть потеряны')
    x=s.astype('string').str.strip()
    if strict and (x.isna().any() or (~x.str.fullmatch(r'\d+').fillna(False)).any()):raise ValueError('Некорректный/пустой ОКТМО')
    return x

def num(s):
    return pd.to_numeric(s.astype('string').str.replace('\u00a0','',regex=False).str.replace(' ','',regex=False).str.replace(',','.',regex=False),errors='coerce').astype(float)

def exact_lookup(frame,source,yearcol,valuecol,offset,merge_yearcol='information_cutoff'):
    left=frame[['territory_id',merge_yearcol]].copy();left['_year']=left[merge_yearcol]-offset;left['_ix']=np.arange(len(left))
    right=source[['territory_id',yearcol,valuecol]].rename(columns={yearcol:'_year'})
    if right.duplicated(['territory_id','_year']).any():raise ValueError('JOIN_GATE many-to-many in exact lookup '+valuecol)
    x=left.merge(right,on=['territory_id','_year'],how='left',validate='m:1',sort=False).sort_values('_ix')
    return pd.to_numeric(x[valuecol],errors='coerce').to_numpy(dtype=float,na_value=np.nan)

def gate_panel(p,h,min_regions=5,min_territories=60,min_rows=150):
    a=p[p.target_rate.notna() & p.lag0.notna() & ~p.territory_type_changed].copy()
    years=int(a.target_year.nunique());r=int(a.region_id.nunique());t=int(a.territory_id.nunique())
    qa={'horizon':h,'all_rows':len(p),'eligible':len(a),'regions':r,'territories':t,
        'target_years':years,'missing_target_pct':float(100*p.target_rate.isna().mean()),
        'target_min':int(a.target_year.min()) if len(a) else None,'target_max':int(a.target_year.max()) if len(a) else None}
    qa['status']='PASS' if len(a)>=min_rows and r>=min_regions and t>=min_territories and years>=4 else 'NOT_FEASIBLE'
    return a,qa

def numeric_design(df,features):
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder,StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn import __version__ as v
    numeric=[f for f in features if f in df.columns]
    cat=['region_id','mun_type']
    for x in cat:
        if x not in df.columns:raise ValueError('Absent categorical column '+x)
    prep=ColumnTransformer([
      ('numeric',Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True,add_indicator=True)),('scale',StandardScaler())]),numeric),
      ('category',Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='unknown')),
         ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat)
    ],sparse_threshold=0)
    return prep,numeric+cat

def models(seed=11,profile='balanced'):
    from sklearn.linear_model import Ridge,ElasticNet,HuberRegressor
    from sklearn.ensemble import HistGradientBoostingRegressor
    from catboost import CatBoostRegressor
    from lightgbm import LGBMRegressor
    settings={
      'ridge':[{'alpha':1.},{'alpha':30.}],
      'elasticnet':[{'alpha':.03,'l1_ratio':.3},{'alpha':.2,'l1_ratio':.8}],
      'huber':[{'epsilon':1.35,'alpha':.001}],
      'histgb':[{'max_iter':140,'max_leaf_nodes':15,'learning_rate':.06}, {'max_iter':240,'max_leaf_nodes':25,'learning_rate':.045}],
      'catboost':[{'iterations':250,'depth':5,'learning_rate':.045,'l2_leaf_reg':6}, {'iterations':350,'depth':6,'learning_rate':.04,'l2_leaf_reg':8}],
      'lightgbm':[{'n_estimators':220,'num_leaves':15,'learning_rate':.05,'reg_lambda':5,'colsample_bytree':.8,'subsample':.8,'subsample_freq':1}, {'n_estimators':320,'num_leaves':23,'learning_rate':.04,'reg_lambda':10,'colsample_bytree':.8,'subsample':.8,'subsample_freq':1}]
    }
    if profile=='fast':settings={k:v[:1] for k,v in settings.items()}
    return settings

def instantiate(name, params,seed):
    from sklearn.linear_model import Ridge,ElasticNet,HuberRegressor
    from sklearn.ensemble import HistGradientBoostingRegressor
    from catboost import CatBoostRegressor
    from lightgbm import LGBMRegressor
    constructors={'ridge':lambda:Ridge(**params),'elasticnet':lambda:ElasticNet(**params,max_iter=1500,random_state=seed),
        'huber':lambda:HuberRegressor(**params,max_iter=200),
        'histgb':lambda:HistGradientBoostingRegressor(**params,random_state=seed,l2_regularization=5,early_stopping=True,n_iter_no_change=20,validation_fraction=.12),
        'catboost':lambda:CatBoostRegressor(**params,random_seed=seed,loss_function='MAE',thread_count=CPU_MODEL_THREADS,verbose=False,allow_writing_files=False,task_type=CATBOOST_DEVICE),
        'lightgbm':lambda:LGBMRegressor(**params,random_state=seed,n_jobs=CPU_MODEL_THREADS,verbosity=-1,min_child_samples=25)}
    return constructors[name]()

def fit_predict(train,valid,features,name,parameters,seed=11,target='target_rate'):
    from sklearn.pipeline import Pipeline
    prep,cols=numeric_design(train,features)
    pipeline=Pipeline([('preprocess',prep),('model',instantiate(name,parameters,seed))])
    with warnings.catch_warnings(record=True) as warn:
        warnings.simplefilter('always')
        pipeline.fit(train[cols].astype({'region_id':str,'mun_type':str}),train[target].to_numpy(float))
    pred=np.asarray(pipeline.predict(valid[cols].astype({'region_id':str,'mun_type':str})),float)
    return pred,pipeline,[str(w.message)[:160] for w in warn[:4]]

def baselines(train,test):
    y=train.target_rate
    mean=train.groupby('region_id').target_rate.mean()
    return {'last_known':test.lag0.to_numpy(float),
       'trailing_MA3':test.mean3.fillna(test.lag0).to_numpy(float),
       'damped_trend':(test.lag0+.35*test.drift3.fillna(0).clip(-50,50)).to_numpy(float),
       'regional_mean':test.region_id.map(mean).fillna(y.mean()).to_numpy(float)}

def score(y,pop,yhat):
    a=np.asarray(y,float);b=np.asarray(yhat,float);w=np.asarray(pop,float)
    ok=np.isfinite(a)&np.isfinite(b)
    if not ok.all():raise ValueError('Predictions/targets have NaN or inf')
    valid=np.isfinite(w)&(w>0)
    e=np.abs(a-b)
    return {'n':len(a),'MAE':float(e.mean()),'RMSE':float(np.sqrt(((a-b)**2).mean())),
      'bias':float((b-a).mean()),'p90_abs_error':float(np.quantile(e,.9)),
      'weighted_MAE':float(np.average(e[valid],weights=w[valid])) if valid.any() else np.nan}

def paired_bootstrap(df,candidate,baseline='last_known',iterations=1000,seed=2026):
    base=df[df.model==baseline][['row_id','prediction']].rename(columns={'prediction':'baseline_prediction'})
    test=df[df.model==candidate][['row_id','region_id','target_rate','prediction']]
    if base.row_id.duplicated().any() or test.row_id.duplicated().any():raise ValueError('Must compare one model seed per key')
    joined=test.merge(base,on='row_id',how='inner',validate='1:1')
    if len(joined)!=len(test) or len(joined)!=len(base):return {'status':'FAIL','reason':'Unequal test key sets'}
    joined['delta']=np.abs(joined.target_rate-joined.baseline_prediction)-np.abs(joined.target_rate-joined.prediction)
    g=joined.groupby('region_id').delta.agg(['sum','count'])
    if len(g)<5:return {'status':'NOT_FEASIBLE','reason':'<5 region blocks'}
    rng=np.random.default_rng(seed);ix=rng.integers(len(g),size=(iterations,len(g)))
    sums=g['sum'].to_numpy();counts=g['count'].to_numpy();boot=sums[ix].sum(axis=1)/counts[ix].sum(axis=1)
    low,high=np.quantile(boot,[.025,.975]);p=(1+2*min(np.sum(boot<=0),np.sum(boot>=0)))/(iterations+1)
    return {'status':'OK','delta_MAE':float(joined.delta.mean()),'CI_low':float(low),'CI_high':float(high),
       'p_boot':float(min(1,p)),'regions':int(len(g)),'keys':len(joined)}

def holm(p):
    a=np.asarray(p,float);out=np.full(len(a),np.nan);finite=np.where(np.isfinite(a))[0]
    order=finite[np.argsort(a[finite])];last=0
    for rank,i in enumerate(order):
        last=max(last,a[i]*(len(order)-rank));out[i]=min(1,last)
    return out

def benjamini_hochberg(p):
    x=np.asarray(p,float);out=np.full(len(x),np.nan);valid=np.where(np.isfinite(x))[0]
    order=valid[np.argsort(x[valid])];last=1.
    for rank in range(len(order)-1,-1,-1):
        i=order[rank];last=min(last,float(x[i])*len(order)/(rank+1));out[i]=last
    return out

def export_excel(path,sheets):
    from openpyxl import Workbook,load_workbook
    from openpyxl.styles import PatternFill,Font,Alignment
    from openpyxl.utils import get_column_letter
    wb=Workbook();wb.remove(wb.active)
    for title,df in sheets.items():
        if df is None or len(df)==0:continue
        if len(title)>31 or title in wb.sheetnames:raise ValueError('Invalid Excel sheet name '+title)
        ws=wb.create_sheet(title);x=df.head(1_048_500).copy()
        ws.append([str(s) for s in x.columns])
        for row in x.itertuples(index=False,name=None):
            entries=[]
            for v in row:
                if isinstance(v,(np.integer,np.floating)):v=v.item()
                if isinstance(v,(dict,list,tuple,set)):v=json.dumps(list(v) if isinstance(v,set) else v,ensure_ascii=False,default=str)
                if isinstance(v,(pd.Timestamp,pd.Interval,pd.Period)):v=str(v)
                if v is None or (not isinstance(v,(dict,list,tuple)) and pd.isna(v)):v=None
                entries.append(v)
            ws.append(entries)
        ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
        for c in ws[1]:
            c.fill=PatternFill('solid',fgColor='193B55');c.font=Font(name='Calibri',bold=True,color='FFFFFF')
            c.alignment=Alignment(wrap_text=True,vertical='center')
        ws.row_dimensions[1].height=32
        for j,key in enumerate(x.columns,1):
            sample=x[key].head(150).dropna().astype(str)
            width=max([len(str(key)),*sample.str.len().tolist()]) if len(sample) else len(str(key))
            ws.column_dimensions[get_column_letter(j)].width=min(50,max(12,width+2))
    if not wb.worksheets:raise ValueError('Excel requires nonempty scientific sheets')
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);wb.save(path)
    check=load_workbook(path,read_only=True);assert check.sheetnames==wb.sheetnames;check.close()
    return {'sha256':sha256_file(path),'size':path.stat().st_size,'sheets':wb.sheetnames}


EXPORT_SERIALIZATION_AUDIT = []

def _stable_scalar_for_parquet(value):
    """Normalize unusual pandas scalar without replacing missing values with zero."""
    import datetime
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return None if np.isnan(value) else str(value)
    if isinstance(value, (np.generic,)):
        value=value.item()
    if isinstance(value, (pd.Interval, pd.Period, datetime.date, datetime.datetime)):
        return str(value)
    if isinstance(value, (dict, list, tuple, set, np.ndarray)):
        if isinstance(value, set): value=sorted(value, key=str)
        if isinstance(value, np.ndarray): value=value.tolist()
        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
    if isinstance(value, (str, bool, int, float, bytes)):
        return value
    try:
        if pd.isna(value): return None
    except (ValueError, TypeError):
        pass
    return str(value)

def normalize_parquet_schema(frame, name='table'):
    """Keep numeric scientific data intact; safely serialize Interval/Categorical/nested objects."""
    out=frame.copy()
    changes=[]
    for col in out.columns:
        series=out[col]
        dtype=series.dtype
        reason=None
        if isinstance(dtype, pd.CategoricalDtype):
            # In particular, pd.cut creates categorical<Interval>, unsupported by PyArrow.
            out[col]=pd.Series([None if pd.isna(x) else str(x) for x in series],
                               index=out.index, dtype='string')
            reason='categorical_to_string'
        elif isinstance(dtype, (pd.IntervalDtype, pd.PeriodDtype)):
            out[col]=pd.Series([_stable_scalar_for_parquet(x) for x in series],
                               index=out.index, dtype='string')
            reason='interval_period_to_string'
        elif pd.api.types.is_object_dtype(dtype):
            nonnull=series.dropna()
            # Standard homogeneous strings are safe; mixed/object complex columns are not.
            types={type(x) for x in nonnull}
            if types and not types.issubset({str}):
                if types.issubset({int,float,bool,np.int64,np.int32,np.float64,np.float32}):
                    out[col]=pd.to_numeric(series,errors='raise')
                    reason='numeric_object_to_number'
                else:
                    out[col]=pd.Series([_stable_scalar_for_parquet(x) for x in series],
                                       index=out.index, dtype='string')
                    reason='mixed_or_nested_object_to_json_text'
        elif isinstance(dtype, pd.ArrowDtype) and ('interval' in str(dtype).lower() or
                                                     'struct' in str(dtype).lower() or
                                                     'list' in str(dtype).lower()):
            out[col]=pd.Series([_stable_scalar_for_parquet(x) for x in series],
                               index=out.index, dtype='string')
            reason='arrow_extension_to_string'
        if reason:
            # Nullity and positional identity must not change when serializing text columns.
            old_null=int(series.isna().sum())
            new_null=int(out[col].isna().sum())
            if old_null!=new_null:
                raise ValueError(f'Serialization changed missingness: {name}.{col} {old_null}->{new_null}')
            changes.append({'table':name,'column':str(col),'original_dtype':str(dtype),
                            'export_dtype':str(out[col].dtype),'reason':reason,
                            'n_rows':len(out),'n_null':new_null})
    if len(out)!=len(frame) or list(out.columns)!=list(frame.columns):
        raise AssertionError('Export normalization changed structure')
    return out,changes

def export_evidence(path,tables,figs,report,configs,log_path=None):
    """Write and independently read every evidence Parquet; verify ZIP members, SHA and row counts."""
    import tempfile
    import pyarrow.parquet as pq
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory() as temporary:
        folder=Path(temporary)
        file_manifest=[];audit=[]
        for name,df in tables.items():
            if df is None or len(df)==0:
                continue
            if not isinstance(df,pd.DataFrame): raise TypeError(f'{name} must be a DataFrame')
            if not re.fullmatch(r'[a-zA-Z0-9_]+',name): raise ValueError('Unsafe table name '+name)
            fixed,changes=normalize_parquet_schema(df,name)
            audit.extend(changes)
            dst=folder/(name+'.parquet')
            tmp=folder/(name+'.write.tmp.parquet')
            fixed.to_parquet(tmp,index=False,engine='pyarrow',compression='zstd')
            # Check actual parquet schema / columns / row counts, not just ZIP existence.
            md=pq.ParquetFile(tmp).metadata
            if md.num_rows!=len(df):raise ValueError(f'Parquet row count mismatch: {name}')
            back=pd.read_parquet(tmp,engine='pyarrow')
            if list(back.columns)!=list(df.columns) or len(back)!=len(df):
                raise ValueError(f'Parquet readback/schema failure: {name}')
            tmp.replace(dst)
            file_manifest.append({'path':dst.name,'sha256':sha256_file(dst),
                                  'rows':len(df),'columns':len(df.columns),'bytes':dst.stat().st_size})
        audit_frame=pd.DataFrame(audit,columns=['table','column','original_dtype','export_dtype',
                                                  'reason','n_rows','n_null'])
        EXPORT_SERIALIZATION_AUDIT.clear();EXPORT_SERIALIZATION_AUDIT.extend(audit)
        # A nonempty QA ledger is always present, even if nothing required conversion.
        audit_file=folder/'serialization_qa.parquet'
        audit_frame.to_parquet(audit_file,index=False)
        file_manifest.append({'path':audit_file.name,'sha256':sha256_file(audit_file),
                              'rows':len(audit_frame),'columns':len(audit_frame.columns),
                              'bytes':audit_file.stat().st_size})
        (folder/'METHODOLOGY.md').write_text(report,encoding='utf-8')
        (folder/'configs.json').write_text(json.dumps(configs,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
        figures=folder/'figures';figures.mkdir(exist_ok=True)
        for src in figs:
            if Path(src).is_file():
                (figures/Path(src).name).write_bytes(Path(src).read_bytes())
        if log_path and Path(log_path).exists():
            with gzip.open(folder/'technical.log.gz','wb') as stream:
                stream.write(Path(log_path).read_bytes())
        (folder/'files_manifest.json').write_text(json.dumps(file_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
        tmpzip=path.with_suffix('.temporary.zip')
        with zipfile.ZipFile(tmpzip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,
                             allowZip64=True) as z:
            for f in sorted(folder.rglob('*')):
                if f.is_file():z.write(f,arcname=str(f.relative_to(folder)))
        with zipfile.ZipFile(tmpzip) as z:
            bad=z.testzip()
            if bad is not None:raise ValueError('Corrupted evidence member: '+bad)
            names=set(z.namelist())
            for essential in ('predictions.parquet','splits.parquet','serialization_qa.parquet','files_manifest.json'):
                if essential not in names:raise ValueError('Evidence missing '+essential)
            for member in file_manifest:
                with z.open(member['path']) as f:
                    h=hashlib.sha256()
                    for block in iter(lambda:f.read(4*1024**2),b''):h.update(block)
                if h.hexdigest()!=member['sha256']:
                    raise ValueError('ZIP checksum invalid '+member['path'])
        tmpzip.replace(path)
    return {'sha256':sha256_file(path),'size':path.stat().st_size,
            'tables':len(file_manifest),'converted_columns':len(audit)}


def verify_local(root):
    from openpyxl import load_workbook
    import pyarrow.parquet as pq
    root=Path(root)
    manifest=json.loads((root/'run_manifest.json').read_text(encoding='utf-8'))
    for filename, expected in manifest['outputs'].items():
        f=root/filename
        if not f.is_file() or sha256_file(f)!=expected['sha256']:
            raise ValueError('Manifest output checksum mismatch: '+filename)
    excel=load_workbook(root/'ML1_MIGRATION_RESULTS.xlsx',read_only=True)
    sheetnames=list(excel.sheetnames);excel.close()
    if not sheetnames:raise ValueError('No Excel worksheets')
    checked=0
    with zipfile.ZipFile(root/'ML1_EVIDENCE.zip') as z:
        if z.testzip() is not None:raise ValueError('Bad Evidence ZIP CRC')
        files=json.loads(z.read('files_manifest.json'))
        for m in files:
            with z.open(m['path']) as stream:
                content=stream.read()
            if hashlib.sha256(content).hexdigest()!=m['sha256']:
                raise ValueError('Evidence SHA mismatch '+m['path'])
            md=pq.ParquetFile(io.BytesIO(content)).metadata
            if md.num_rows!=m['rows']:raise ValueError('Evidence row count differs '+m['path'])
            checked+=1
        if not pd.read_parquet(io.BytesIO(z.read('predictions.parquet'))).shape[0]>0:
            raise ValueError('Missing predictions')
        if not pd.read_parquet(io.BytesIO(z.read('splits.parquet'))).shape[0]>0:
            raise ValueError('Missing splits')
    return {'status':'PASS_LOCAL_ONLY','sheets':len(sheetnames),
            'parquet_files_verified':checked,'sha256_verified':True,
            'files':list(manifest['outputs'])}

def checkpoint_job(jobid,store,execute,mode='FULL'):
    """Типобезопасный атомарный checkpoint: DataFrame/Parquet или dict/JSON.

    Возвращает (value, resumed). Контроль SHA-256 обязателен.
    Оптимизатор возвращает dict; эту запись нельзя ошибочно направлять в Parquet.
    """
    cache_id=tuple(jobid)+('classification_alignment_fix_v2_1',) if jobid[0]=='CLASSIFY' else tuple(jobid)
    root=Path(store);root.mkdir(parents=True,exist_ok=True)
    stem=hash_obj(cache_id)
    locations={'dataframe':root/(stem+'.parquet'), 'optuna_dict':root/(stem+'.value.json')}
    mark=root/(stem+'.meta.json')
    if mode in ('RESUME','RECOVER'):
        if mark.is_file():
            try:
                saved=json.loads(mark.read_text(encoding='utf-8'))
                kind=saved.get('kind')
                dest=locations.get(kind)
                if (saved.get('jobid')==list(cache_id) and dest is not None
                    and dest.is_file() and sha256_file(dest)==saved.get('sha256')):
                    if kind=='dataframe':return pd.read_parquet(dest),True
                    result=json.loads(dest.read_text(encoding='utf-8'))
                    if isinstance(result,dict):return result,True
                if mode=='RECOVER':
                    raise ValueError('Verified checkpoint identity, SHA256 or payload mismatch: '+str(cache_id))
            except (OSError,ValueError,TypeError,KeyError):
                if mode=='RECOVER':raise
                # Ordinary RESUME may recompute damaged entries; strict RECOVER must NEVER retrain.
                pass
        elif mode=='RECOVER':
            raise FileNotFoundError('Strict recovery refuses missing checkpoint: '+str(cache_id))
    elif mode not in ('FULL',):
        raise ValueError('checkpoint_job mode must be FULL, RESUME or RECOVER')
    out=execute()
    if isinstance(out,pd.DataFrame):
        kind='dataframe';dest=locations[kind]
        temp=root/(stem+'.temporary.parquet')
        out.to_parquet(temp,index=False)
    elif isinstance(out,dict):
        kind='optuna_dict';dest=locations[kind]
        temp=root/(stem+'.temporary.value.json')
        payload=json.dumps(out,ensure_ascii=False,sort_keys=True,allow_nan=False,
            default=lambda x: x.item() if isinstance(x,np.generic) else str(x))
        assert isinstance(json.loads(payload),dict)
        temp.write_text(payload,encoding='utf-8')
    else:
        raise TypeError(f'Unsupported checkpoint type: {type(out).__name__}; expected DataFrame or dict')
    temp.replace(dest)
    marker={'jobid':list(cache_id),'kind':kind,'sha256':sha256_file(dest),
            'timestamp':utc(),'size_bytes':dest.stat().st_size}
    # Метаданные записываются после успешной сериализации payload.
    temp_marker=root/(stem+'.temporary.meta.json')
    temp_marker.write_text(json.dumps(marker,ensure_ascii=False,indent=2),encoding='utf-8')
    temp_marker.replace(mark)
    return out,False


In [ ]:
# ML-1 V2: corrected territorial units, source provenance, leakage and feasibility checks.
VERSION = 'ML1_V4_4_STREAM_XLSX_READBACK_SAFE_2026-10-26'
SOURCE_LAG_YEARS = 2
HORIZONS = (1, 3, 5)
FINAL_SEEDS = tuple(range(1001, 1051))
DEV_SEEDS = (11, 23, 37)

FEATURE_QUERIES = [
 ('wage','B_economy',r'заработн.{0,45}плат|оплат[аы] труда|среднемесячн.{0,35}работник',r'учител|педагог|врач|культур|бюджетн'),
 ('unemployment','B_economy',r'безработн|безработиц',r'пособи|выплат'),
 ('employment','B_economy',r'занят.{0,30}экономик|численност.{0,35}работник|работник.{0,35}организац',r'учител|врач|оплат|детск'),
 ('income','B_economy',r'доход.{0,35}населен|денежн.{0,35}доход|среднедушев.{0,25}доход',r'налогов|бюджет'),
 ('small_business','B_economy',r'малых предприяти|средних предприяти|субъект.{0,20}предпринимател',r'работник|налог|оборот'),
 ('investment','B_economy',r'инвестиц.{0,40}основн|основн.{0,30}капитал',r'бюджетн.{0,12}инвестиц'),
 ('housing_new','C_housing_services',r'ввод.{0,40}жил|введено.{0,40}жил|ввод.{0,40}жиль',r'строительн.{0,15}организац'),
 ('housing_stock','C_housing_services',r'жилищн.{0,20}фонд|общая площадь жилых|площадь жилых помещени',r'аварийн|ветх'),
 ('utilities','C_housing_services',r'водопровод|канализац|газифиц|благоустроенн|теплоснабжен',r'стоимост|оплат|затрат'),
 ('education','C_housing_services',r'общеобразовательн.{0,25}организац|дошкольн.{0,25}учрежден|образовательн.{0,30}организац',r'расход|заработ|педагог'),
 ('health','C_housing_services',r'врач.{0,30}населен|медицинск.{0,30}организац|медицинск.{0,30}учрежден|число больничн',r'расход|заработ|болезн'),
 ('budget','C_housing_services',r'доход.{0,35}бюджет|расход.{0,35}бюджет',r'федеральн.{0,30}бюджет'),
]
FEATURE_BASE=['lag0','lag1','lag2','lag3','lag4','mean3','mean5','drift3','population_log','origin_year']


def clean_d01(raw):
    """The raw D01 rate is inferred against counts, never assumed to be per thousand.
    D01 migration is a count; average_population is the preferred population-year exposure.
    Both 1 and 1000 raw-rate multipliers are evaluated; ambiguous evidence is a hard gate.
    """
    required={'oktmo','year','region','migration','migration_rate','population'}
    if not required.issubset(raw.columns): raise ValueError(f'D01 missing {required-set(raw.columns)}')
    d=raw.copy()
    d['territory_id']=oktmo_str(d.oktmo)
    d['year']=num(d.year)
    if d.year.isna().any() or (d.year%1).fillna(1).ne(0).any():raise ValueError('Invalid years')
    d.year=d.year.astype(int)
    if d.duplicated(['territory_id','year']).any():raise ValueError('D01 duplicate OKTMO/year — JOIN FAIL')
    d['region_id']=d.region.astype('string').str.strip()
    if d.region_id.isna().any() or d.region_id.eq('').any():raise ValueError('D01 region NA')
    if d.groupby('territory_id').region_id.nunique().gt(1).any():raise ValueError('D01 code migrated across region; crosswalk necessary')
    for c in ['migration','migration_rate','population','average_population']:
        if c in d:d[c]=num(d[c])
    d['migration_net']=d.migration
    exposures=[x for x in ['average_population','population'] if x in d]
    candidate=[]
    for exposure in exposures:
        good=d[['migration_net','migration_rate',exposure]].notna().all(axis=1)&d[exposure].gt(0)
        good &= (d.migration_net.abs()>=2)&(d.migration_rate.abs()>1e-9)
        v=d.loc[good].copy()
        if len(v)<100:continue
        implied=1000*v.migration_net/v[exposure]
        for scale in (1.,1000.):
            residual=(implied-v.migration_rate*scale).abs()
            relative=residual/np.maximum(1.,implied.abs())
            candidate.append((float(relative.median()),float((relative<.15).mean()),exposure,scale,len(v)))
    if not candidate:raise ValueError('No empirical sample for migration_rate units inference')
    candidate.sort(key=lambda t:(t[0],-t[1]))
    best=candidate[0]
    if best[1]<.7 or (len(candidate)>1 and candidate[1][0] < best[0]*1.2 and candidate[1][3] != best[3]):
        raise ValueError(f'Ambiguous migration rate units: {candidate}')
    exposure,scale=best[2],best[3]
    d['population_exposure']=d[exposure].where(d[exposure].gt(0))
    d['migration_rate_raw']=d['migration_rate']
    d['migration_rate']=d.migration_rate*scale  # persons per 1000 residents
    implied=d.migration_net/d.population_exposure*1000
    comparable=d[['migration_net','migration_rate','population_exposure']].notna().all(axis=1)
    deviation=(implied-d.migration_rate).abs()
    tolerance=np.maximum(1.,.10*np.maximum(d.migration_rate.abs(),implied.abs()))
    d['count_rate_conflict']=comparable & deviation.gt(tolerance)
    d['net_verified']=comparable & ~d.count_rate_conflict
    d['mun_type']=d.get('mun_type',pd.Series(pd.NA,index=d.index)).astype('string')
    changed=(d.groupby('territory_id').mun_type.nunique(dropna=True)>1)
    d['territory_type_changed']=d.territory_id.isin(changed[changed].index)
    d['row_source_id']=d.territory_id+':'+d.year.astype(str)
    qa={'observations':len(d),'regions':int(d.region_id.nunique()),'territories':int(d.territory_id.nunique()),
        'years':sorted(int(k) for k in d.year.unique()),'rate_multiplier':scale,
        'rate_inference':{'median_relative_discrepancy':best[0],'fraction_within_15pct':best[1],
                          'denominator':exposure,'support':best[4],'candidate_scores':candidate},
        'rate_unit':'persons per 1000 residents','net_verified':int(d.net_verified.sum()),
        'count_rate_conflict':int(d.count_rate_conflict.sum()),'rate_nonnull':int(d.migration_rate.notna().sum()),
        'municipal_type_changed_territories':int(changed.sum()),
        'territorial_vintage':'2026 repository stage I; older releases by origin not provided',
        'boundaries_continuity_verified':False}
    return d.sort_values(['territory_id','year']).reset_index(drop=True),qa


def panel_for_h(d,h,lag=2):
    p=d[['territory_id','region_id','mun_type','territory_type_changed','year']].rename(columns={'year':'origin_year'}).copy()
    p['horizon']=h;p['target_year']=p.origin_year+h
    p['information_cutoff']=p.origin_year-lag
    p['max_feature_year']=p.information_cutoff
    obs=d[['territory_id','year','migration_rate','migration_net','net_verified','population_exposure']].copy()
    obs.loc[~obs.net_verified,'migration_net']=np.nan
    p=p.merge(obs[['territory_id','year','migration_rate','migration_net']].rename(columns={
        'year':'target_year','migration_rate':'target_rate','migration_net':'target_net'}),
        on=['territory_id','target_year'],how='left',validate='m:1')
    for k in range(5):p[f'lag{k}']=exact_lookup(p,obs,'year','migration_rate',k)
    p['mean3']=p[['lag0','lag1','lag2']].mean(axis=1,skipna=False)
    p['mean5']=p[[f'lag{k}' for k in range(5)]].mean(axis=1,skipna=False)
    p['drift3']=(p.lag0-p.lag2)/2
    for k in range(3):p[f'net_lag{k}']=exact_lookup(p,obs,'year','migration_net',k)
    p['net_mean3']=p[['net_lag0','net_lag1','net_lag2']].mean(axis=1,skipna=False)
    p['population_exposure']=exact_lookup(p,obs,'year','population_exposure',0)
    p['population_log']=np.log1p(p.population_exposure.clip(lower=0))
    series=[]
    for offset in range(1,max(h,3)+1):
        left=p[['territory_id','origin_year']].copy()
        left['_target_year']=left.origin_year+offset;left['_ord']=np.arange(len(left))
        right=obs[['territory_id','year','migration_net']].rename(columns={'year':'_target_year','migration_net':'_net'})
        s=left.merge(right,on=['territory_id','_target_year'],how='left',validate='m:1').sort_values('_ord')
        series.append(s._net.to_numpy(float))
    arr=np.asarray(series)
    p['target_cum_net']=np.where(np.isfinite(arr[:h]).all(axis=0),arr[:h].sum(axis=0),np.nan)
    p['stable_outflow_3y']=np.where(np.isfinite(arr[:3]).all(axis=0),
        (arr[:3]<0).all(axis=0).astype(float),np.nan)
    p['target_sign']=np.where(p.target_net.gt(0),1.,np.where(p.target_net.lt(0),0.,np.nan))
    p['practical_outflow']=np.where(p.target_rate.notna(),p.target_rate.lt(-5.).astype(float),np.nan)
    p['row_id']=p.territory_id+':'+p.origin_year.astype(str)+':h'+str(h)
    if p.row_id.duplicated().any():raise ValueError('Duplicate prediction key')
    if p.max_feature_year.gt(p.information_cutoff).any():raise ValueError('Future leakage in panel')
    return p


def future_canary(d):
    for h in HORIZONS:
        a=panel_for_h(d,h)
        origin=int(a.origin_year.max())-h
        before=a[a.origin_year.eq(origin)].set_index('row_id').sort_index()
        if before.empty:raise ValueError('Canary requires rows')
        mutated=d.copy();future=mutated.year.gt(origin-SOURCE_LAG_YEARS)
        for c,new in [('migration_rate',987654.),('migration_net',-88888.),('population_exposure',1e10)]:
            mutated.loc[future,c]=new
        after=panel_for_h(mutated,h)
        after=after[after.origin_year.eq(origin)].set_index('row_id').sort_index()
        check=[f'lag{k}' for k in range(5)]+['mean3','mean5','drift3','net_lag0','net_mean3','population_log']
        if not before[check].equals(after[check]):return False
    return True


def plan_splits(p,seed=2026):
    """Four distinct protocols, fixed before model fitting. Fit labels published by DEV and TEST cutoffs."""
    years=sorted(int(z) for z in p.target_year.unique())
    if len(years)<5:return {},[{'protocol':'ALL','reason':'<5 distinct target years','status':'NOT_FEASIBLE'}]
    regions=sorted(map(str,p.region_id.unique()))
    if len(regions)<8:return {},[{'protocol':'REGION','reason':'<8 regions','status':'NOT_FEASIBLE'}]
    rng=np.random.default_rng(seed);held=set(rng.permutation(regions)[:max(2,round(.30*len(regions)))])
    spec=[('ROLLING_TIME',p[p.target_year.eq(years[-1])],p[p.target_year.lt(years[-1])]),
          ('ROLLING_TIME_PREV',p[p.target_year.eq(years[-2])],p[p.target_year.lt(years[-2])]),
          ('TERRITORY_ONLY',p[p.region_id.isin(held)&p.target_year.isin(years[-2:])],p[~p.region_id.isin(held)]),
          ('CROSSED',p[p.region_id.isin(held)&p.target_year.eq(years[-1])],
                        p[(~p.region_id.isin(held))&p.target_year.lt(years[-1])])]
    out={};issues=[]
    for protocol,test,candidates in spec:
        if test.empty:issues.append({'protocol':protocol,'status':'NOT_FEASIBLE','reason':'empty test'});continue
        test_info=int(test.information_cutoff.min())
        train=candidates[candidates.target_year.le(test_info)].copy()
        # geographic isolation for regional tests including dev and training
        if protocol in ('CROSSED','TERRITORY_ONLY') and set(train.region_id)&set(test.region_id):
            raise ValueError('Region overlap')
        if train.empty:issues.append({'protocol':protocol,'status':'NOT_FEASIBLE','reason':'no published train labels'});continue
        dev_year=int(train.target_year.max())
        dev=train[train.target_year.eq(dev_year)].copy()
        dev_cutoff=int(dev.information_cutoff.min())
        fit=train[train.target_year.le(dev_cutoff)].copy()
        if len(fit)<80 or len(dev)<20 or len(test)<20 or fit.region_id.nunique()<3:
            issues.append({'protocol':protocol,'status':'NOT_FEASIBLE','reason':f'fit={len(fit)} dev={len(dev)} test={len(test)}'});continue
        if set(train.row_id)&set(test.row_id):raise ValueError('Train/test keys overlap')
        if (train.target_year>test_info).any() or (fit.target_year>dev_cutoff).any():raise ValueError('Label availability leakage')
        out[protocol]={'train':train,'fit':fit,'dev':dev,'test':test,'cutoff':test_info,
            'withheld_regions':sorted(held) if protocol in ('CROSSED','TERRITORY_ONLY') else [],
            'split_hash':hash_obj({'train':sorted(train.row_id),'dev':sorted(dev.row_id),'test':sorted(test.row_id)})}
    return out,issues


def record_predictions(test,pred,model,family,protocol,h,seed,cutoff):
    keys=['row_id','territory_id','region_id','mun_type','origin_year','target_year','information_cutoff',
          'max_feature_year','horizon','population_exposure','target_rate','target_net','target_sign',
          'practical_outflow','target_cum_net','stable_outflow_3y']
    z=test[keys].copy()
    z['prediction']=np.asarray(pred,float)
    z['model']=model;z['feature_set']=family;z['protocol']=protocol
    z['seed']=seed;z['max_label_year_fit']=cutoff;z['split']='TEST';z['model_version']=VERSION
    if z.max_feature_year.gt(z.information_cutoff).any():raise ValueError('Feature leakage')
    if z.max_label_year_fit.gt(z.information_cutoff).any():raise ValueError('Label leakage')
    return z


def feature_cols(family,available):
    groups={'A_history':[], 'B_economy':available.get('B_economy',[]),
            'C_housing_services':available.get('B_economy',[])+available.get('C_housing_services',[]),
            'D_spatial':available.get('B_economy',[])+available.get('C_housing_services',[])+available.get('D_spatial',[]),
            'E_all':available.get('B_economy',[])+available.get('C_housing_services',[])+available.get('D_spatial',[])+available.get('E_all',[])}
    return list(dict.fromkeys(FEATURE_BASE+groups[family]))


def valid_families(available):
    # Keep the actual comparison hierarchy A->B->C->D->E, even if a family is infeasible.
    families=['A_history'];prior=set(feature_cols('A_history',available))
    for name,req in [('B_economy','B_economy'),('C_housing_services','C_housing_services'),('D_spatial','D_spatial'),('E_all','E_all')]:
        present=set(feature_cols(name,available))
        if available.get(req) and present!=prior:families.append(name)
        prior=present
    return families


def profiles_from_d01(d,threshold=5.):
    x=d.groupby(['territory_id','region_id','mun_type'],dropna=False).agg(
        observed_years=('year','nunique'),migration_rate_mean=('migration_rate','mean'),
        migration_rate_last=('migration_rate','last'),population_last=('population_exposure','last'),
        net_last=('migration_net','last')).reset_index()
    x['profile']=np.select([x.migration_rate_mean<-threshold,x.migration_rate_mean>threshold],
                            ['Устойчивое отрицательное сальдо','Устойчивое положительное сальдо'],default='Нейтральный профиль')
    return x

# scikit-learn >=1.7 can reject CatBoost's old estimator tag protocol when it is
# a last step of sklearn.Pipeline. Fitting the preprocessor and estimator separately
# preserves train-only transformations and enables all model families consistently.
from sklearn.base import BaseEstimator, RegressorMixin

class FittedTabularRegressor(RegressorMixin,BaseEstimator):
    def __init__(self,preprocessor,estimator,columns):
        self.preprocessor=preprocessor;self.estimator=estimator;self.columns=columns
        self.named_steps={'preprocess':preprocessor,'model':estimator}
        self.is_fitted_=True
    def fit(self,X,y=None):
        raise RuntimeError('Already fitted on historical TRAIN; refitting inside permutation/SHAP is forbidden')
    def predict(self,frame):
        x=frame[self.columns].astype({'region_id':str,'mun_type':str})
        return np.asarray(self.estimator.predict(self.preprocessor.transform(x)),float)


def fit_predict(train,valid,features,name,parameters,seed=11,target='target_rate'):
    prep,columns=numeric_design(train,features)
    Xtr=train[columns].astype({'region_id':str,'mun_type':str})
    Xte=valid[columns].astype({'region_id':str,'mun_type':str})
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        matrix=prep.fit_transform(Xtr)
        estimator=instantiate(name,parameters,seed)
        estimator.fit(matrix,train[target].to_numpy(float))
        pred=np.asarray(estimator.predict(prep.transform(Xte)),float)
    wrapper=FittedTabularRegressor(prep,estimator,columns)
    return pred,wrapper,[str(w.message)[:160] for w in caught[:5]]


def ml1_base_model_name(name):
    """Strictly normalize recursive display names before estimator/DEV-selection calls."""
    base=str(name).removeprefix('recursive_')
    if base not in OPTUNA_TRIALS:
        raise ValueError(f'Unsupported estimator: {name} -> {base}')
    return base


In [ ]:
# V4 | удаление исключительно известного Release ML-1 V2.3; исходные данные не затрагиваются
CLEAN_PREVIOUS_ML1_RELEASE=True
OLD_ML1_TAG='stage2-ml1-v2.3.0-20261022T105208Z-db7180cc'
assert OLD_ML1_TAG.startswith('stage2-ml1-v2.3.0-')
def v4_delete_old_release():
    if not CLEAN_PREVIOUS_ML1_RELEASE:return {'status':'SKIPPED'}
    # Ограниченный список только одного старого расчёта ML-1; НИКОГДА data-* или main.
    s=github_session(token_from_colab())
    uri=GITHUB_API+'/releases/tags/'+OLD_ML1_TAG
    res=s.get(uri,timeout=(20,60))
    if res.status_code==404:return {'status':'ALREADY_REMOVED'}
    res.raise_for_status()
    rid=res.json()['id'];deleted=s.delete(GITHUB_API+'/releases/'+str(rid),timeout=(30,120))
    if deleted.status_code!=204:raise RuntimeError('Cannot delete old ML1 release: '+str(deleted.status_code))
    tag_res=s.delete(GITHUB_API+'/git/refs/tags/'+OLD_ML1_TAG,timeout=(30,120))
    if tag_res.status_code not in (204,404):raise RuntimeError('Old ML1 git tag deletion failed')
    if s.get(uri,timeout=(20,60)).status_code!=404:raise RuntimeError('Old ML1 release still exists')
    return {'status':'OLD_ML1_RELEASE_REMOVED','tag':OLD_ML1_TAG}
try:
    V4_GITHUB_CLEANUP=v4_delete_old_release()
except Exception as exc:
    V4_GITHUB_CLEANUP={'status':'INCOMPLETE','reason':type(exc).__name__+': '+str(exc)[:180]}
print('Prior ML-1 Release cleanup:',V4_GITHUB_CLEANUP['status'])


## 3. Разбор фактического каталога и факторный DATA_GATE

V2.3 ошибочно отклоняла в том числе показатели жилья из-за 0,1–0,2% конфликтов и считала покрытие по несопоставимым годам. V3 различает **полные точные повторы**, **разрезы показателей с неоднозначными значениями** и **неизвестную территориальную идентичность**. Показатель принимается только при проверке точного кода/единицы/уровня, явного итогового разреза или исключения исключительно конфликтных ключей в пределах заранее заданного малого допуска; значения разных разрезов никогда не суммируются без научного основания.

Региональные ряды заработной платы, безработицы, занятости, инвестиций маркируются как `region-level context`; муниципальные показатели жилья, бюджетов и образования сохраняют уровень ОКТМО × год. В ноутбуке остаются все исходные уровни и ограничения. Для географии допустим только документированный прокси по координатам наблюдённых населённых пунктов; ни административные границы, ни центроиды не выдумываются.

In [ ]:
# V2 registry: enumerate all economic assets, choose only independently identifiable
# time/territory/indicator series, retain all rejected candidates as audit evidence.
import unicodedata

def normalize_geo(value):
    if pd.isna(value):return ''
    x=unicodedata.normalize('NFKC',str(value)).casefold().strip()
    x=re.sub(r'\b(область|край|республика|город|г\.|г |муниципальный|район)\b',' ',x)
    x=re.sub(r'[^а-яёa-z0-9]+',' ',x)
    return ' '.join(x.split())


def inventory_asset(session,asset,cache):
    import duckdb
    path=fetch_asset(session,asset,cache);cols=asset.get('columns_manifest',[])
    needed={'indicator_code','indicator_name','indicator_unit','indicator_value','year'}
    key='object_oktmo' if 'object_oktmo' in cols else 'oktmo' if 'oktmo' in cols else None
    lev='object_level' if 'object_level' in cols else 'mun_level' if 'mun_level' in cols else None
    if not needed.issubset(cols) or key is None or lev is None:
        return pd.DataFrame([{'asset_id':asset['asset_id'],'status':'NOT_FEASIBLE_SCHEMA','reason':'No indicator/code/unit/year/municipality columns'}]),path
    con=duckdb.connect(':memory:')
    try:
        query=f'''SELECT CAST(indicator_code AS VARCHAR) indicator_code,
               CAST(indicator_name AS VARCHAR) indicator_name,
               CAST(indicator_unit AS VARCHAR) indicator_unit,
               CAST("{lev}" AS VARCHAR) observation_level,
               COUNT(*) n_rows,COUNT(DISTINCT CAST("{key}" AS VARCHAR)) n_keys,
               COUNT(DISTINCT TRY_CAST("year" AS INTEGER)) AS n_years,
               MIN(TRY_CAST("year" AS INTEGER)) AS min_year,MAX(TRY_CAST("year" AS INTEGER)) AS max_year
          FROM read_parquet(?) WHERE indicator_code IS NOT NULL AND indicator_name IS NOT NULL GROUP BY ALL'''
        data=con.execute(query,[str(path)]).df()
    finally:con.close()
    data['asset_id']=asset['asset_id'];data['key_column']=key;data['level_column']=lev
    data['status']='CATALOGED'
    return data,path


def _ml1_feature_sql(key,lev,region_col=None):
    """Generate one validated DuckDB SELECT with explicit, nonreserved aliases."""
    # Column names can only come from actual parquet schema, never from user input.
    for name in (key,lev,region_col):
        if name is not None and not re.fullmatch(r'[A-Za-z_][A-Za-z_0-9]*',str(name)):
            raise ValueError('Unsafe column identifier in real parquet schema')
    extra=(f', CAST("{region_col}" AS VARCHAR) AS region_name' if region_col
           else ', CAST(NULL AS VARCHAR) AS region_name')
    return f'''SELECT
       CAST("{key}" AS VARCHAR) AS territory_id,
       TRY_CAST("year" AS INTEGER) AS feature_year,
       TRY_CAST(REPLACE(REPLACE(CAST("indicator_value" AS VARCHAR), ' ', ''), ',', '.') AS DOUBLE) AS feature_value,
       CAST("indicator_name" AS VARCHAR) AS source_indicator
       {extra}
       FROM read_parquet(?)
       WHERE CAST("indicator_code" AS VARCHAR) = ?
         AND CAST("indicator_name" AS VARCHAR) = ?
         AND CAST("indicator_unit" AS VARCHAR) = ?
         AND CAST("{lev}" AS VARCHAR) = ?'''


def choose_features(catalog,asset_paths,source_d,minimum_coverage=.06,manual_overrides=None):
    """Candidates are empirically tested for unique municipality-year, unit, scope,
    regional consistency and temporal coverage, then ranked using TRAIN-INDEPENDENT metadata.
    Does not use future outcomes or test-set predictive performance to choose indicators.
    """
    import duckdb
    manual_overrides=manual_overrides or {}
    catalogue=catalog[catalog.status.eq('CATALOGED')].copy() if len(catalog) else pd.DataFrame()
    if catalogue.empty:return [],pd.DataFrame([{'status':'NOT_FEASIBLE','reason':'No cataloged municipal indicators'}])
    catalogue['description']=catalogue.indicator_name.fillna('').astype(str).str.lower()
    catalogue['level_desc']=catalogue.observation_level.fillna('').astype(str).str.lower()
    catalogue=catalogue[catalogue.n_years.ge(4)&catalogue.n_keys.ge(40)]
    catalogue=catalogue[catalogue.level_desc.str.contains(r'муницип|городск.{0,8}округ|район|поселен',na=False)]
    catalogue=catalogue[~catalogue.level_desc.str.contains('регион|субъект|страна|российск|федерац',na=False)]
    accepted=[];audit=[]
    registry=source_d[['territory_id','region_id']].drop_duplicates()
    for variable,family,inc,exc in FEATURE_QUERIES:
        potential=catalogue[catalogue.description.str.contains(inc,regex=True,na=False)&
                             ~catalogue.description.str.contains(exc,regex=True,na=False)].copy()
        if variable in manual_overrides:
            v=manual_overrides[variable]
            potential=potential[(potential.asset_id==v['asset_id'])&
                                 potential.indicator_code.astype(str).eq(str(v['indicator_code']))&
                                 potential.indicator_unit.astype(str).eq(str(v['indicator_unit']))]
        if potential.empty:
            audit.append({'feature':variable,'family':family,'status':'NOT_FEASIBLE','reason':'No candidate after code/unit/level checks'})
            continue
        tests=[]
        # Bound expensive code-configuration inspection: candidate rankings by metadata, not outcome.
        potential=potential.sort_values(['n_keys','n_years','n_rows'],ascending=False).head(25)
        for _,cand in potential.iterrows():
            item=cand.to_dict()
            aid=item['asset_id'];path=asset_paths[aid]
            key=item['key_column'];lev=item['level_column']
            columns={c[0] for c in duckdb.query(f"DESCRIBE SELECT * FROM read_parquet('{str(path).replace(chr(39),chr(39)*2)}')").fetchall()}
            region_col=next((s for s in ('region_name','report_region_name') if s in columns),None)
            # DuckDB: YEAR and VALUE are ambiguous SQL tokens in alias positions.
            # Explicit AS with non-keyword aliases is required; restore the public
            # DataFrame schema after execution so no downstream code is changed.
            query=_ml1_feature_sql(key,lev,region_col)
            con=duckdb.connect(':memory:')
            try:
                x=con.execute(query,[str(path),str(cand.indicator_code),str(cand.indicator_name),str(cand.indicator_unit),str(cand.observation_level)]).df()
                x=x.rename(columns={'feature_year':'year','feature_value':'value'})
            finally:con.close()
            x=x[x.territory_id.str.fullmatch(r'\d+',na=False)&x.year.notna()].copy()
            x['year']=x.year.astype(int)
            overlap=x.territory_id.isin(registry.territory_id).mean() if len(x) else 0.
            actual=x[x.territory_id.isin(registry.territory_id)].copy()
            dup=float(actual.duplicated(['territory_id','year']).mean()) if len(actual) else 1.
            coverage=float(actual[['territory_id','year']].drop_duplicates().shape[0]/max(1,source_d[['territory_id','year']].drop_duplicates().shape[0]))
            geography=True
            if region_col and len(actual)>0:
                check=actual[['territory_id','region_name']].merge(registry,on='territory_id',how='left',validate='m:1')
                check['left_geo']=check.region_name.map(normalize_geo)
                check['right_geo']=check.region_id.map(normalize_geo)
                usable=check.left_geo.ne('')&check.right_geo.ne('')
                if usable.sum()>30:geography=float((check.loc[usable,'left_geo']==check.loc[usable,'right_geo']).mean())>=.90
            valid=dup==0 and geography and actual.value.notna().sum()>=50 and coverage>=minimum_coverage
            # Explicit candidate tie rule prioritizes coverage & years but does not pick on test outcome.
            score=round(coverage*3+min(1.,float(cand.n_years)/10)+float(actual.territory_id.nunique())/max(100,registry.territory_id.nunique()),6)
            entry={'feature':variable,'family':family,'asset_id':aid,'indicator_code':str(cand.indicator_code),
                   'indicator_name':str(cand.indicator_name),'indicator_unit':str(cand.indicator_unit),
                   'observation_level':str(cand.observation_level),'key_column':key,'level_column':lev,
                   'years':int(cand.n_years),'source_coverage':coverage,'territories':int(actual.territory_id.nunique()),
                   'duplicate_fraction':dup,'region_consistent':geography,'score':score,'status':'CANDIDATE_OK' if valid else 'REJECTED_SOURCE_QA',
                   'reason':'' if valid else f'dup={dup:.4f}; region={geography}; coverage={coverage:.3f}'}
            audit.append(entry)
            if valid:tests.append((score,entry,x))
        if not tests:
            audit.append({'feature':variable,'family':family,'status':'NOT_FEASIBLE','reason':'No unique, geographic and sufficiently covered municipal series'})
            continue
        tests.sort(key=lambda v:(-v[0],v[1]['asset_id'],v[1]['indicator_code']))
        if len(tests)>1 and abs(tests[0][0]-tests[1][0])<1e-6 and variable not in manual_overrides:
            audit.append({'feature':variable,'status':'NOT_FEASIBLE_TIE','reason':'Exact ranking tie: manual explicit code needed'})
            continue
        chosen=tests[0][1].copy();chosen['publication_lag_years']=SOURCE_LAG_YEARS
        chosen['status']='SELECTED';chosen['source_sha256']=next((v['sha256'] for v in ACTIVE_INDEX['assets'] if v['asset_id']==chosen['asset_id']),'')
        accepted.append({'spec':chosen,'source_frame':tests[0][2][['territory_id','year','value']].copy()})
        audit.append(chosen)
    return accepted,pd.DataFrame(audit)


def attach_factors_v2(panel,selected,publication_lag=2):
    p=panel.copy();available=defaultdict(list);audit=[]
    for item in selected:
        cfg=item['spec'];name=cfg['feature'];src=item['source_frame'].rename(columns={'value':name})
        if src.duplicated(['territory_id','year']).any():
            raise ValueError(f'JOIN_GATE {name} duplicates not resolved before merge')
        p[name]=exact_lookup(p,src,'year',name,0)
        coverage=float(p[name].notna().mean())
        audit.append({'feature':name,'family':cfg['family'],'coverage':coverage,'source_sha256':cfg.get('source_sha256'),
                      'status':'JOIN_PASS' if coverage>=.06 else 'NOT_FEASIBLE_PANEL_COVERAGE'})
        if coverage>=.06:available[cfg['family']].append(name)
        else:p.drop(columns=[name],inplace=True)
    # fully lagged effect modifiers, available only from same information cutoffs
    for left,right,outname in [('wage','housing_new','interaction_wage_housing'),('employment','education','interaction_employment_education')]:
        if left in p and right in p:
            p[outname]=p[left]*p[right]
            available['E_all'].append(outname)
    return p,dict(available),audit



def spatial_from_transport(session,index,cache,d,p):
    # Full observed transport edges; never fabricate nearest neighbors or coordinates.
    asset=next((a for a in index['assets'] if a['segment']=='06' and a.get('source_id')=='TRANSPORT'),None)
    if asset is None:return p,[],pd.DataFrame([{'status':'NOT_FEASIBLE','reason':'No transport registry'}])
    import pyarrow.parquet as pq
    f=fetch_asset(session,asset,cache)
    cols=['oktmo_from','oktmo_to','distance_km','region_from','region_to']
    original=pq.read_table(f,columns=cols).to_pandas()
    original['territory_id']=oktmo_str(original.oktmo_from,False)
    original['neighbor_id']=oktmo_str(original.oktmo_to,False)
    original['distance_km']=num(original.distance_km)
    regional=d[['territory_id','region_id']].drop_duplicates()
    original=original[original.territory_id.isin(regional.territory_id)&
                      original.neighbor_id.isin(regional.territory_id)&
                      original.distance_km.gt(0)].copy()
    if original.empty:return p,[],pd.DataFrame([{'status':'NOT_FEASIBLE','reason':'Transport endpoints not municipality IDs'}])
    name_map=regional.set_index('territory_id').region_id.to_dict()
    actual_from=original.territory_id.map(name_map).map(normalize_geo)
    actual_to=original.neighbor_id.map(name_map).map(normalize_geo)
    source_from=original.region_from.map(normalize_geo)
    source_to=original.region_to.map(normalize_geo)
    checks=(actual_from==source_from)&(actual_to==source_to)
    # Compare only where both registry and municipal region labels exist.
    checks &= actual_from.ne('')&source_from.ne('')&actual_to.ne('')&source_to.ne('')
    original=original[checks].drop_duplicates(['territory_id','neighbor_id'],keep=False).copy()
    if len(original)<40:return p,[],pd.DataFrame([{'status':'NOT_FEASIBLE','reason':'<40 unique, region-validated transport edges'}])
    hist=d[['territory_id','year','migration_rate']].rename(columns={'territory_id':'neighbor_id','year':'information_cutoff'})
    x=original[['territory_id','neighbor_id','distance_km']].merge(hist,on='neighbor_id',how='inner',validate='m:m')
    x=x.dropna(subset=['migration_rate']);x['weight']=1/x.distance_km.clip(lower=1)
    x['weighted']=x.weight*x.migration_rate
    s=x.groupby(['territory_id','information_cutoff'],as_index=False).agg(weighted=('weighted','sum'),
                                weight=('weight','sum'),neighbor_count=('neighbor_id','nunique'))
    s['neighbor_rate']=s.weighted/s.weight
    result=p.merge(s[['territory_id','information_cutoff','neighbor_rate','neighbor_count']],
        on=['territory_id','information_cutoff'],how='left',validate='m:1')
    cov=float(result.neighbor_rate.notna().mean())
    if cov<.04:return p,[],pd.DataFrame([{'status':'NOT_FEASIBLE','reason':f'Transport spatial coverage={cov:.3%}'}])
    return result,['neighbor_rate','neighbor_count'],pd.DataFrame([{'status':'JOIN_PASS','method':'observed transport edges',
        'edges':len(original),'coverage':cov,'sha256':asset['sha256'],
        'warning':'City-to-city transport observed subnetwork, not national municipal contiguity graph'}])

def attach_spatial_v2(session,index,cache,d,p,k=5):
    """Use only demonstrably municipal centroids or official edges.
    No settlement coordinates are silently reused as municipal coordinates.
    """
    import pyarrow.parquet as pq
    spatial_assets=[a for a in index['assets'] if a['segment']=='06' and 'source_oktmo' in a.get('columns_manifest',[]) and 'latitude' in a.get('columns_manifest',[])]
    if not spatial_assets:return spatial_from_transport(session,index,cache,d,p)
    sp=next((a for a in spatial_assets if a.get('role')=='derived'),spatial_assets[0])
    f=fetch_asset(session,sp,cache)
    columns=['source_oktmo','latitude','longitude','object_level']
    if 'coordinates_valid' in sp['columns_manifest']:columns.append('coordinates_valid')
    x=pq.read_table(f,columns=columns).to_pandas()
    x['territory_id']=oktmo_str(x.source_oktmo,False)
    x['latitude']=num(x.latitude);x['longitude']=num(x.longitude)
    x=x[x.latitude.between(-90,90)&x.longitude.between(-180,180)]
    if 'coordinates_valid' in x:x=x[x.coordinates_valid.astype(str).str.casefold().isin(['true','1','да'])]
    x=x[x.object_level.astype(str).str.casefold().str.contains('муницип|городск.{0,8}округ|муниципальн.{0,8}район',regex=True,na=False)]
    x=x[x.territory_id.isin(d.territory_id)].copy()
    duplicate_codes=x[x.duplicated('territory_id',keep=False)].territory_id.unique()
    x=x[~x.territory_id.isin(duplicate_codes)].copy()
    if x.territory_id.nunique()<60:
        return spatial_from_transport(session,index,cache,d,p)
    from sklearn.neighbors import BallTree
    coords=np.deg2rad(x[['latitude','longitude']].to_numpy(float))
    dst,ids=BallTree(coords,metric='haversine').query(coords,k=min(k+1,len(x)))
    names=x.territory_id.to_list();edges=[]
    for i,name in enumerate(names):
        for j,rad in zip(ids[i,1:],dst[i,1:]):
            if names[j]!=name:edges.append((name,names[j],float(rad*6371.0088)))
    graph=pd.DataFrame(edges,columns=['territory_id','neighbor_id','distance_km'])
    if graph.empty:return p,[],pd.DataFrame([{'status':'NOT_FEASIBLE','reason':'Empty municipality neighbor graph'}])
    hist=d[['territory_id','year','migration_rate']].rename(columns={'territory_id':'neighbor_id','year':'information_cutoff'})
    # Every neighbor observation is exact year information_cutoff; no future data.
    merged=graph.merge(hist,on='neighbor_id',how='inner',validate='m:m').dropna(subset=['migration_rate'])
    merged['weight']=1./merged.distance_km.clip(lower=1)
    merged['weighted']=merged.weight*merged.migration_rate
    agg=merged.groupby(['territory_id','information_cutoff'],as_index=False).agg(
        sw=('weighted','sum'),tw=('weight','sum'),neighbor_count=('neighbor_id','nunique'))
    agg['neighbor_rate']=agg.sw/agg.tw
    result=p.merge(agg[['territory_id','information_cutoff','neighbor_rate','neighbor_count']],
                   on=['territory_id','information_cutoff'],how='left',validate='m:1')
    coverage=result.neighbor_rate.notna().mean()
    if coverage<.06:return spatial_from_transport(session,index,cache,d,p)
    return result,['neighbor_rate','neighbor_count'],pd.DataFrame([{'status':'JOIN_PASS','centroids':len(x),
        'edges':len(graph),'coverage':coverage,'geometry_source_sha256':sp['sha256'],'type':'inverse haversine kNN, historical lag'}])

def preflight_duckdb_sql():
    """Fail early on feature and inventory SQL syntax, before scanning 8M-row assets."""
    import duckdb, tempfile
    from pathlib import Path
    fixture=pd.DataFrame({'object_oktmo':['12345678','12345678'],
            'object_level':['муниципальный округ']*2,
            'year':[2020,2021], 'indicator_code':['TEST']*2,
            'indicator_name':['Тест']*2,'indicator_unit':['человек']*2,
            'indicator_value':['1,5','2,5'], 'region_name':['Регион']*2})
    with tempfile.TemporaryDirectory() as temporary:
        path=Path(temporary)/'fixture.parquet'
        fixture.to_parquet(path,index=False)
        sql=_ml1_feature_sql('object_oktmo','object_level','region_name')
        with duckdb.connect(':memory:') as con:
            got=con.execute(sql,[str(path),'TEST','Тест','человек','муниципальный округ']).df()
            assert got.feature_year.tolist()==[2020,2021]
            assert abs(got.feature_value.iloc[0]-1.5)<1e-10
            inv=con.execute("""SELECT CAST(indicator_code AS VARCHAR) AS indicator_code,
                COUNT(DISTINCT TRY_CAST("year" AS INTEGER)) AS n_years
                FROM read_parquet(?) GROUP BY ALL""",[str(path)]).df()
            assert int(inv.n_years.iloc[0])==2
    return 'PASS: DuckDB aliases, feature query, year cast, inventory query'

print(preflight_duckdb_sql())


In [ ]:
# ML-1 V3: научно проверяемый факторный регистр, а не совпадение названий
V3_PUBLICATION_LAG=2
V3_MIN_MUNICIPAL_COVERAGE=.04
V3_MIN_REGIONAL_COVERAGE=.04
V3_REGION_CODE_SPECS={
    'regional_median_wage':('Y477110208','B_economy'),
    'regional_employment':('Y477170012','B_economy'),
    'regional_wage':('Y477170013','B_economy'),
    'regional_unemployment':('Y477170028','B_economy'),
    'regional_investment_pc':('Y477140023','B_economy'),
    'regional_working_age_share':('Y477110050','E_all'),
}
V3_MUNICIPAL_CODE_SPECS={
    'housing_new':('Y48010001','C_housing_services'),
    'housing_individual':('Y48010002','C_housing_services'),
    'housing_apartment':('Y48010003','C_housing_services'),
    'housing_waitlist_families':('Y48011011','C_housing_services'),
    'housing_improved_families':('Y48011010','C_housing_services'),
    'preschool_count':('Y48014001','C_housing_services'),
    'preschool_capacity':('Y48014002','C_housing_services'),
    'preschool_children':('Y48014003','C_housing_services'),
    'preschool_waitlist':('Y48014004','C_housing_services'),
    'school_count':('Y48015001','C_housing_services'),
    'student_count':('Y48015002','C_housing_services'),
    'school_branches':('Y48015005','C_housing_services'),
    'budget_balance':('Y48013003','E_all'),
    'municipal_sme_support':('Y48013012','B_economy'),
    'budget_own_share':('Y48013015','E_all'),
    'budget_admin_cost_pc':('Y48013004','E_all'),
    'budget_revenue':('Y48013001','E_all'),
    'budget_expenditure':('Y48013002','E_all'),
}

def _v3_norm_key(s):
    return s.astype('string').fillna('').map(normalize_geo)

def _v3_read_series(path,item,scope):
    import duckdb
    import pyarrow.parquet as pq
    columns=set(pq.ParquetFile(path).schema.names)
    valcol='indicator_value'; lev=item['level_column'];key=item['key_column']
    namecol='object_name' if 'object_name' in columns else 'region_name' if 'region_name' in columns else None
    extras=[]
    if namecol:extras.append('CAST("'+namecol+'" AS VARCHAR) AS region_label')
    else:extras.append('CAST(NULL AS VARCHAR) AS region_label')
    excluded={'year','indicator_value','indicator_name','indicator_code','indicator_unit','indicator_period',
      'oktmo','oktmo_stable','oktmo_history','oktmo_year_from','oktmo_year_to','object_oktmo',
      'municipality','region_id','region_name','report_region_name','region_label',
      'object_name','object_level','mun_level','mun_type','mun_type_oktmo','mun_district',
      'comment','source','version_date','source_url','null_value_reason','report_date','report_type',
      'tax_authority','index_form','indicator_section','indicator_section_code'}
    dimensions=[c for c in columns if c not in excluded and (c in ('strashod','stdohod','mosh','gilsem','mater','dol','obras','grup_2','scorper','progobr','tipushr','vozr','udob'))]
    for j,c in enumerate(sorted(dimensions)):extras.append('CAST("'+c+'" AS VARCHAR) AS dim_'+str(j))
    sql=f'''SELECT CAST("{key}" AS VARCHAR) AS source_id, TRY_CAST("year" AS INTEGER) AS year,
        TRY_CAST(REPLACE(REPLACE(CAST(indicator_value AS VARCHAR),' ',''),',','.') AS DOUBLE) AS value,
        {', '.join(extras)} FROM read_parquet(?)
        WHERE CAST(indicator_code AS VARCHAR)=? AND CAST(indicator_name AS VARCHAR)=?
        AND CAST(indicator_unit AS VARCHAR)=? AND CAST("{lev}" AS VARCHAR)=?'''
    with duckdb.connect(':memory:') as con:
        x=con.execute(sql,[str(path),str(item['indicator_code']),str(item['indicator_name']),
             str(item['indicator_unit']),str(item['observation_level'])]).df()
    # Единственный документированный разрыв исходных единиц измерения.
    if str(item['indicator_code'])=='Y48013004' and '2008' in str(item['indicator_unit']):
        x.loc[x.year.eq(2008),'value'] *= 1000.0
    return x, dimensions

def _v3_collapse_series(x, keys, min_pairs=40, tolerance=.01):
    """Убирает только повторенные одинаковые значения. Конфликты не усредняет и не суммирует."""
    z=x.copy()
    z=z.dropna(subset=keys+['value']);z=z[np.isfinite(z['value'].to_numpy(float))].copy()
    if z.empty:return z, {'status':'EMPTY','n_conflicted_keys':0,'n_keys':0}
    z=z.drop_duplicates(keys+['value'])
    pairs=z.groupby(keys,dropna=False).value.nunique().rename('n_unique').reset_index()
    conflict=pairs[pairs.n_unique>1][keys]
    frac=len(conflict)/max(1,len(pairs))
    # Неразрешимые многомерные разрезы не сворачиваются произвольным средним.
    if frac>tolerance:
        return pd.DataFrame(),{'status':'UNRESOLVED_DISAGGREGATION','n_conflicted_keys':len(conflict),
                   'conflict_fraction':frac,'n_keys':len(pairs)}
    if len(conflict):
        ids=pd.MultiIndex.from_frame(conflict)
        z=z[~pd.MultiIndex.from_frame(z[keys]).isin(ids)].copy()
    z=z.drop_duplicates(keys)
    assert not z.duplicated(keys).any()
    return z,{'status':'DEDUP_PASS' if len(conflict)==0 else 'CONFLICT_KEYS_EXCLUDED',
             'n_conflicted_keys':len(conflict),'conflict_fraction':frac,'n_keys':len(pairs),
             'n_valid':len(z)}

def _v3_aggregate_dimension_candidates(raw,keys, dimensions):
    """Проверка явно маркированного итога; только строгие итоги, а не любые строки разрезов."""
    yield 'NO_SLICE',raw
    dcols=['dim_'+str(j) for j in range(len(dimensions))]
    markers={'всего','итого','все','общая сумма','общий итог','total','all'}
    if dcols:
        mask=pd.Series(True,index=raw.index)
        for col in dcols:
            normalized=raw[col].astype('string').str.casefold().str.strip()
            # Не приравниваем NaN к итогу, если переменная имеет содержательные разрезы.
            mask &= normalized.isin(markers)
        if mask.any():yield 'EXPLICIT_TOTAL_ALL_DIMS',raw.loc[mask].copy()
        for col in dcols:
            normalized=raw[col].astype('string').str.casefold().str.strip()
            chosen=raw.loc[normalized.isin(markers)].copy()
            if len(chosen):yield 'EXPLICIT_TOTAL_'+col,chosen

def choose_features(catalog,asset_paths,source_d,minimum_coverage=.04,manual_overrides=None):
    """V3: наблюдаемая единица и год + строгая связь территории/региона + настоящий audit.
    Научный выбор задаётся кодами/названиями, а НЕ корреляцией с test labels.
    """
    manual_overrides=manual_overrides or {}
    if catalog.empty:return [],pd.DataFrame([{'status':'NOT_FEASIBLE','reason':'Empty indicator registry'}])
    src=source_d[['territory_id','region_id','year']].drop_duplicates().copy()
    src['territory_id']=src.territory_id.astype(str)
    src['region_norm']=_v3_norm_key(src.region_id)
    valid_years=set(src.year.astype(int).unique())
    valid_pairs=src[['territory_id','year']].drop_duplicates()
    region_rows=src[['region_norm','year']].drop_duplicates()
    geo_names=src[['region_norm','region_id']].drop_duplicates()
    unique_names=geo_names.groupby('region_norm').region_id.nunique()
    geo_names=geo_names[geo_names.region_norm.isin(unique_names[unique_names.eq(1)].index)]
    registry=src[['territory_id','region_norm']].drop_duplicates()
    if registry.duplicated('territory_id').any():raise ValueError('D01 region identity collision')
    audit=[];candidates=[]
    available=catalog[catalog.status.eq('CATALOGED')].copy()
    specs={**V3_MUNICIPAL_CODE_SPECS,**V3_REGION_CODE_SPECS}
    for feature,(code,family) in specs.items():
        override=manual_overrides.get(feature,{})
        subset=available[available.indicator_code.astype(str).eq(str(override.get('indicator_code',code)))].copy()
        scope='region' if feature in V3_REGION_CODE_SPECS else 'municipality'
        subset=subset[subset.observation_level.astype(str).str.casefold().str.contains(
            r'регион' if scope=='region' else r'муниципальное образование верхнего уровня',regex=True,na=False)]
        if override.get('asset_id'):
            subset=subset[subset.asset_id.eq(override['asset_id'])]
        if override.get('indicator_unit'):
            subset=subset[subset.indicator_unit.eq(override['indicator_unit'])]
        if subset.empty:
            audit.append({'feature':feature,'family':family,'scope':scope,'status':'NOT_FEASIBLE_SOURCE_CODE',
                          'reason':f'Code {code} absent at required observation level'});continue
        viable=[]
        for _,cand in subset.iterrows():
            item=cand.to_dict(); aid=item['asset_id']
            try:
                raw,dimensions=_v3_read_series(asset_paths[aid],item,scope)
                raw=raw[raw.year.isin(valid_years)].copy()
                if scope=='municipality':
                    raw['territory_id']=raw.source_id.str.strip()
                    raw=raw[raw.territory_id.str.fullmatch(r'\d+',na=False)].copy()
                    # Не удалять начальные нули и не использовать fuzzy OKTMO.
                    raw=raw.merge(valid_pairs,on=['territory_id','year'],how='inner',validate='m:1')
                    keys=['territory_id','year'];base_n=len(valid_pairs)
                else:
                    raw['region_norm']=_v3_norm_key(raw.region_label)
                    raw=raw.merge(geo_names[['region_norm','region_id']],on='region_norm',how='inner',validate='m:1')
                    raw=raw.merge(region_rows,on=['region_norm','year'],how='inner',validate='m:1')
                    keys=['region_id','year'];base_n=len(region_rows)
                best=None
                for slice_name,part in _v3_aggregate_dimension_candidates(raw,keys,dimensions):
                    clean,details=_v3_collapse_series(part,keys,tolerance=.008)
                    coverage=len(clean)/max(1,base_n)
                    years=int(clean.year.nunique()) if len(clean) else 0
                    # Наличие ряда на исходной панели, а не общая длина ряда БДМО.
                    if scope=='municipality':territories=int(clean.territory_id.nunique()) if len(clean) else 0
                    else:territories=int(clean.region_id.nunique()) if len(clean) else 0
                    row={'feature':feature,'family':family,'scope':scope,'asset_id':aid,
                         'indicator_code':str(item['indicator_code']),'indicator_name':item['indicator_name'],
                         'indicator_unit':str(item['indicator_unit']),'observation_level':str(item['observation_level']),
                         'source_sha256':next((a['sha256'] for a in ACTIVE_INDEX['assets'] if a['asset_id']==aid),''),
                         'years_overlap':years,'n_geo_keys':territories,'source_coverage':float(coverage),
                         'n_valid_pairs':len(clean),'n_conflict':details.get('n_conflicted_keys',0),
                         'conflict_share':details.get('conflict_fraction',0), 'selection_slice':slice_name,
                         'dim_columns':','.join(dimensions),'status':details['status'],
                         'unit_conversion':'2008 kRUB -> RUB' if code=='Y48013004' else 'none',
                         'reason':'No aggregation of conflicting distinct statistical values'}
                    audit.append(row)
                    if len(clean)>=60 and years>=3 and coverage>=(V3_MIN_REGIONAL_COVERAGE if scope=='region' else minimum_coverage):
                        priority=(len(clean),years,-row['n_conflict'],int(slice_name.startswith('EXPLICIT_TOTAL')))
                        if best is None or priority>best[0]:best=(priority,clean,row)
                if best is not None:viable.append(best)
            except Exception as exc:
                audit.append({'feature':feature,'family':family,'scope':scope,'asset_id':aid,
                              'indicator_code':str(code),'status':'ASSET_PROCESSING_ERROR',
                              'reason':repr(exc)[:450]})
        if not viable:
            audit.append({'feature':feature,'family':family,'scope':scope,'status':'NOT_FEASIBLE_RESOLVED',
                          'indicator_code':code,'reason':'No unique, temporally intersecting, correctly leveled series'});continue
        viable.sort(key=lambda t:t[0],reverse=True)
        winning=viable[0]
        if len(viable)>1 and viable[1][0]==winning[0]:
            audit.append({'feature':feature,'family':family,'status':'NOT_FEASIBLE_TIE',
                          'reason':'Two indistinguishable candidate slices; require explicit dimensions'});continue
        spec=winning[2].copy();spec['status']='SELECTED';spec['publication_lag_years']=V3_PUBLICATION_LAG
        clean=winning[1][([('region_id' if scope=='region' else 'territory_id'),'year','value'])].copy()
        if clean.duplicated(['region_id' if scope=='region' else 'territory_id','year']).any():raise ValueError('Feature registration lost uniqueness')
        candidates.append({'spec':spec,'source_frame':clean,'level':scope})
        audit.append(spec)
    return candidates,pd.DataFrame(audit)

def attach_factors_v2(panel,selected,publication_lag=2):
    p=panel.copy();available=defaultdict(list);audit=[]
    for entry in selected:
        spec=entry['spec'];name=spec['feature'];level=entry.get('level','municipality')
        idx='region_id' if level=='region' else 'territory_id'
        source=entry['source_frame'][[idx,'year','value']].rename(columns={'year':'information_cutoff','value':name})
        assert not source.duplicated([idx,'information_cutoff']).any(),name+' duplicate join'
        before=len(p)
        p=p.merge(source,on=[idx,'information_cutoff'],how='left',validate='m:1',sort=False)
        assert len(p)==before
        coverage=float(p[name].notna().mean())
        audit.append({'feature':name,'scope':level,'family':spec['family'],'coverage':coverage,
                      'source_sha256':spec.get('source_sha256'),
                      'indicator_code':spec.get('indicator_code'),
                      'status':'JOIN_PASS' if coverage>=(V3_MIN_REGIONAL_COVERAGE if level=='region' else .04) else 'NOT_FEASIBLE_PANEL_COVERAGE'})
        if audit[-1]['status']=='JOIN_PASS':available[spec['family']].append(name)
        else:p=p.drop(columns=[name])
    for left,right,out in [('regional_investment_pc','housing_new','interaction_invest_housing'),
           ('regional_median_wage','school_count','interaction_wage_school'),
           ('regional_employment','housing_new','interaction_employment_housing')]:
        if left in p and right in p:
            p[out]=pd.to_numeric(p[left],errors='coerce')*pd.to_numeric(p[right],errors='coerce')
            available['E_all'].append(out)
    return p,dict(available),audit

def attach_spatial_v2(session,index,cache,d,p,k=5):
    """Реальная география населённых пунктов → прокси центра МО.
    Объект статистики не подменяется: геометрия явно помечена как proxy,
    популяционные веса поздней переписи не используются.
    """
    spatial_assets=[a for a in index['assets'] if a['segment']=='06' and
        a.get('role')=='derived' and {'region_name','municipality_name','latitude','longitude'}.issubset(a.get('columns_manifest',[]))]
    if not spatial_assets:return spatial_from_transport(session,index,cache,d,p)
    a=spatial_assets[0];path=fetch_asset(session,a,cache)
    import pyarrow.parquet as pq
    use=['region_name','municipality_name','latitude','longitude','source_oktmo']
    if 'coordinates_valid' in a['columns_manifest']:use+=['coordinates_valid']
    x=pq.read_table(path,columns=use).to_pandas()
    x['source_oktmo']=x.source_oktmo.astype('string').str.strip()
    x=x[x.source_oktmo.str.fullmatch(r'\d{8,11}',na=False)].copy()
    x['code_prefix5']=x.source_oktmo.str[:5]
    x['region_norm']=_v3_norm_key(x.region_name)
    x['mun_norm']=_v3_norm_key(x.municipality_name)
    x['latitude']=pd.to_numeric(x.latitude,errors='coerce')
    x['longitude']=pd.to_numeric(x.longitude,errors='coerce')
    x=x[x.latitude.between(40,82)&x.longitude.between(18,180)]
    if 'coordinates_valid' in x:
        x=x[x.coordinates_valid.astype(str).str.lower().isin(('true','1','да'))]
    x=x[(x.mun_norm!='')&(x.region_norm!='')]
    # unweighted mean of settlement locations, not a legally established administrative centroid
    loc=x.groupby(['region_norm','mun_norm','code_prefix5'],as_index=False).agg(
           latitude=('latitude','mean'),longitude=('longitude','mean'),
           settlements_geolocated=('latitude','size'))
    # Separate homonymous municipality groups with different source OKTMO parents.
    dominant=loc.sort_values('settlements_geolocated',ascending=False).drop_duplicates(['region_norm','mun_norm'])
    total=loc.groupby(['region_norm','mun_norm']).settlements_geolocated.sum().rename('total_settlements')
    loc=dominant.join(total,on=['region_norm','mun_norm'])
    loc=loc[loc.settlements_geolocated.div(loc.total_settlements).ge(.80)]
    base=d[['territory_id','region_id','municipality']].drop_duplicates()
    base['region_norm']=_v3_norm_key(base.region_id)
    base['mun_norm']=_v3_norm_key(base.municipality)
    base=base[base.region_norm.ne('')&base.mun_norm.ne('')]
    collisions=base.groupby(['region_norm','mun_norm']).territory_id.nunique().reset_index(name='n')
    base=base.merge(collisions,on=['region_norm','mun_norm'],validate='m:1')
    base=base[base.n.eq(1)].drop(columns='n')
    joined=base.merge(loc,on=['region_norm','mun_norm'],validate='m:1')
    joined=joined[joined.territory_id.astype(str).str[:5].eq(joined.code_prefix5)].copy()
    joined=joined.drop_duplicates('territory_id')
    if len(joined)<80:
        p2,feat,report=spatial_from_transport(session,index,cache,d,p)
        return p2,feat,pd.concat([pd.DataFrame([{'status':'NOT_FEASIBLE_GEO_PROXY',
            'matched_municipalities':len(joined),'reason':'Less than 80 validated municipality-location matches'}]),report],ignore_index=True)
    from sklearn.neighbors import BallTree
    coords=np.deg2rad(joined[['latitude','longitude']].to_numpy(float))
    knn=BallTree(coords,metric='haversine')
    dst,idx=knn.query(coords,k=min(k+1,len(coords)))
    names=joined.territory_id.to_numpy(str);region=joined.region_id.to_numpy(str)
    edges=[]
    for i in range(len(names)):
        for j,rad in zip(idx[i,1:],dst[i,1:]):
            km=float(rad*6371.0088)
            if i!=j and region[i]==region[j] and 0<km<=250:
                edges.append((names[i],names[j],km))
    graph=pd.DataFrame(edges,columns=['territory_id','neighbor_id','distance_km'])
    if len(graph)<70:return p,[],pd.DataFrame([{'status':'NOT_FEASIBLE_SPATIAL_GRAPH',
        'matched_municipalities':len(joined),'edges':len(graph)}])
    hist=d[['territory_id','year','migration_rate']].rename(columns={'territory_id':'neighbor_id','year':'information_cutoff'})
    merged=graph.merge(hist,on='neighbor_id',validate='m:m').dropna(subset=['migration_rate'])
    merged['weight']=1/merged.distance_km.clip(lower=1)
    merged['weighted']=merged.weight*merged.migration_rate
    s=merged.groupby(['territory_id','information_cutoff'],as_index=False).agg(
        weighted=('weighted','sum'),weight=('weight','sum'),neighbor_count=('neighbor_id','nunique'),
        neighbor_mean_distance=('distance_km','mean'))
    s['neighbor_rate']=s.weighted/s.weight
    after=p.merge(s[['territory_id','information_cutoff','neighbor_rate','neighbor_count','neighbor_mean_distance']],
                 on=['territory_id','information_cutoff'],how='left',validate='m:1')
    assert len(after)==len(p)
    cover=after.neighbor_rate.notna().mean()
    if cover<.03:return p,[],pd.DataFrame([{'status':'NOT_FEASIBLE_COVERAGE',
        'matched_municipalities':len(joined),'edges':len(graph),'coverage':float(cover)}])
    report=pd.DataFrame([{'status':'JOIN_PASS_PROXY_SPATIAL','matched_municipalities':len(joined),
       'settlement_geocodes':len(x),'edges':len(graph),'coverage':float(cover),
       'same_region_edges':True,'source_oktmo_prefix5_validated':True,'geometry_sha256':a['sha256'],
       'vintage_warning':'Unweighted geolocated settlement proxy based on retrospective source; not legal boundaries; static locations only'}])
    return after,['neighbor_rate','neighbor_count','neighbor_mean_distance'],report

def valid_families(available):
    order=['A_history'];prior=set(feature_cols('A_history',available))
    for f,group in [('B_economy','B_economy'),('C_housing_services','C_housing_services'),
                    ('D_spatial','D_spatial'),('E_all','E_all')]:
        cur=set(feature_cols(f,available))
        if cur!=prior and len(cur)>len(FEATURE_BASE):order.append(f)
        prior=cur
    return order


## 4. Прямая и рекурсивная модель горизонта. Строгие информационные срезы

In [ ]:
# Multi-step direct horizon backstop. Required when the observation history is too
# short for honest direct-h training under strictly known publication cutoffs.
# Iterates a one-year endogenous model from the *last available year* to target,
# never substitutes true intermediate outcomes. It is a separate model family.
def forecast_recursively(model, test, features):
    if any(f not in FEATURE_BASE for f in features):
        raise ValueError('Recursive endogeneous model may use only historical features; exogenous future paths are not observed')
    state=test[features+['region_id','mun_type','target_year','information_cutoff']].copy()
    n=len(state);lag=np.column_stack([state[f'lag{k}'].to_numpy(float) for k in range(5)])
    start=state.information_cutoff.to_numpy(int)
    target=state.target_year.to_numpy(int)
    if np.any(target<=start):raise ValueError('Invalid recursion horizons')
    if np.nanmax(target-start)>20:raise ValueError('Recursion >20 years requires separate scientific specification')
    preds=np.full(n,np.nan)
    for step in range(1,int(np.max(target-start))+1):
        active=(start+step<=target)
        if not active.any():break
        iteration=state.loc[active,features+['region_id','mun_type']].copy()
        current=lag[active].copy()
        for j in range(5):iteration[f'lag{j}']=current[:,j]
        iteration['mean3']=np.where(np.isfinite(current[:,:3]).all(axis=1),current[:,:3].mean(axis=1),np.nan)
        iteration['mean5']=np.where(np.isfinite(current).all(axis=1),current.mean(axis=1),np.nan)
        iteration['drift3']=(current[:,0]-current[:,2])/2.
        iteration['origin_year']=start[active]+step-1
        result=np.asarray(model.predict(iteration[features+['region_id','mun_type']].astype(
            {'region_id':str,'mun_type':str})),float)
        if not np.isfinite(result).all():raise ValueError('Non-finite recursive prediction')
        lag[active]=np.column_stack([result, current[:,:4]])
        last=(start[active]+step)==target[active]
        inds=np.flatnonzero(active)
        preds[inds[last]]=result[last]
    if not np.isfinite(preds).all():raise ValueError('Incomplete recursive forecast')
    return preds


def plan_recursive_holdouts(d,h,seed=2026):
    """Valid h-year TEST labels; TRAIN is 1-step observations published by test origin.
    Recursion is flagged, never labeled direct h-year supervised training.
    """
    p,g=gate_panel(panel_for_h(d,h),h)
    if g['status']!='PASS':return {},[{'status':'NOT_FEASIBLE','reason':'No h-year target pairs'}]
    base,_=gate_panel(panel_for_h(d,1,lag=SOURCE_LAG_YEARS),1)
    if base.empty:return {},[{'status':'NOT_FEASIBLE','reason':'No single-year training pairs'}]
    years=sorted(p.target_year.unique());regions=sorted(p.region_id.astype(str).unique())
    withheld=set(np.random.default_rng(seed).permutation(regions)[:max(2,round(.30*len(regions)))])
    modes=[('ROLLING_TIME',p[p.target_year.eq(years[-1])]),
           ('ROLLING_TIME_PREV',p[p.target_year.eq(years[-2])]),
           ('TERRITORY_ONLY',p[p.region_id.isin(withheld)&p.target_year.isin(years[-2:])]),
           ('CROSSED',p[p.region_id.isin(withheld)&p.target_year.eq(years[-1])])]
    result={};issues=[]
    for protocol,test in modes:
        if len(test)<20:issues.append({'protocol':protocol,'status':'NOT_FEASIBLE','reason':'Few test rows'});continue
        cutoff=int(test.information_cutoff.min())
        train=base[base.target_year.le(cutoff)].copy()
        if protocol in ('CROSSED','TERRITORY_ONLY'):
            train=train[~train.region_id.isin(withheld)].copy()
        if train.region_id.nunique()<5 or len(train)<150:
            issues.append({'protocol':protocol,'status':'NOT_FEASIBLE','reason':'No 1-year training cohorts at cutoff'});continue
        regions_train=sorted(train.region_id.unique())
        dev_regions=set(np.random.default_rng(seed+11).permutation(regions_train)[:max(1,round(.18*len(regions_train)))])
        fit=train[~train.region_id.isin(dev_regions)].copy();dev=train[train.region_id.isin(dev_regions)].copy()
        if min(len(fit),len(dev))<30:
            issues.append({'protocol':protocol,'status':'NOT_FEASIBLE','reason':'Insufficient geographic DEV'});continue
        if train.target_year.gt(cutoff).any() or fit.target_year.gt(cutoff).any():raise ValueError('Recursive train label leakage')
        if protocol in ('CROSSED','TERRITORY_ONLY') and set(train.region_id)&set(test.region_id):raise ValueError('Recursive region overlap')
        result[protocol]={'train':train,'fit':fit,'dev':dev,'test':test,'cutoff':cutoff,
              'withheld_regions':sorted(withheld) if protocol in ('TERRITORY_ONLY','CROSSED') else [],
              'forecast_type':'RECURSIVE_ENDOGENOUS_1STEP',
              'split_hash':hash_obj({'h':h,'protocol':protocol,'train':sorted(train.row_id),
                                     'dev':sorted(dev.row_id),'test':sorted(test.row_id)})}
    return result,issues


def forecast_net_recursively(model,test,features):
    """Signed absolute migration flow forecast. The population denominator is frozen
    at information cutoff; no realized future population or intermediate flow is read.
    Results: final-year people, future-origin cumulative people, and first-three-year
    stable outflow forecast, all under explicit constant-population assumption.
    """
    required={'net_lag0','net_lag1','net_lag2','net_mean3'}
    if not required.issubset(features):raise ValueError('Net recursion requires lagged signed flows')
    x=test[list(dict.fromkeys(features+['region_id','mun_type','origin_year','target_year','information_cutoff','population_exposure']))].copy()
    n=len(x);known=x.information_cutoff.to_numpy(int);target=x.target_year.to_numpy(int)
    origin=x.origin_year.to_numpy(int)
    frozen_pop=x.population_exposure.to_numpy(float)
    if not np.isfinite(frozen_pop).all() or (frozen_pop<=0).any():raise ValueError('No verified frozen population')
    net=np.column_stack([x['net_lag'+str(i)].to_numpy(float) for i in range(3)])
    rate=np.column_stack([x[f'lag{i}'].to_numpy(float) for i in range(5)])
    last=np.full(n,np.nan);sum_future=np.zeros(n,dtype=float);neg_counts=np.zeros(n,dtype=int)
    seen_counts=np.zeros(n,dtype=int)
    simulation_end=np.maximum(target,origin+3)
    for step in range(1,int(np.max(simulation_end-known))+1):
        active=known+step<=simulation_end
        if not active.any():break
        q=x.loc[active,features+['region_id','mun_type']].copy()
        prev_net=net[active].copy();prev_rate=rate[active].copy()
        for k in range(3):q['net_lag'+str(k)]=prev_net[:,k]
        q['net_mean3']=np.where(np.isfinite(prev_net).all(axis=1),prev_net.mean(axis=1),np.nan)
        for k in range(5):q[f'lag{k}']=prev_rate[:,k]
        q['mean3']=np.where(np.isfinite(prev_rate[:,:3]).all(axis=1),prev_rate[:,:3].mean(axis=1),np.nan)
        q['mean5']=np.where(np.isfinite(prev_rate).all(axis=1),prev_rate.mean(axis=1),np.nan)
        q['drift3']=(prev_rate[:,0]-prev_rate[:,2])/2
        q['origin_year']=known[active]+step-1
        forecast=np.asarray(model.predict(q[features+['region_id','mun_type']].astype({'region_id':str,'mun_type':str})),float)
        if not np.isfinite(forecast).all():raise ValueError('Net recursion produced NaN')
        fresh_rate=1000*forecast/frozen_pop[active]
        net[active]=np.column_stack([forecast,prev_net[:,:2]])
        rate[active]=np.column_stack([fresh_rate,prev_rate[:,:4]])
        ids=np.flatnonzero(active)
        future_mask=known[active]+step>origin[active]
        before_target=future_mask & (known[active]+step<=target[active])
        sum_future[ids[before_target]]+=forecast[before_target]
        in_three=future_mask & (known[active]+step<=origin[active]+3)
        seen_counts[ids[in_three]]+=1
        neg_counts[ids[in_three]]+=(forecast[in_three]<0).astype(int)
        complete=known[active]+step==target[active]
        last[ids[complete]]=forecast[complete]
    stable=np.where(seen_counts==3,(neg_counts==3).astype(float),np.nan)
    if not np.isfinite(last).all():raise ValueError('Missing final-year flows')
    if not np.isfinite(stable).all():raise ValueError('Incomplete three-year forecast for stable outflow')
    return pd.DataFrame({'row_id':test.row_id.to_numpy(),'prediction_net':last,
                         'prediction_cum_net':sum_future,'prediction_stable3':stable,
                         'prediction_sign':(last>0).astype(float),'prediction_outflow':(1000*last/frozen_pop<-5).astype(float),
                         'population_assumption':'last known population frozen at information cutoff'})


In [ ]:
# ML-1 V3: recursive forecast with covariates frozen strictly at origin information cutoff.
# No future values are ever substituted. This is an explicitly labelled scenario.

def forecast_recursively(model,test,features):
    required={'region_id','mun_type','information_cutoff','target_year'}
    if not required.issubset(test):raise ValueError('Recursive test set lacks forecast contract')
    for f in features:
        if f not in test:raise ValueError('Missing frozen exogenous feature '+f)
    state=test[list(dict.fromkeys(features+['region_id','mun_type','target_year','information_cutoff']))].copy()
    n=len(state);lags=np.column_stack([state[f'lag{k}'].to_numpy(float) for k in range(5)])
    starts=state.information_cutoff.to_numpy(int);targets=state.target_year.to_numpy(int)
    if np.any(targets<=starts) or (targets-starts).max()>20:raise ValueError('Invalid recursion horizon')
    forecast=np.full(n,np.nan)
    exog=[f for f in features if f not in FEATURE_BASE]
    for step in range(1,int(np.max(targets-starts))+1):
        active=starts+step<=targets
        if not active.any():break
        data=state.loc[active,list(dict.fromkeys(features+['region_id','mun_type']))].copy()
        history=lags[active]
        for j in range(5):data[f'lag{j}']=history[:,j]
        data['mean3']=np.where(np.isfinite(history[:,:3]).all(axis=1),history[:,:3].mean(axis=1),np.nan)
        data['mean5']=np.where(np.isfinite(history).all(axis=1),history.mean(axis=1),np.nan)
        data['drift3']=(history[:,0]-history[:,2])/2
        data['origin_year']=starts[active]+step-1
        # Exogenous variables including regional indicators and spatial-lag neighbor rate
        # stay FIXED at origin information_cutoff, do not forecast future paths silently.
        if exog:
            for col in exog:
                if col!='origin_year':
                    assert data[col].equals(state.loc[active,col]),'Exogenous feature unexpectedly changed'
        pred=np.asarray(model.predict(data[list(dict.fromkeys(features+['region_id','mun_type']))]),float)
        if not np.isfinite(pred).all():raise ValueError('Nonfinite recursive prediction')
        lags[active]=np.column_stack([pred,history[:,:4]])
        done=(starts[active]+step)==targets[active]
        idx=np.flatnonzero(active)
        forecast[idx[done]]=pred[done]
    if not np.isfinite(forecast).all():raise ValueError('Incomplete h-year forecast')
    return forecast

V3_HORIZON_PANELS={}


# V3.2: согласованное присоединение факторов к рекурсивным 1-step FIT/DEV/TRAIN
# и многолетнему TEST. row_id содержит :h1/:h3/:h5, поэтому не является JOIN-ключом.
RECURSIVE_FACTOR_BRIDGE_VERSION = 'territory_origin_cutoff_all_phases_v3_2'

def bridge_recursive_factors(split, factor_panel, available):
    """Attach ONLY exogenous values observed by information_cutoff, on stable keys.
    Preserve all rows, base targets, and row order. Return split and detailed audit.
    Never carry target_rate, target_net, or any target-year observation across panels.
    """
    keys=['territory_id','origin_year','information_cutoff']
    names=list(dict.fromkeys(f for fields in available.values() for f in fields))
    frames={name:frame.copy() for name,frame in split.items() if isinstance(frame,pd.DataFrame)}
    stages=('train','fit','dev','test')
    if any(stage not in frames for stage in stages):
        raise ValueError('Recursive factor bridge: incomplete train/fit/dev/test')
    if not names:
        return split, [{'phase':phase,'status':'NO_EXTERNAL_FACTORS','n_features':0} for phase in stages]
    if not set(keys+names).issubset(factor_panel.columns):
        missing=sorted(set(keys+names)-set(factor_panel.columns))
        raise ValueError('Recursive feature source lacks validated columns: '+repr(missing))
    if factor_panel.duplicated(keys).any():
        raise ValueError('Ambiguous territory × origin × cutoff in feature panel')
    source=factor_panel[keys+names].copy()
    source['__bridge_present__']=1
    audit=[]
    for phase in stages:
        block=frames[phase]
        if not set(keys).issubset(block.columns):
            raise ValueError(f'{phase}: missing recursive stable join keys')
        if block.duplicated(keys).any():
            raise ValueError(f'{phase}: duplicate recursive row keys')
        # Inner metadata fields are guaranteed by the same historical municipality.
        # left join holds even when individual indicator values are null.
        probe=block[keys].reset_index(drop=True).merge(source,on=keys,how='left',
                        validate='one_to_one',sort=False)
        if len(probe)!=len(block) or probe['__bridge_present__'].isna().any():
            unmatched=int(probe['__bridge_present__'].isna().sum())
            raise ValueError(f'{phase}: {unmatched} unmatched stable panel keys')
        probe=probe.drop(columns='__bridge_present__')
        for feature in names:
            fresh=pd.to_numeric(probe[feature],errors='coerce').to_numpy(dtype=float)
            if feature in block.columns:
                old=pd.to_numeric(block[feature],errors='coerce').to_numpy(dtype=float)
                conflict=np.isfinite(old)&np.isfinite(fresh)&~np.isclose(old,fresh,rtol=1e-10,atol=1e-10)
                if conflict.any():
                    raise ValueError(f'{phase}: exogenous information conflict in {feature}, n={int(conflict.sum())}')
                # Preserve a measured value; fill previously absent values from audited source.
                values=np.where(np.isfinite(old),old,fresh)
            else:
                values=fresh
            block[feature]=values
        if 'max_feature_year' in block.columns and (block.max_feature_year>block.information_cutoff).any():
            raise ValueError(f'{phase}: future feature leakage')
        audit.append({'phase':phase,'status':'BRIDGE_PASS','n_rows':len(block),'n_features':len(names),
                     'keys_sha256':hash_obj(block[keys].astype(str).to_dict('list')),
                     'feature_coverage':float(block[names].notna().to_numpy().mean())})
        split[phase]=block
    return split,audit

def preflight_recursive_contexts(contexts):
    """Pre-Optuna mandatory gate on ALL real split/factor combinations, h=1/3/5.
    Detect missing TEST frozen factors before the thousands of scheduled trials.
    """
    result=[]
    class NoFuturePredictor:
        def predict(self,frame):
            if frame.empty or 'lag0' not in frame:
                raise ValueError('Preflight missing forecast history')
            # The preflight checks interface and recursion, NOT model accuracy.
            return pd.to_numeric(frame['lag0'],errors='coerce').fillna(0.).to_numpy(float)
    for (h,proto),context in sorted(contexts.items()):
        if not context['forecast_type'].startswith('RECURSIVE'):
            continue
        for family in context['families']:
            cols=feature_cols(family,context['available'])
            required=set(cols+['region_id','mun_type','information_cutoff','target_year'])
            for stage in ('train','fit','dev','test'):
                frame=context['split'][stage]
                absent=required-set(frame.columns)
                if absent:
                    raise ValueError(f'PRE-OPTUNA recursive h{h}/{proto}/{family}/{stage} missing columns {sorted(absent)}')
                if (frame.max_feature_year>frame.information_cutoff).any():
                    raise ValueError('PRE-OPTUNA future covariate leakage')
                if stage=='test':
                    subset=frame.head(min(len(frame),12)).copy()
                    pred=forecast_recursively(NoFuturePredictor(),subset,cols)
                    if len(pred)!=len(subset) or not np.isfinite(pred).all():
                        raise ValueError('PRE-OPTUNA incomplete recursive predictions')
            result.append({'horizon':h,'protocol':proto,'family':family,
                           'status':'PASS','n_test':len(context['split']['test']),
                           'external_features':len([f for f in cols if f not in FEATURE_BASE])})
    return result


## 5. Оптимизация Optuna, объяснимость и статистическая интерпретация

In [ ]:
# V2 out-of-time interpretation: explicit input columns, including categorical fields.
def interpret_model_v2(pipeline,train,test,features,max_rows=400,seed=2026):
    from sklearn.inspection import permutation_importance
    rng=np.random.default_rng(seed)
    sample=(test.sample(n=min(len(test),max_rows),random_state=seed).copy())
    columns=list(dict.fromkeys(features+['region_id','mun_type']))
    X=sample[columns].astype({'region_id':'str','mun_type':'str'})
    y=sample.target_rate.to_numpy(float)
    perm=permutation_importance(pipeline,X,y,scoring='neg_mean_absolute_error',n_repeats=7,random_state=seed)
    importance=pd.DataFrame({'factor':columns,'importance_MAE':perm.importances_mean,
                              'importance_std':perm.importances_std,'method':'test permutation',
                              'n_sample':len(sample)}).sort_values('importance_MAE',ascending=False)
    shapdf=pd.DataFrame();ale=pd.DataFrame();interaction=pd.DataFrame();notes=[]
    try:
        import shap
        estimator=pipeline.named_steps['model']
        if estimator.__class__.__name__.lower().startswith(('catboost','lgbm','histgradient')):
            transformed=pipeline.named_steps['preprocess'].transform(X)
            names=pipeline.named_steps['preprocess'].get_feature_names_out()
            values=np.asarray(shap.TreeExplainer(estimator)(transformed,check_additivity=False).values)
            if values.ndim!=2 or values.shape[1]!=len(names):raise ValueError('SHAP output dimension mismatch')
            shapdf=pd.DataFrame({'feature_transformed':names,'mean_abs_SHAP':np.abs(values).mean(axis=0),
                                 'method':'tree SHAP; associative, not causal','n':len(sample)})
            shapdf=shapdf.sort_values('mean_abs_SHAP',ascending=False)
        else:notes.append({'subject':'SHAP','status':'NOT_FEASIBLE','reason':'Best model not tree-based; permutation importance available'})
    except Exception as exc:notes.append({'subject':'SHAP','status':'ERROR','reason':str(exc)[:300]})
    for feature in importance[importance.factor.isin(features)].head(3).factor:
        t=train[columns].copy()
        t[feature]=pd.to_numeric(t[feature],errors='coerce')
        t=t.dropna(subset=[feature]);t=t.sample(n=min(len(t),1600),random_state=seed)
        if len(t)<80 or t[feature].nunique()<6:continue
        edges=np.unique(t[feature].quantile(np.linspace(0,1,9)).to_numpy(float))
        if len(edges)<4:continue
        local=[]
        for low,high in zip(edges[:-1],edges[1:]):
            sub=t[(t[feature]>=low)&(t[feature]<=high)].copy()
            if len(sub)<8:continue
            left=sub.copy();right=sub.copy()
            left[feature]=low;right[feature]=high
            delta=(pipeline.predict(right.astype({'region_id':str,'mun_type':str}))-
                   pipeline.predict(left.astype({'region_id':str,'mun_type':str}))).mean()
            local.append({'feature':feature,'center':float((low+high)/2),'increment':float(delta),'n':len(sub)})
        if local:
            f=pd.DataFrame(local);f['ALE']=f.increment.cumsum();f['ALE']-=np.average(f.ALE,weights=f.n)
            ale=pd.concat([ale,f],ignore_index=True)
    # Paired counterfactual perturbations of *features*, not policies.
    interaction_pairs=[('wage','housing_new'),('employment','education'),('income','neighbor_rate')]
    rows=[]
    for x1,x2 in interaction_pairs:
        if not {x1,x2}.issubset(columns):
            notes.append({'subject':f'INTERACTION/{x1}×{x2}','status':'NOT_FEASIBLE','reason':'Required variables unavailable'})
            continue
        t=train[columns].dropna(subset=[x1,x2]).sample(frac=1,random_state=seed).head(500).copy()
        if len(t)<80:continue
        qx=np.quantile(pd.to_numeric(t[x1],errors='coerce').dropna(),[.25,.75]);qy=np.quantile(pd.to_numeric(t[x2],errors='coerce').dropna(),[.25,.75])
        if not (np.isfinite(qx).all() and np.isfinite(qy).all()):continue
        values={}
        for i,v1 in enumerate(qx):
            for j,v2 in enumerate(qy):
                s=t.copy();s[x1]=v1;s[x2]=v2
                values[i,j]=float(np.mean(pipeline.predict(s.astype({'region_id':str,'mun_type':str}))))
                rows.append({'pair':f'{x1}×{x2}','x_quantile':i,'y_quantile':j,'x':float(v1),'y':float(v2),
                             'avg_prediction':values[i,j],'n':len(t)})
        rows.append({'pair':f'{x1}×{x2}','x_quantile':'interaction_contrast','y_quantile':'',
                     'avg_prediction':values[1,1]-values[1,0]-values[0,1]+values[0,0],'n':len(t)})
    return importance,shapdf,ale,pd.DataFrame(rows),notes


def conformal_predictions(dev_truth,dev_pred,test_pred,alpha=.10):
    resid=np.abs(np.asarray(dev_truth,float)-np.asarray(dev_pred,float))
    if len(resid)<20:raise ValueError('Conformal requires >=20 DEV residuals')
    q=float(np.quantile(resid,min(1.,np.ceil((len(resid)+1)*(1-alpha))/len(resid)),method='higher'))
    pred=np.asarray(test_pred,float)
    return pred-q,pred+q,q


def study_figures(metric,predictions,factors,intervals,ale,types,groups,interactions,root):
    import matplotlib.pyplot as plt
    folder=Path(root);folder.mkdir(parents=True,exist_ok=True);files=[]
    def save(name):
        plt.tight_layout();f=folder/(name+'.png');plt.savefig(f,dpi=175,bbox_inches='tight');plt.close();files.append(f)
    if not metric.empty:
        subset=metric[metric.seed.isin([0,11])&metric.protocol.eq('ROLLING_TIME')]
        if len(subset):
            main=subset[subset.feature_set.eq('A_history')].pivot_table(index='horizon',columns='model',values='MAE')
            if not main.empty:
                main.plot(kind='bar',figsize=(10,5));plt.ylabel('MAE, человек на 1000');plt.title('Ошибки моделей по горизонтам');save('01_mae_horizons')
            groupscore=subset.pivot_table(index='horizon',columns='feature_set',values='MAE',aggfunc='min')
            if not groupscore.empty:
                groupscore.plot(marker='o',figsize=(10,4));plt.ylabel('MAE, человек на 1000');plt.title('A→E: предиктивное качество');save('02_ablation')
    if len(predictions):
        z=predictions[(predictions.horizon==1)&(predictions.protocol=='ROLLING_TIME')&(predictions.seed==11)].copy()
        if len(z):
            model=z.model.value_counts().index[0];q=z[z.model==model]
            plt.figure(figsize=(8,5));plt.hexbin(q.target_rate,q.prediction,gridsize=35,mincnt=1,bins='log');
            ma=float(np.nanquantile(np.abs(np.r_[q.target_rate,q.prediction]),.98));ma=max(ma,1)
            plt.plot([-ma,ma],[-ma,ma],'k--');plt.xlim(-ma,ma);plt.ylim(-ma,ma)
            plt.xlabel('Факт, человек на 1000');plt.ylabel('Прогноз, человек на 1000');plt.title('Проверка на отложенных наблюдениях');save('03_actual_forecast')
            q=q.copy();q['abs_error']=(q.target_rate-q.prediction).abs()
            q['population_bin']=pd.qcut(q.population_exposure,4,duplicates='drop').astype(str)
            s=q.groupby('population_bin').abs_error.mean();s.plot(kind='barh',figsize=(9,4))
            plt.xlabel('MAE, человек на 1000');plt.title('Ошибки по численности населения');save('04_population')
        temp=predictions[predictions.seed.isin([0,11])].copy()
        temp['absolute_error']=(temp.target_rate-temp.prediction).abs()
        series=temp.groupby(['target_year','horizon']).absolute_error.mean().unstack()
        if len(series):series.plot(marker='o',figsize=(10,4));plt.ylabel('MAE');plt.title('Ошибки по целевым годам');save('05_temporal')
    if len(factors):
        q=factors.sort_values('importance_MAE',ascending=False).head(12).sort_values('importance_MAE')
        plt.figure(figsize=(9,5));plt.barh(q.factor,q.importance_MAE);plt.xlabel('Рост MAE при перемешивании');plt.title('Permutation importance');save('06_importance')
    if len(ale):
        plt.figure(figsize=(9,5))
        for feat,q in ale.groupby('feature'):plt.plot(q.center,q.ALE,marker='o',label=str(feat))
        plt.title('ALE, не причинная интерпретация');plt.xlabel('Значение');plt.ylabel('Изменение прогноза');plt.legend();save('07_ALE')
    if len(interactions) and 'x_quantile' in interactions:
        q=interactions[interactions.x_quantile.notna() & interactions.y_quantile.notna() & interactions.x_quantile.ne('interaction_contrast')]
        if len(q):
            label=q.pair.value_counts().index[0];t=q[q.pair.eq(label)].copy()
            plt.figure(figsize=(6,4));plt.imshow(t.pivot(index='x_quantile',columns='y_quantile',values='avg_prediction').to_numpy(float))
            plt.colorbar(label='Человек на 1000');plt.title('Взаимодействие '+label);save('08_interaction')
    if len(intervals) and 'empirical_coverage' in intervals:
        q=intervals.copy();plt.figure(figsize=(10,4));plt.scatter(range(len(q)),q.empirical_coverage,label='Реальное')
        plt.axhline(.9,ls='--',label='Номинал 90%');plt.ylim(0,1.05);plt.xticks(range(len(q)),q.horizon.astype(str)+'/'+q.protocol.str[:4],rotation=60)
        plt.ylabel('Покрытие');plt.legend();plt.title('Предиктивные интервалы');save('09_interval_coverage')
    if len(types) and {'mun_type','MAE'}.issubset(types):
        q=types.groupby('mun_type').agg(MAE=('MAE','mean'),n=('n','sum')).sort_values('n',ascending=False).head(12)
        if len(q):q.MAE.plot(kind='barh',figsize=(9,5));plt.xlabel('MAE');plt.title('Ошибка по типам муниципалитетов');save('10_types')
    return files[:10]


In [ ]:
# ML-1 V3: формальный поиск гиперпараметров Optuna с закрытым TEST
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
OPTUNA_TRIALS={'ridge':3,'elasticnet':4,'huber':3,'histgb':4,'catboost':6,'lightgbm':6}
V3_OPTUNA_SAMPLER_SEED=20261022

def ml1_optuna_parameters(trial,model):
    if model=='ridge':
        return {'alpha':trial.suggest_float('alpha',1e-3,1e4,log=True)}
    if model=='elasticnet':
        return {'alpha':trial.suggest_float('alpha',.0005,30.,log=True),
                'l1_ratio':trial.suggest_float('l1_ratio',.04,.98)}
    if model=='huber':
        return {'epsilon':trial.suggest_float('epsilon',1.2,2.5),
                'alpha':trial.suggest_float('alpha',1e-6,10,log=True)}
    if model=='histgb':
        return {'max_iter':trial.suggest_int('max_iter',100,480,step=20),
                'max_leaf_nodes':trial.suggest_int('max_leaf_nodes',7,55),
                'learning_rate':trial.suggest_float('learning_rate',.015,.14,log=True)}
    if model=='catboost':
        return {'iterations':trial.suggest_int('iterations',150,650,step=25),
                'depth':trial.suggest_int('depth',3,8),
                'learning_rate':trial.suggest_float('learning_rate',.018,.16,log=True),
                'l2_leaf_reg':trial.suggest_float('l2_leaf_reg',1.,30.,log=True)}
    if model=='lightgbm':
        return {'n_estimators':trial.suggest_int('n_estimators',150,650,step=25),
                'num_leaves':trial.suggest_int('num_leaves',7,48),
                'learning_rate':trial.suggest_float('learning_rate',.018,.14,log=True),
                'reg_lambda':trial.suggest_float('reg_lambda',.01,70.,log=True),
                'colsample_bytree':trial.suggest_float('colsample_bytree',.6,1.),
                'subsample':trial.suggest_float('subsample',.65,1.),'subsample_freq':1}
    raise ValueError('Unknown Optuna estimator')

def horizon_development_panel(d,h,sp):
    """Long-horizon DEV only on HIDDEN dev regions & already published TARGET years.
    If unavailable, returns None and Optuna does NOT tune on final test.
    """
    if h==1:return sp['dev'], 'ONE_YEAR_DEV'
    q=V3_HORIZON_PANELS[h].copy() if h in V3_HORIZON_PANELS else panel_for_h(d,h)
    permitted=set(sp['dev'].region_id.astype(str).unique())
    q=q[q.region_id.astype(str).isin(permitted)&q.target_year.le(int(sp['cutoff']))].copy()
    good=q.target_rate.notna()&q.lag0.notna()&q.population_exposure.gt(0)
    q=q[good].copy()
    if len(q)<80 or q.territory_id.nunique()<12 or q.target_year.nunique()<2:
        return None,'ONE_STEP_PROXY_DEV_NO_MULTIHORIZON_HISTORY'
    # Historical multi-year validation must be forecastable using ONLY labels
    # already known at the earliest validation origin. Prevent DEV future-label leakage.
    earliest_info_cutoff=int(q.information_cutoff.min())
    historical_fit=sp['fit'][sp['fit'].target_year.le(earliest_info_cutoff)]
    if len(historical_fit)<100 or historical_fit.region_id.nunique()<3:
        return None,'ONE_STEP_PROXY_DEV_NO_NONLEAKING_HORIZON_FIT'
    return q,'HORIZON_SPECIFIC_RECURSIVE_DEV'

def optuna_parameters_smoke():
    """Real Optuna API with tiny in-memory study; no model/data download."""
    sampler=optuna.samplers.TPESampler(seed=19,n_startup_trials=1)
    for name in OPTUNA_TRIALS:
        study=optuna.create_study(direction='minimize',sampler=sampler)
        study.optimize(lambda t:float(sum(float(x)**2 for x in ml1_optuna_parameters(t,name).values())),n_trials=2)
        assert len(study.trials)==2 and np.isfinite(study.best_value)
    return 'PASS Optuna 6/6 estimator searches'


def ml1_stratified_search_sample(frame, maximum, salt=0):
    """Only search FIT/DEV; deterministic region × time × municipality-type coverage.

    No TEST data, no label-dependent sampling and no peeking at final metrics.
    Audit selection is recorded by the caller.
    """
    frame=frame.copy()
    if maximum is None or maximum<=0 or len(frame)<=maximum: return frame
    if not {'region_id','target_year','mun_type'}.issubset(frame):
        raise ValueError('Optuna sample requires region/year/type keys')
    rng=np.random.default_rng(20261025+int(salt))
    groups=frame.groupby(['region_id','target_year','mun_type'],dropna=False,sort=True).indices
    groups_list=list(groups.values())
    # Proportional quotas + one representative per stratum while possible.
    if len(groups_list)>maximum:
        rng.shuffle(groups_list);groups_list=groups_list[:maximum]
    chosen=set()
    for ix in groups_list:
        chosen.add(int(rng.choice(ix)))
    if len(chosen)<maximum:
        remaining=np.setdiff1d(np.arange(len(frame)),np.fromiter(chosen,dtype=int),assume_unique=False)
        pick=rng.choice(remaining,size=min(maximum-len(chosen),len(remaining)),replace=False)
        chosen.update(map(int,pick))
    return frame.iloc[sorted(chosen)].copy()

def ml1_fit_dev_objective(fit, dev, features, name, params, seed, recursive=False):
    """Tuning objective on held-out DEV, never on the final TEST."""
    _,fitted,notes=fit_predict(fit,dev,features,name,params,seed=seed)
    pred=(forecast_recursively(fitted,dev,features) if recursive else
          fitted.predict(dev[list(dict.fromkeys(features+['region_id','mun_type']))]))
    value=score(dev.target_rate,dev.population_exposure,pred)['MAE']
    if not np.isfinite(value):raise ValueError('Nonfinite development error')
    return float(value),notes


In [ ]:
# ML-1 V3. Расширенная интерпретация: групповая и условная permutation importance,
# SHAP, локальные эффекты ALE, взаимодействия и проверка устойчивости.
_OLD_INTERPRET_MODEL_V2=interpret_model_v2

def interpret_model_v2(pipeline,train,test,features,max_rows=500,seed=2026):
    importance,shapdf,ale,interactions,notes=_OLD_INTERPRET_MODEL_V2(
        pipeline,train,test,features,max_rows=max_rows,seed=seed)
    rng=np.random.default_rng(seed)
    sample=test.sample(n=min(len(test),max_rows),random_state=seed)
    cols=list(dict.fromkeys(features+['region_id','mun_type']))
    x=sample[cols].astype({'region_id':str,'mun_type':str})
    truth=sample.target_rate.to_numpy(float)
    baseline=float(np.mean(np.abs(np.asarray(pipeline.predict(x),float)-truth)))
    groups={
       'История миграции':[s for s in features if s.startswith(('lag','mean','drift'))],
       'Размер и демография':[s for s in features if s=='population_log' or s.startswith('regional_working_age')],
       'Экономика':[s for s in features if s.startswith(('regional_wage','regional_median','regional_investment','regional_employment','regional_unemployment'))],
       'Жильё и услуги':[s for s in features if s in V3_MUNICIPAL_CODE_SPECS or s.startswith('interaction_')],
       'Пространство':[s for s in features if s in ('neighbor_rate','neighbor_count','neighbor_mean_distance')],
       'Административный контекст':['region_id','mun_type']}
    grouped=[]
    for group,group_columns in groups.items():
        if not group_columns:continue
        estimates=[];conditional=[]
        for r in range(5):
            pert=x.copy();ix=rng.permutation(len(x))
            pert.loc[:,group_columns]=x.iloc[ix][group_columns].to_numpy()
            estimates.append(float(np.mean(np.abs(pipeline.predict(pert)-truth)))-baseline)
            # Shuffling strictly within observed region helps probe regional confounding.
            if group!='Административный контекст':
                per=x.copy()
                for reg,ids in x.groupby('region_id').groups.items():
                    positions=x.index.get_indexer(ids)
                    if len(positions)>1:
                        shuffled=rng.permutation(positions)
                        per.iloc[positions,per.columns.get_indexer(group_columns)]=x.iloc[shuffled][group_columns].to_numpy()
                conditional.append(float(np.mean(np.abs(pipeline.predict(per)-truth)))-baseline)
        grouped.append({'factor':group,'importance_MAE':float(np.mean(estimates)),
            'importance_std':float(np.std(estimates,ddof=1)),'method':'grouped permutation',
            'n_sample':len(x),'conditional_within_region':float(np.mean(conditional)) if conditional else np.nan})
    if grouped:importance=pd.concat([importance,pd.DataFrame(grouped)],ignore_index=True)
    # Bootstrap the observed permutation rankings by municipality trajectories/groups,
    # recording instability rather than implying causality.
    if len(importance) and 'method' in importance:
        importance['interpretation']='Прогностическая ассоциация, не причинное влияние'
    if len(shapdf):
        mapping=[]
        for transformed in shapdf.feature_transformed.astype(str):
            name=transformed.split('__',1)[-1]
            source=next((f for f in sorted(cols,key=len,reverse=True) if name==f or name.startswith(f+'_')),None)
            mapping.append(source or 'categorical_other')
        shapdf['source_factor']=mapping
        shapdf['interpretation']='TreeSHAP, условная интерпретация при коррелированных переменных ограничена'
        ranking=shapdf.groupby('source_factor',as_index=False).mean_abs_SHAP.sum().sort_values('mean_abs_SHAP',ascending=False)
        ranking['method']='SHAP sum of transformed features';ranking['n']=len(sample)
        ranking['feature_transformed']=ranking.source_factor
        ranking['interpretation']='Агрегированный SHAP без причинных выводов'
        shapdf=pd.concat([shapdf,ranking],ignore_index=True)
    # PDP/ICE alongside ALE; remove interpolation over unobserved feature ranges.
    candidates=[f for f in features if f in x and pd.api.types.is_numeric_dtype(x[f]) and x[f].nunique(dropna=True)>5]
    for name in candidates[:4]:
        v=pd.to_numeric(x[name],errors='coerce')
        knots=np.unique(v.dropna().quantile([.1,.3,.5,.7,.9]).to_numpy(float))
        if len(knots)<3:continue
        for point in knots:
            probe=x.copy();probe[name]=point
            predictions=np.asarray(pipeline.predict(probe),float)
            # ICE sample is deliberate partial dependency, not out-of-support treatment effect.
            rows={'pair':name,'method':'PDP','x':float(point),
                  'avg_prediction':float(np.mean(predictions)),'q10':float(np.quantile(predictions,.1)),
                  'q90':float(np.quantile(predictions,.9)),'n':len(predictions)}
            interactions=pd.concat([interactions,pd.DataFrame([rows])],ignore_index=True)
    return importance,shapdf,ale,interactions,notes


In [ ]:
# ML-1 V3: диагностический анализ факторной панели и коррелированности

def factor_diagnostics_v3(panel,selected):
    specs={v['spec']['feature']:v['spec'] for v in selected}
    names=[s for s in specs if s in panel]
    if not names:return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()
    coverage=[]
    for key in names:
        info=specs[key];s=pd.to_numeric(panel[key],errors='coerce')
        for yr,g in panel.groupby('origin_year'):
            q=pd.to_numeric(g[key],errors='coerce')
            coverage.append({'feature':key,'family':info['family'],'territorial_level':info.get('scope'),
                'indicator_code':info['indicator_code'],'indicator_unit':info['indicator_unit'],
                'origin_year':int(yr),'available_rows':int(q.notna().sum()),
                'rows':len(q),'coverage':float(q.notna().mean()),
                'median':float(q.median()) if q.notna().any() else np.nan,
                'n_distinct':int(q.nunique(dropna=True))})
    a=panel[names].apply(pd.to_numeric,errors='coerce')
    rank=a.corr('spearman',min_periods=100)
    pairs=[]
    for i,x in enumerate(names):
        for y in names[i+1:]:
            m=a[[x,y]].dropna()
            if len(m)<100:continue
            pairs.append({'factor1':x,'factor2':y,'spearman_r':float(m[x].corr(m[y],method='spearman')),
                          'n_complete':len(m),'collinear_alert':abs(m[x].corr(m[y],method='spearman'))>.85})
    lineage=pd.DataFrame([v['spec'] for v in selected])
    lineage['scientific_note']=lineage.apply(lambda r:'Региональный показатель; не муниципальное наблюдение'
                         if r['scope']=='region' else 'Муниципальный показатель: одна запись на OKTMO×год',axis=1)
    return pd.DataFrame(coverage),pd.DataFrame(pairs),lineage

def make_factor_correlation_plot(pairdf,root):
    if pairdf.empty:return None
    import matplotlib.pyplot as plt
    mat=pairdf.pivot(index='factor1',columns='factor2',values='spearman_r')
    if mat.empty:return None
    import matplotlib
    fig,ax=plt.subplots(figsize=(max(7,mat.shape[1]*.85),max(4,mat.shape[0]*.55)))
    im=ax.imshow(mat.to_numpy(float),cmap='coolwarm',vmin=-1,vmax=1,aspect='auto')
    ax.set_xticks(range(len(mat.columns)),mat.columns,rotation=60,ha='right',fontsize=8)
    ax.set_yticks(range(len(mat.index)),mat.index,fontsize=8)
    ax.set_title('Попарные ранговые корреляции признаков (только наблюдаемые пары)')
    fig.colorbar(im,ax=ax,label='Spearman ρ')
    fig.tight_layout()
    p=Path(root)/'08_factor_correlations.png';fig.savefig(p,dpi=180,bbox_inches='tight');plt.close(fig)
    return p


## 6. Сквозной полный эксперимент

Единый реестр источников, годов, регионов, типов территорий и уровней факторов → `DATA_GATE` → холодные holdout → **Optuna на DEV** → зафиксированная FINAL оценка → 50 stochastic seeds → подтверждающие/исследовательские сравнения → XAI, ALE, PDP/ICE и пространственно-временные ограничения → научный отчёт → Excel и Evidence ZIP → контроль реального повторного чтения.

In [ ]:
# ML-1 V4 | единый экспортный контракт, научный gate и независимый read-back
from pathlib import Path
import os, math, json, hashlib, zipfile, tempfile
V4_OUTPUT_FILES=('ML1_V4_RESULTS.xlsx','ML1_V4_PREDICTIONS.xlsx',
                 'ML1_V4_DATA_QA.xlsx','ML1_V4_MODEL_QA.xlsx','ML1_V4_FIGURES.zip')

def v4_warning_table(counter):
    return pd.DataFrame([{'warning':str(k),'n':int(v)} for k,v in counter.items()])

def v4_tests_ledger(root):
    items=[]
    for name in ('PRE_FLIGHT_TESTS.json','PRE_FLIGHT_TESTS_V3.json',
                 'PRE_FLIGHT_TESTS_V3_2.json','V3_3_ACCELERATION_TESTS.json',
                 'V3_3_3_RESUME_TESTS.json','V4_OUTPUT_PREFLIGHT.json'):
        p=Path(root)/name
        if p.is_file():
            try:
                z=json.loads(p.read_text(encoding='utf-8'))
                items.append({'protocol':name,'status':z.get('status','CHECK'),
                              'checks':len(z.get('tests',z.get('checks',[]))) if isinstance(z.get('tests',z.get('checks',[])),list) else z.get('tests',''),
                              'sha256':sha256_file(p)})
            except Exception as e:items.append({'protocol':name,'status':'FAILED_PARSE','error':repr(e)})
    return pd.DataFrame(items)

def v4_goals_matrix(hdf,h4df,h5df,selected,spatial_audit,seeds,seed_contexts,problems,pred,contexts,extra=None):
    def checked(df,col='status'):
        return bool(len(df)) and (col in df) and df[col].notna().any()
    spatial=any(x.get('status')=='AVAILABLE' or (x.get('status')=='PASS' and
                    x.get('n_features',0)>0) for x in spatial_audit)
    rows=[]
    def add(goal,status,detail):rows.append({'Цель':goal,'Статус':status,'Основание':detail})
    add('Муниципальная панель','PASS' if len(pred) and contexts else 'FAIL',f'{len(pred)} TEST-прогнозов, {len(contexts)} контекстов')
    add('3 горизонта','PASS' if set(pred.horizon.unique())=={1,3,5} else 'NOT_FEASIBLE',str(sorted(pred.horizon.unique())))
    add('Несколько независимых holdout-протоколов','PASS' if pred.protocol.nunique()>=3 else 'NOT_FEASIBLE',str(sorted(pred.protocol.unique())))
    for hyp in ('H1','H2','H3'):
        part=hdf[hdf.hypothesis.eq(hyp)] if len(hdf) and 'hypothesis' in hdf else pd.DataFrame()
        st='TESTED' if len(part) and not part.status.eq('NOT_FEASIBLE').all() else 'NOT_FEASIBLE'
        add(hyp,st,f'{len(part)} тестов; поддержано {int(part.get("decision",pd.Series(dtype=str)).eq("SUPPORTED").sum())}')
    add('H4 типология','TESTED' if checked(h4df) else 'NOT_FEASIBLE',f'{len(h4df)} записей')
    add('H5 переносимость','TESTED' if checked(h5df) else 'NOT_FEASIBLE',f'{len(h5df)} записей')
    add('50 seeds на стохастическую модель','PASS' if len(seeds)==len(seed_contexts)*50 else 'FAIL',f'{len(seeds)} / {len(seed_contexts)*50}')
    add('Пространственный источник','TESTED' if spatial else 'NOT_FEASIBLE',f'{len(spatial_audit)} записей QA; не синтезировать координаты')
    add('Прогноз без утечки','PASS' if pred.max_feature_year.le(pred.information_cutoff).all() and pred.max_label_year_fit.le(pred.information_cutoff).all() else 'FAIL','max_feature_year и max_label_year_fit <= cutoff')
    add('Систематические вычислительные ошибки','PASS' if not problems else 'FAIL',f'{len(problems)} ошибок')
    extra=extra or {}
    if extra:
        def table(k):return extra.get(k) if isinstance(extra.get(k),pd.DataFrame) else pd.DataFrame()
        metrics=table('metrics');optuna=table('optuna');importance=table('importance')
        shap=table('shap');ale=table('ale');intervals=table('intervals')
        cls=table('classification');flow=table('absolute_flows')
        add('Базовые модели','PASS' if 'last_known' in set(pred.model) else 'FAIL',
            'Обязательный persistence baseline')
        add('Модели и метрики на TEST','PASS' if len(metrics)>0 and len(optuna)>0 else 'FAIL',
            f'{len(metrics)} модельных метрик, {len(optuna)} trials')
        add('Оптимизация без TEST-подбора','PASS' if len(optuna) and 'selected_optuna' in optuna and
            optuna.loc[optuna['selected_optuna'],'MAE_DEV_FULL'].notna().all() else 'FAIL',
            'Отдельные метрики SEARCH и FULL DEV')
        add('Абсолютные миграционные потоки','TESTED' if len(flow) else 'NOT_FEASIBLE',f'{len(flow)} записей')
        add('Классификация оттока','TESTED' if len(cls) else 'NOT_FEASIBLE',f'{len(cls)} записей')
        add('Честные прогнозные интервалы','TESTED' if len(intervals) else 'NOT_FEASIBLE',f'{len(intervals)} записей')
        add('Объяснимость permutation','TESTED' if len(importance) else 'NOT_FEASIBLE',f'{len(importance)} записей')
        add('Объяснимость SHAP','TESTED' if len(shap) else 'NOT_FEASIBLE',f'{len(shap)} записей')
        add('Объяснимость ALE','TESTED' if len(ale) else 'NOT_FEASIBLE',f'{len(ale)} записей')
        plots=extra.get('figures',[])
        add('Графики по реальным вычислениям','PASS' if plots else 'FAIL',f'{len(plots)} рисунков')
    return pd.DataFrame(rows)

def v4_scalar(v):
    if isinstance(v,(dict,list,tuple,set)):return json.dumps(list(v) if isinstance(v,set) else v,ensure_ascii=False,default=str)
    if isinstance(v,(pd.Period,pd.Interval,pd.Timestamp)):return str(v)
    if isinstance(v,(np.integer,np.floating,np.bool_)):v=v.item()
    if v is None:return None
    try:
        if pd.isna(v):return None
    except (TypeError,ValueError):pass
    if isinstance(v,float) and not np.isfinite(v):return None
    return v

def v4_excel_actual_rows(ws, expected=None):
    """Count rows reliably in streaming OpenPyXL XLSX (max_row may be None).

    Use cheap worksheet dimension when present; otherwise stream data rows from XML.
    Fail closed on missing/truncated workbooks. No model data or statistics are changed.
    """
    rows=ws.max_row
    if rows is None:
        rows=sum(1 for _ in ws.iter_rows(values_only=True))
    if expected is not None and rows!=expected:
        raise ValueError(f'Excel readback rows mismatch {ws.title}: got {rows}, expected {expected}')
    return rows

def v4_workbook_openpyxl(path,sheets):
    """Fallback for missing XlsxWriter: streaming XLSX, same V4 file contract/readback.
    No in-memory materialisation of forecast tables; Excel hard row limits preserved.
    """
    from openpyxl import Workbook, load_workbook
    from openpyxl.cell import WriteOnlyCell
    from openpyxl.styles import Font, PatternFill, Alignment
    path=Path(path);temp=path.with_name(path.stem+'.tmp.xlsx')
    wb=Workbook(write_only=True)
    meta={}; names=[]
    header_font=Font(bold=True,color='FFFFFF')
    header_fill=PatternFill('solid',fgColor='17324E')
    for title,df in sheets.items():
        if df is None or len(df)==0:continue
        if not isinstance(df,pd.DataFrame):df=pd.DataFrame(df)
        if len(title)>31 or title in names:raise ValueError('Invalid workbook sheet '+str(title))
        if df.shape[1]>16384:raise ValueError('Too many columns')
        if not df.shape[1]:raise ValueError('No columns in sheet '+str(title))
        maxrows=1048575;groups=int(math.ceil(len(df)/maxrows))
        for chunk in range(groups):
            a=chunk*maxrows;b=min(len(df),(chunk+1)*maxrows)
            sheetname=title if groups==1 else title[:26]+'_'+str(chunk+1)
            if sheetname in names or len(sheetname)>31:raise ValueError('Excel sheet name collision')
            part=df.iloc[a:b];ws=wb.create_sheet(sheetname);names.append(sheetname)
            ws.freeze_panes='A2'
            header=[]
            for label in df.columns:
                cell=WriteOnlyCell(ws,value=str(label));cell.font=header_font
                cell.fill=header_fill;cell.alignment=Alignment(wrap_text=True,vertical='center')
                header.append(cell)
            ws.append(header)
            for row in part.itertuples(index=False,name=None):
                items=[]
                for item in row:
                    val=v4_scalar(item)
                    if isinstance(val,(int,float)) and not isinstance(val,bool):items.append(val)
                    elif val is None:items.append(None)
                    else:
                        # Prevent Excel formula injection (including strings starting with '=').
                        string=str(val)[:32767]
                        if string.startswith(('=','+','-','@')):
                            cell=WriteOnlyCell(ws,value=string);cell.data_type='s';items.append(cell)
                        else:items.append(string)
                ws.append(items)
            ws.auto_filter.ref='A1:'+__import__('openpyxl').utils.get_column_letter(df.shape[1])+str(len(part)+1)
            meta[sheetname]={'rows':len(part),'columns':df.shape[1]}
    if not names:raise ValueError('No populated sheets')
    wb.save(temp)
    temp.replace(path)
    check=load_workbook(path,read_only=True)
    try:
        if set(check.sheetnames)!=set(names):raise ValueError('Excel readback sheet mismatch')
        for title,d in meta.items():
            v4_excel_actual_rows(check[title],d['rows']+1)
    finally:check.close()
    return {'sha256':sha256_file(path),'size':path.stat().st_size,'sheets':meta}

def v4_workbook(path,sheets):
    try:
        import xlsxwriter
    except (ImportError,OSError):
        return v4_workbook_openpyxl(path,sheets)
    path=Path(path);temp=path.with_name(path.stem+'.tmp.xlsx')
    wb=xlsxwriter.Workbook(str(temp),{'constant_memory':True,'strings_to_urls':False,'nan_inf_to_errors':False})
    header=wb.add_format({'bold':True,'font_color':'white','bg_color':'#17324E',
                          'text_wrap':True,'valign':'vcenter','bottom':1,'bottom_color':'#456783'})
    names=[];meta={}
    try:
        for title,df in sheets.items():
            if df is None or len(df)==0:continue
            if not isinstance(df,pd.DataFrame):df=pd.DataFrame(df)
            if len(title)>31 or title in names:raise ValueError('Invalid workbook sheet '+title)
            if df.shape[1]>16384:raise ValueError('Too many columns')
            maxrows=1048575
            groups=int(math.ceil(len(df)/maxrows))
            for chunk in range(groups):
                a=chunk*maxrows;b=min(len(df),(chunk+1)*maxrows)
                sheetname=title if groups==1 else (title[:26]+'_'+str(chunk+1))
                if sheetname in names or len(sheetname)>31:raise ValueError('Excel sheet name collision')
                part=df.iloc[a:b];ws=wb.add_worksheet(sheetname);names.append(sheetname)
                ws.freeze_panes(1,0);ws.set_row(0,32);ws.set_default_row(18)
                ncol=len(df.columns)
                ws.write_row(0,0,list(map(str,df.columns)),header)
                widths=[]
                for c in df.columns:
                    sample=part[c].head(75).dropna().astype(str)
                    widths.append(min(43,max(12,len(str(c))+2,int(sample.str.len().max())+2 if len(sample) else 12)))
                for j,w in enumerate(widths):ws.set_column(j,j,w)
                for i,row in enumerate(part.itertuples(index=False,name=None),1):
                    for j,value in enumerate(row):
                        val=v4_scalar(value)
                        if val is None:continue
                        if isinstance(val,str) and val.startswith(('=','+','-','@')):
                            ws.write_string(i,j,val)
                        elif isinstance(val,(int,float)) and not isinstance(val,bool):
                            ws.write_number(i,j,val)
                        else:ws.write_string(i,j,str(val)[:32767])
                ws.autofilter(0,0,len(part),ncol-1)
                meta[sheetname]={'rows':len(part),'columns':ncol}
        if not names:raise ValueError('No populated sheets')
    finally:wb.close()
    temp.replace(path)
    from openpyxl import load_workbook
    check=load_workbook(path,read_only=True)
    try:
        if set(check.sheetnames)!=set(names):raise ValueError('Excel readback sheet mismatch')
        for title,d in meta.items():
            v4_excel_actual_rows(check[title],d['rows']+1)
    finally:check.close()
    return {'sha256':sha256_file(path),'size':path.stat().st_size,'sheets':meta}

def v4_export_bundle(root,results,predictions,dataqa,modelqa,figures,report):
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    tables=[('ML1_V4_RESULTS.xlsx',results),('ML1_V4_PREDICTIONS.xlsx',predictions),
            ('ML1_V4_DATA_QA.xlsx',dataqa),('ML1_V4_MODEL_QA.xlsx',modelqa)]
    meta={}
    for name,contents in tables:meta[name]=v4_workbook(root/name,contents)
    archive=root/'ML1_V4_FIGURES.zip';temp=archive.with_name('ML1_V4_FIGURES.tmp.zip')
    unique={Path(p).name:Path(p) for p in figures if Path(p).is_file()}
    if len(unique)>10:unique=dict(list(unique.items())[:10])
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for name,path in unique.items():z.write(path,arcname='figures/'+name)
        z.writestr('MANIFEST.json',json.dumps({'plots':sorted(unique),'sha256':{k:sha256_file(v) for k,v in unique.items()},
            'source':'ACTUAL_CALCULATIONS_ONLY'},ensure_ascii=False,indent=2))
    with zipfile.ZipFile(temp) as z:
        if z.testzip() is not None:raise ValueError('Figures ZIP CRC mismatch')
    temp.replace(archive)
    meta[archive.name]={'sha256':sha256_file(archive),'size':archive.stat().st_size,'figures':list(unique)}
    assert set(meta)==set(V4_OUTPUT_FILES)
    return meta

def assemble_v4_delivery(output_dir=RESULT_DIR):
    root=Path(output_dir)
    payload=list(V4_OUTPUT_FILES)
    if not all((root/name).is_file() for name in payload):raise FileNotFoundError('V4 output package incomplete')
    archive=root/'ML1_V4_DELIVERY.zip';tmp=archive.with_name('ML1_V4_DELIVERY.tmp.zip')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
        for name in payload:z.write(root/name,arcname=name)
    with zipfile.ZipFile(tmp) as z:
        if z.testzip() is not None or set(z.namelist())!=set(payload):raise ValueError('Delivery ZIP invalid')
    tmp.replace(archive)
    return archive

def verify_local(root):
    from openpyxl import load_workbook
    root=Path(root);manifest=json.loads((root/'run_manifest.json').read_text(encoding='utf-8'))
    if set(manifest['outputs'])!=set(V4_OUTPUT_FILES):raise ValueError('V4 output file set mismatch')
    results={}
    for name,meta in manifest['outputs'].items():
        path=root/name
        if not path.is_file() or sha256_file(path)!=meta['sha256']:
            raise ValueError('V4 output SHA mismatch: '+name)
        if name.endswith('.xlsx'):
            wb=load_workbook(path,read_only=True)
            try:
                assert wb.sheetnames and set(wb.sheetnames)==set(meta['sheets'])
                for sn,sm in meta['sheets'].items():
                    v4_excel_actual_rows(wb[sn],sm['rows']+1)
            finally:wb.close()
        else:
            with zipfile.ZipFile(path) as z:
                if z.testzip() is not None:raise ValueError('Figures ZIP CRC mismatch')
                figures=json.loads(z.read('MANIFEST.json'))
                for name0,digest in figures['sha256'].items():
                    if hashlib.sha256(z.read('figures/'+name0)).hexdigest()!=digest:
                        raise ValueError('Figure hash mismatch: '+name0)
        results[name]={'sha256':meta['sha256'],'bytes':path.stat().st_size}
    if manifest['n_prediction_rows']<=0:raise ValueError('No predictions')
    return {'status':'PASS_LOCAL_ONLY','sha256_verified':True,'files':results}


In [ ]:
# ML-1 V3: Optuna, spatial/context factors, deep XAI; binary targets and forecasts must align by row_id, never by pandas index.
# Scientific plan/version intentionally unchanged for safe regression checkpoint RESUME.
NOTEBOOK_PATCH = 'V4_1_2026-10-26_PREFLIGHT_5_FILE_CONTRACT'

def align_and_validate_binary(test_rows, simulated, target, prediction_col):
    required_left={'row_id',target}
    required_right={'row_id',prediction_col}
    if not required_left.issubset(test_rows) or not required_right.issubset(simulated):
        raise ValueError('Missing classification row_id, target or forecast column')
    left=test_rows[['row_id',target]].copy().rename(columns={target:'truth'})
    right=simulated[['row_id',prediction_col]].copy().rename(columns={prediction_col:'probability'})
    if left.row_id.isna().any() or right.row_id.isna().any():
        raise ValueError('Missing classification row IDs')
    if left.row_id.duplicated().any() or right.row_id.duplicated().any():
        raise ValueError('Duplicate classification row IDs')
    merged=left.merge(right,on='row_id',how='left',validate='one_to_one',indicator=True,sort=False)
    if not merged['_merge'].eq('both').all() or len(merged)!=len(left):
        raise ValueError('Classification simulation did not cover exactly the test row IDs')
    return validate_binary_predictions(merged.drop(columns='_merge'),require_all_labels=True)

def validate_binary_predictions(frame,require_all_labels=False):
    r=frame[['row_id','truth','probability']].copy()
    r['truth']=pd.to_numeric(r.truth,errors='coerce')
    r['probability']=pd.to_numeric(r.probability,errors='coerce')
    if require_all_labels and not np.isfinite(r.truth.to_numpy(float)).all():
        raise ValueError('Unexpected missing binary target after test label eligibility gate')
    r=r.loc[np.isfinite(r.truth.to_numpy(float))].copy()
    if r.empty:raise ValueError('No labelled test rows for classification')
    if not r.truth.isin([0.,1.]).all():raise ValueError('Binary truth must be exactly 0/1')
    prob=r.probability.to_numpy(float)
    if not np.isfinite(prob).all():raise ValueError('Non-finite classification probability after row_id merge')
    if np.any((prob<0)|(prob>1)):raise ValueError('Probability outside [0,1]')
    if r.row_id.duplicated().any():raise ValueError('Duplicate classified row_id')
    return r.reset_index(drop=True)

# Full research orchestration; GitHub publication follows only verified local outputs.

# V3.3.1: generated Optuna export schema; SEARCH != FULL-DEV.
def build_optuna_report_v331(dev_stats, dev_selection):
    """Preserve every successful search trial; mark the one selected from search.

    Search MAE is not interchangeable with a full-DEV MAE. No pandas attribute
    access is used for columns; column contract checked before model export.
    """
    columns=['horizon','protocol','family','model','trial_id','seed',
             'MAE_DEV_SEARCH','dev_design','n_dev','n_fit','parameters']
    report=pd.DataFrame(dev_stats)
    if report.empty:
        report=pd.DataFrame(columns=columns)
    missing=set(columns).difference(report.columns)
    if missing:
        raise ValueError('Invalid dev_stats schema; missing: '+', '.join(sorted(missing)))
    keys=['horizon','protocol','family','model']
    selected=[]
    for (h,protocol,family,name),info in dev_selection.items():
        selected.append({'horizon':h,'protocol':protocol,'family':family,'model':name,
                         'selected_trial_id':int(info['selected_trial_id']),
                         'MAE_DEV_FULL_SELECTED':float(info['dev_mae']),
                         'MAE_DEV_SEARCH_SELECTED':float(info['dev_mae_search'])})
    if not selected:
        report['selected_optuna']=False
        report['MAE_DEV_FULL']=np.nan
        return report
    sel=pd.DataFrame(selected)
    if sel.duplicated(keys).any() or report.duplicated(keys+['trial_id']).any():
        raise ValueError('Ambiguous Optuna selection keys or duplicate trial IDs')
    report=report.merge(sel,how='left',on=keys,validate='many_to_one')
    report['selected_optuna']=(report['trial_id'].eq(report['selected_trial_id'])).fillna(False)
    report['MAE_DEV_FULL']=report['MAE_DEV_FULL_SELECTED'].where(report['selected_optuna'])
    for _,row in sel.iterrows():
        mask=report[keys].eq(row[keys]).all(axis=1)&report['selected_optuna']
        if int(mask.sum())!=1:
            raise ValueError('Selected Optuna trial absent/duplicated: '+str(tuple(row[keys])))
        candidate=report.loc[mask].iloc[0]
        if not np.isclose(float(candidate['MAE_DEV_SEARCH']),float(row['MAE_DEV_SEARCH_SELECTED']),rtol=1e-9,atol=1e-9):
            raise ValueError('Chosen Optuna trial search score mismatch')
    return report


def recover_optuna_trial_values_v331(job_prefix, number_of_trials, store):
    """Read VERIFIED previous completed trials without training or creating new trials.

    Missing trial checkpoints represent pruned/unexecuted trials in the earlier
    experiment. They are NOT represented as successful Optuna trials.
    """
    found={};missing=[]
    for i in range(number_of_trials):
        job=tuple(job_prefix)+(i,0)
        stem=hash_obj(job)
        meta=Path(store)/(stem+'.meta.json')
        value_path=Path(store)/(stem+'.value.json')
        if not meta.is_file():
            missing.append(i)
            continue
        info=json.loads(meta.read_text(encoding='utf-8'))
        if info.get('jobid')!=list(job) or info.get('kind')!='optuna_dict':
            raise ValueError('Checkpoint identity/type conflict: '+str(job))
        if not value_path.is_file() or sha256_file(value_path)!=info.get('sha256'):
            raise ValueError('Optuna checkpoint SHA-256 mismatch: '+str(job))
        record=json.loads(value_path.read_text(encoding='utf-8'))
        required={'mae','parameters','dev_design','n_dev','n_fit'}
        if not isinstance(record,dict) or not required.issubset(record):
            raise ValueError('Incomplete cached Optuna record: '+str(job))
        if not np.isfinite(float(record['mae'])) or not isinstance(record['parameters'],dict):
            raise ValueError('Nonfinite or invalid cached Optuna trial: '+str(job))
        found[i]=record
    return found,missing


def run_v331_late_export_preflight():
    """Run before loading D01 and before any long compute; one synthetic example."""
    import tempfile
    trial_rows=[{'horizon':3,'protocol':'CROSSED','family':'B_economy','model':'huber',
                 'trial_id':0,'seed':11,'MAE_DEV_SEARCH':5.0,'dev_design':'H3',
                 'n_dev':50,'n_fit':140,'parameters':'{"epsilon": 1.4}'},
                {'horizon':3,'protocol':'CROSSED','family':'B_economy','model':'huber',
                 'trial_id':1,'seed':11,'MAE_DEV_SEARCH':4.0,'dev_design':'H3',
                 'n_dev':50,'n_fit':140,'parameters':'{"epsilon": 1.2}'}]
    selected={(3,'CROSSED','B_economy','huber'):{'selected_trial_id':1,
                        'dev_mae':5.3,'dev_mae_search':4.0,'parameters':{'epsilon':1.2}}}
    r=build_optuna_report_v331(trial_rows,selected)
    assert list(r['selected_optuna'])==[False,True]
    assert r['MAE_DEV_FULL'].isna().sum()==1 and float(r.iloc[1]['MAE_DEV_FULL'])==5.3
    try:
        build_optuna_report_v331([{k:v for k,v in trial_rows[0].items() if k!='MAE_DEV_SEARCH'}], selected)
        raise AssertionError('Missing-schema check failed')
    except ValueError: pass
    with tempfile.TemporaryDirectory() as temp:
        from pathlib import Path
        store=Path(temp)
        task=('OPTUNA',3,'CROSSED','B_economy','huber',0,0)
        value={'mae':4.0,'parameters':{'epsilon':1.2},'dev_design':'H3','n_dev':40,'n_fit':120}
        _=checkpoint_job(task,store,lambda:value,'FULL')
        recovered,miss=recover_optuna_trial_values_v331(task[:5],2,store)
        assert 0 in recovered and miss==[1]
        destination=store/(hash_obj(task)+'.value.json')
        destination.write_text('tampered',encoding='utf-8')
        try:
            recover_optuna_trial_values_v331(task[:5],2,store)
            raise AssertionError('SHA rejection test failed')
        except ValueError: pass
    return {'status':'PASS','checks':5,'scope':'Optuna selected-trial schema, dev score separation, verified cached trials, SHA corruption'}



def ml1_resume_checkpoint_audit(store, work, seed_contexts, min_fraction=0.60):
    """Fail-closed audit of ORIGINAL same-plan checkpoints, regardless of FAILED_RUN.json.

    No synthesis of completed jobs, no reuse of another plan or corrupted files.
    Require all 50 FINAL seed checkpoints per eligible horizon (late-export recovery).
    """
    store=Path(store)
    marks=sorted(store.glob('*.meta.json')) if store.is_dir() else []
    if not marks:
        return {'status':'ABSENT','valid':0,'needed_seed_count':0,'present_seed_count':0}
    allowed={hash_obj(tuple(job)): tuple(job) for job in work if job[0]!='CLASSIFY'}
    allowed.update({hash_obj(tuple(job)+('classification_alignment_fix_v2_1',)):
                    tuple(job)+('classification_alignment_fix_v2_1',)
                    for job in work if job[0]=='CLASSIFY'})
    kinds={'dataframe':'.parquet','optuna_dict':'.value.json'}
    seen=set()
    for mark in marks:
        stem=mark.name[:-len('.meta.json')]
        if stem in seen: raise ValueError('Duplicate cached job: '+stem)
        seen.add(stem)
        if stem not in allowed:raise ValueError('Checkpoint does not belong to current scientific plan: '+stem)
        meta=json.loads(mark.read_text(encoding='utf-8'))
        typ=meta.get('kind');payload=store/(stem+kinds.get(typ,'.bad'))
        if (typ not in kinds or meta.get('jobid')!=list(allowed[stem]) or
            not payload.is_file() or sha256_file(payload)!=meta.get('sha256')):
            raise ValueError('Checkpoint missing, corrupt or mismatched: '+str(meta.get('jobid')))
    necessary=[('SEED',h,proto,'DEV_BEST_STOCHASTIC',0,0,seed)
               for (h,proto) in seed_contexts for seed in FINAL_SEEDS]
    missing=[job for job in necessary if hash_obj(job) not in seen]
    # An incomplete mid-training run is not an export-only recovery, even with valid partial checkpoints.
    minimum=max(1,int(len(work)*min_fraction))
    if len(seen)<minimum or missing:
        return {'status':'INCOMPLETE','valid':len(seen),'work_total':len(work),
                'minimum_required':minimum,'needed_seed_count':len(necessary),
                'present_seed_count':len(necessary)-len(missing),
                'missing_seed_examples':[str(x) for x in missing[:5]]}
    return {'status':'READY_FOR_LATE_RECOVERY','valid':len(seen),
            'work_total':len(work),'needed_seed_count':len(necessary),
            'present_seed_count':len(necessary),'checkpoint_dir':str(store)}


def ml1_decide_run_mode(requested, audit):
    """AUTO selects strictly validated rescue or explicitly announced FULL. Never guesses."""
    if requested=='FULL':return 'FULL'
    if requested not in ('AUTO','RESUME'):
        raise ValueError('Unsupported mode '+str(requested))
    state=audit.get('status')
    if state=='READY_FOR_LATE_RECOVERY':return 'RECOVER'
    if requested=='AUTO' and state=='ABSENT':return 'FULL'
    raise RuntimeError('Verified late-run checkpoint unavailable: '+json.dumps(audit,ensure_ascii=False,default=str)[:800]+
                       '. For a new Colab runtime use FULL, or restore a previously downloaded checkpoint ZIP.')


def run_v333_resume_tests():
    """No network, no training; safeguards tested before opening Stage-I data."""
    import tempfile
    with tempfile.TemporaryDirectory() as td:
        store=Path(td)
        work=[('OPTUNA',1,'ROLLING_TIME','A_history','ridge',0,0),
              ('SEED',1,'ROLLING_TIME','DEV_BEST_STOCHASTIC',0,0,11)]
        prior=globals()['FINAL_SEEDS']
        try:
            globals()['FINAL_SEEDS']=[11]
            assert ml1_decide_run_mode('AUTO',ml1_resume_checkpoint_audit(store,work,[(1,'ROLLING_TIME')]))=='FULL'
            assert ml1_decide_run_mode('FULL',{'status':'INCOMPLETE'})=='FULL'
            value={'mae':3.14,'parameters':{'alpha':1},'dev_design':'TEMP','n_dev':30,'n_fit':120}
            checkpoint_job(work[0],store,lambda:value,'FULL')
            checkpoint_job(work[1],store,lambda:pd.DataFrame({'pred':[1.]}),'FULL')
            audit=ml1_resume_checkpoint_audit(store,work,[(1,'ROLLING_TIME')])
            assert audit['status']=='READY_FOR_LATE_RECOVERY' and ml1_decide_run_mode('RESUME',audit)=='RECOVER'
            assert checkpoint_job(work[0],store,lambda:(_ for _ in ()).throw(RuntimeError('RETRAIN!')),'RECOVER')[1]
            bad=('FINAL',5,'CROSSED','E_all','ridge',0,11)
            try:checkpoint_job(bad,store,lambda:None,'RECOVER');raise AssertionError('Recovery retrained missing job')
            except FileNotFoundError:pass
            payload=store/(hash_obj(work[0])+'.value.json');payload.write_text('corrupted')
            try:ml1_resume_checkpoint_audit(store,work,[(1,'ROLLING_TIME')]);raise AssertionError('Corrupt SHA not found')
            except ValueError:pass
        finally:globals()['FINAL_SEEDS']=prior
    return {'status':'PASS','tests':6,'note':'AUTO fallback; strict read-only recovery; plan + SHA; late seeds'}




def ml1_import_checkpoint_backup(archive,output_dir):
    """Optional cross-runtime restore from an explicitly user-provided recovery ZIP.

    Whitelisted filenames only, no traversal or silent replacement of conflicting files.
    Full checkpoint hashes/plan are checked later before recovery is activated.
    """
    from pathlib import PurePosixPath
    archive=Path(archive);output=Path(output_dir)
    if not archive.is_file():raise FileNotFoundError('Recovery ZIP not found: '+str(archive))
    copied=0
    with zipfile.ZipFile(archive,'r') as z:
        if z.testzip() is not None:raise ValueError('Recovery ZIP has invalid CRC')
        entries=[]
        for member in z.infolist():
            if member.is_dir():continue
            rel=PurePosixPath(member.filename)
            if (rel.is_absolute() or '..' in rel.parts or '\\' in member.filename or
                not rel.parts or rel.parts[0] not in ('checkpoint','checkpoint_plan.json','FAILED_RUN.json','technical.log')):
                raise ValueError('Unsafe recovery archive path: '+member.filename)
            if rel.parts[0]=='checkpoint' and len(rel.parts)!=3:
                raise ValueError('Invalid checkpoint archive path: '+member.filename)
            if rel.parts[0]!='checkpoint' and len(rel.parts)!=1:
                raise ValueError('Invalid recovery metadata path: '+member.filename)
            entries.append((member,rel))
        if not any(p.parts[0]=='checkpoint' for _,p in entries):
            raise ValueError('Recovery ZIP has no checkpoint data')
        for member,rel in entries:
            target=output.joinpath(*rel.parts)
            target.parent.mkdir(parents=True,exist_ok=True)
            contents=z.read(member)
            if target.exists():
                if hashlib.sha256(target.read_bytes()).digest()!=hashlib.sha256(contents).digest():
                    raise ValueError('Recovery import would overwrite different data: '+str(rel))
                continue
            temp=target.with_suffix(target.suffix+'.restoring')
            temp.write_bytes(contents);temp.replace(target);copied+=1
    return {'status':'IMPORTED_UNVERIFIED_UNTIL_WORKPLAN_GATE','files':copied,'archive':str(archive)}


def run_study_v2(mode='FULL',output_dir='/content/ml1_results_v2',cache_dir='/content/ml1_data_cache',
                 compute_profile='balanced',overrides=None,enable_spatial=True):
    root=Path(output_dir);root.mkdir(parents=True,exist_ok=True)
    cache=Path(cache_dir);cache.mkdir(parents=True,exist_ok=True)
    if mode=='SMOKE':return smoke_suite_v2()
    if mode=='VERIFY':return verify_local(root)
    if mode not in ('FULL','RESUME','AUTO'):raise ValueError('Use AUTO / FULL / RESUME / VERIFY / SMOKE')
    started=time.monotonic();logger=logging.getLogger('ml1_v3')
    for old_handler in list(logger.handlers):
        logger.removeHandler(old_handler);old_handler.close()
    logger.setLevel(logging.INFO);logger.propagate=False  # no duplicated notebook console INFO flood
    stream=logging.FileHandler(root/'technical.log',encoding='utf-8')
    stream.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
    logger.addHandler(stream)
    from collections import Counter
    warning_counts=Counter()
    # Narrowly hide a harmless sklearn/LightGBM metadata warning; do not hide model convergence warnings.
    warnings.filterwarnings('ignore',message=r'X does not have valid feature names, but LGBMRegressor was fitted with feature names',category=UserWarning)
    problems=[];limitations=[];dataqa=[];jobs_out=[]
    session=github_session(token_from_colab())
    index,source_asset_d01,source_manifest=read_handoff(session)
    global ACTIVE_INDEX
    ACTIVE_INDEX=index
    assert isinstance(source_asset_d01,dict) and {'sha256','asset_id','locator'}.issubset(source_asset_d01), 'D01 source metadata contract invalid'
    input_hashes=[{'asset_id':source_asset_d01['asset_id'],'sha256':source_asset_d01['sha256'],'locator':source_asset_d01['locator']}]
    source=fetch_asset(session,source_asset_d01,cache)
    import pyarrow.parquet as pq
    if pq.ParquetFile(source).metadata.num_rows!=int(source_asset_d01['rows_actual']):raise ValueError('D01 count mismatch')
    d,qa=clean_d01(pd.read_parquet(source));dataqa.append({'component':'D01',**qa})
    if not future_canary(d):raise ValueError('Canary: future mutation modified pre-origin factors')
    if qa['count_rate_conflict']/max(1,qa['observations'])>.12:
        limitations.append({'subject':'RATE_NET','status':'WARN','reason':'More than 12% inconsistent count/rate observations after scale validation'})
    limitations.append({'subject':'HISTORICAL_VINTAGE','status':'LIMITATION',
        'reason':'One retrospective 2026 snapshot; historical data vintages and boundary continuity cannot be guaranteed'})
    catalogues=[];asset_paths={};asset_map={x['asset_id']:x for x in index['assets']}
    # Full stage-I economy inventory; no hardcoded omissions of 5-NDFL/7-NDFL or BDMO.
    for a in (x for x in index['assets'] if x['segment']=='02'):
        try:
            cat,path=inventory_asset(session,a,cache)
            if len(cat):catalogues.append(cat)
            asset_paths[a['asset_id']]=path
            input_hashes.append({'asset_id':a['asset_id'],'sha256':a['sha256'],'locator':a['locator']})
        except Exception as e:
            logger.error('ASSET %s: %s',a['asset_id'],traceback.format_exc())
            limitations.append({'subject':a['asset_id'],'status':'ASSET_ERROR','reason':str(e)[:300]})
    catalog=pd.concat(catalogues,ignore_index=True) if catalogues else pd.DataFrame()
    selected,indicator_audit=choose_features(catalog,asset_paths,d,manual_overrides=overrides)
    if not selected:
        limitations.append({'subject':'H1_H2','status':'NOT_FEASIBLE','reason':'No uniquely validated economic/housing municipal features; the hypotheses remain untested'})
    panel_info=[];contexts={};split_audit=[];family_audit=[];spatial_audit=[];territory_panel=[]
    for h in HORIZONS:
        p=panel_for_h(d,h)
        p,available,attached_audit=attach_factors_v2(p,selected)
        for entry in attached_audit:family_audit.append({'horizon':h,**entry})
        if enable_spatial:
            try:
                p,sp_names,qa_space=attach_spatial_v2(session,index,cache,d,p)
                if sp_names:available['D_spatial']=sp_names
                if len(qa_space):spatial_audit.extend(qa_space.assign(horizon=h).to_dict('records'))
            except Exception as exc:
                spatial_audit.append({'horizon':h,'status':'NOT_FEASIBLE','reason':str(exc)[:300]})
                logger.error('SPATIAL: %s',traceback.format_exc())
        families=valid_families(available)
        for family,required in [('B_economy','B_economy'),('C_housing_services','C_housing_services'),('D_spatial','D_spatial'),('E_all','E_all')]:
            family_audit.append({'horizon':h,'feature_family':family,'status':'AVAILABLE' if family in families else 'NOT_FEASIBLE',
                'n_features':len(available.get(required,[]))})
        V3_HORIZON_PANELS[h]=p.copy()
        eligible,gate=gate_panel(p,h)
        panel_info.append(gate)
        if h==1:territory_panel.append(p)
        if gate['status']!='PASS':
            limitations.append({'subject':f'DATA_GATE_h{h}','status':'NOT_FEASIBLE','reason':str(gate)[:350]});continue
        splits,unavailable=plan_splits(eligible)
        # When direct supervised training is temporally impossible, include honest
        # recursive 1-step predictions from the last *published* historical year.
        if not splits and h>1:
            splits,alternative_issues=plan_recursive_holdouts(d,h)
            unavailable.extend(alternative_issues)
            if splits:limitations.append({'subject':f'h{h}_DIRECT_MODEL','status':'NOT_FEASIBLE',
                   'reason':'Insufficient past label history for direct horizon; recursive endogenous alternative evaluated separately'})
        for report in unavailable:split_audit.append({'horizon':h,**report})
        for protocol,s in splits.items():
            if s.get('forecast_type','').startswith('RECURSIVE'):
                # IMPORTANT: plan_recursive_holdouts() constructs a fresh panel_for_h(d,h)
                # without B/C/D/E factors. The h-specific `p` above already includes
                # frozen exogenous data at origin information_cutoff. Transfer by stable
                # territory × origin × cutoff keys into ALL four phases, including TEST.
                # Do not use row_id: h1/h3/h5 carry different row_id suffixes.
                s,bridge_audit=bridge_recursive_factors(s,p,available)
                for item in bridge_audit:
                    family_audit.append({'horizon':h,'protocol':protocol,
                        'feature_family':'RECURSIVE_BRIDGE',**item})
            split_audit.append({'horizon':h,'protocol':protocol,'status':'PASS','n_fit':len(s['fit']),
                               'n_dev':len(s['dev']),'n_train':len(s['train']), 'n_test':len(s['test']),
                               'train_regions':s['train'].region_id.nunique(),'test_regions':s['test'].region_id.nunique(),
                               'held_regions':','.join(s['withheld_regions']),'split_sha256':s['split_hash']})
            # Context-level data gate. No feature enters an Optuna study when it was
            # unavailable throughout its own historical FIT/TRAIN period.
            local_available=defaultdict(list)
            for feature_family,feature_names in available.items():
                for feature_name in feature_names:
                    if feature_name not in s['fit']:
                        family_audit.append({'horizon':h,'protocol':protocol,'feature':feature_name,
                           'feature_family':feature_family,'status':'NOT_FEASIBLE_FIT_NO_COLUMN'})
                        continue
                    fit_vals=pd.to_numeric(s['fit'][feature_name],errors='coerce')
                    observed=int(fit_vals.notna().sum())
                    if observed>=30 and fit_vals.nunique(dropna=True)>=3:
                        local_available[feature_family].append(feature_name)
                    else:
                        family_audit.append({'horizon':h,'protocol':protocol,'feature':feature_name,
                           'feature_family':feature_family,'n_fit_observed':observed,
                           'status':'NOT_FEASIBLE_FIT_COVERAGE',
                           'reason':'Feature unavailable or invariant before training cutoff'})
            local_families=valid_families(local_available)
            contexts[(h,protocol)]={'split':s,'available':dict(local_available),
                'families':local_families,
                'forecast_type':s.get('forecast_type','DIRECT_H')}
    if not contexts:raise RuntimeError('COMPUTE_INCOMPLETE: no feasible horizon x protocol; cannot report FULL')
    # V3.2 real-data complete feature contract PRECEDES every Optuna trial.
    recursive_preflight=preflight_recursive_contexts(contexts)
    print('ML-1 V3.3 FACTOR CONTRACT:',len(recursive_preflight),'recursive horizon/protocol/family tests PASS before Optuna')
    write_json(root/'RECURSIVE_FACTOR_PREFLIGHT_V3_3.json',{'version':VERSION,'status':'PASS','checks':recursive_preflight})
    # Frozen workplan, computed BEFORE model training, with explicit job list and immutable hashes.
    config_grid={key:None for key in OPTUNA_TRIALS}
    stochastic={'catboost','lightgbm','histgb'}
    work=[]
    for (h,protocol),c in sorted(contexts.items()):
        for family in c['families']:
            for name in config_grid:
                for i in range(OPTUNA_TRIALS[name]):
                    work.append(('OPTUNA',h,protocol,family,name,i,0))
                work.append(('FINAL',h,protocol,family,name,0,11))
        for name in ('last_known','trailing_MA3','damped_trend','regional_mean'):
            work.append(('BASELINE',h,protocol,'A_history',name,0,0))
        for target in ('target_net','target_cum_net'):
            for model in ('last_known_net','ridge','catboost'):
                work.append(('SECONDARY',h,protocol,target,model,0,11))
        for target in ('target_sign','practical_outflow','stable_outflow_3y'):
            for model in ('frequency','logistic','catboost_cls'):
                work.append(('CLASSIFY',h,protocol,target,model,0,11))
        work.append(('INTERVALS',h,protocol,0,0,0,0))
        work.append(('XAI',h,protocol,0,0,0,0))
    # Fifty meaningful seeds per horizon on one preregistered primary time protocol, not deterministic repeats.
    seed_contexts=[(h,'ROLLING_TIME') for h in HORIZONS if (h,'ROLLING_TIME') in contexts]
    if not seed_contexts:seed_contexts=[sorted(contexts.keys())[0]]
    for h,protocol in seed_contexts:
        work.extend([('SEED',h,protocol,'DEV_BEST_STOCHASTIC',0,0,seed) for seed in FINAL_SEEDS])
    work.extend([('ANALYSIS','all',0,0,0,0,0),('EXPORT','all',0,0,0,0,0),('VERIFY','all',0,0,0,0,0)])
    plan_data={'version':VERSION,'source_sha':source_asset_d01['sha256'],'input_hashes':input_hashes,'optuna_trials':OPTUNA_TRIALS,
               'selected':[x['spec'] for x in selected],'jobs':work,
               'split_hashes':{f'{h}:{proto}':v['split']['split_hash'] for (h,proto),v in contexts.items()},
               'lag':SOURCE_LAG_YEARS,'profile':compute_profile,
               'recursive_factor_bridge_contract':RECURSIVE_FACTOR_BRIDGE_VERSION,
               'search_protocol':'SAMPLED_FIT_DEV_FULL_DEV_RECHECK_FULL_FINAL',
               'optuna_fit_max':OPTUNA_FIT_MAX_ROWS,'optuna_dev_max':OPTUNA_DEV_MAX_ROWS,
               'optuna_total_budget_s':OPTUNA_GLOBAL_BUDGET_S}
    plan_hash=hash_obj(plan_data);store=root/'checkpoint'/plan_hash
    plan_file=root/'checkpoint_plan.json'
    requested_mode=mode
    checkpoint_audit=ml1_resume_checkpoint_audit(store,work,seed_contexts) if mode in ('RESUME','AUTO') else {'status':'NOT_REQUESTED'}
    effective=ml1_decide_run_mode(mode,checkpoint_audit)
    recovery_of_late_export=(effective=='RECOVER')
    if recovery_of_late_export:
        mode='RESUME'  # report mode; checkpoint_job uses stricter RECOVER explicitly below
        print('ML-1: VERIFIED RECOVERY; no model fitting/Optuna allowed |',checkpoint_audit['valid'],'SHA-checked checkpoints')
    else:
        mode='FULL'
        if requested_mode=='AUTO':
            print('ML-1 AUTO: no previous checkpoints in this runtime; announcing NEW FULL experiment (not RESUME).')
    store.mkdir(parents=True,exist_ok=True)
    write_json(plan_file,{'plan_hash':plan_hash,'tasks':work,'total':len(work),'created_at':utc(),
                         'requested_mode':requested_mode,'actual_mode':mode,
                         'recovery_audit':checkpoint_audit})
    counts={'completed':0,'failed':0,'resumed':0,'not_feasible':0,'pruned':0,'recovered_unrecorded':0};completed_seeds=[]
    error_fingerprints=Counter()  # Систематическая ошибка — остановить расчёт, не тратить тысячи trial.
    progress=tqdm(total=len(work),desc='ML-1 V4 FULL',unit='job',dynamic_ncols=True)
    completed_jobs=[]
    def tick(job,status='DONE'):
        counts['completed']+=1
        if status=='ERROR':counts['failed']+=1
        if status=='PRUNED':counts['pruned']+=1
        if status=='NOT_FEASIBLE':counts['not_feasible']+=1
        if status=='RECOVERED_UNRECORDED':counts['recovered_unrecorded']+=1
        completed_jobs.append({'job':str(job),'status':status})
        progress.update(1)
        progress.set_postfix_str(f'fail={counts["failed"]} NF={counts["not_feasible"]} resumed={counts["resumed"]} pruned={counts["pruned"]}')

    def guarded(job,fn,empty_reason='unavailable'):
        try:
            out,resumed=checkpoint_job(job,store,fn,'RECOVER' if recovery_of_late_export else mode)
            counts['resumed']+=int(resumed);tick(job)
            return out
        except optuna.TrialPruned:
            # Pruning означает рассчитанный промежуточный результат, а НЕ отказ модели.
            tick(job,'PRUNED')
            raise
        except Exception as e:
            reason=f'{type(e).__name__}: {str(e)[:280]}'
            signature=(str(job[0]),type(e).__name__,str(e)[:160])
            error_fingerprints[signature]+=1
            problems.append({'job':str(job),'status':'ERROR','reason':reason})
            logger.error('%s\n%s',job,traceback.format_exc());tick(job,'ERROR')
            # Ошибки типа, сериализации и массовые повторяющиеся ошибки НЕ маскируем TrialPruned.
            if isinstance(e,(TypeError,ImportError,AttributeError)) or error_fingerprints[signature]>=3:
                raise RuntimeError(f'FAIL_FAST after {error_fingerprints[signature]} repeated/fatal failures; '
                                   f'job={job}; root={reason}. Review technical.log') from e
            return None
    metrics=[];predictions=[];splits_evidence=[];dev_selection={};fit_models={};dev_stats=[];time_labels=[]
    optuna_started=time.monotonic();optuna_time_ledger=[]
    try:
        for (h,protocol),c in sorted(contexts.items()):
            sp=c['split'];train,fit,dev,test=(sp[k] for k in ('train','fit','dev','test'))
            # row-level audit separately for train/development/test, including geographic group and maximum label year.
            for stage,rows in [('FIT',fit),('DEV',dev),('TRAIN',train),('TEST',test)]:
                ev=rows[['row_id','territory_id','region_id','origin_year','target_year','information_cutoff','max_feature_year','horizon']].copy()
                ev['protocol']=protocol;ev['split']=stage;ev['max_label_year_fit']=int(sp['train'].target_year.max())
                ev['split_sha256']=sp['split_hash'];splits_evidence.append(ev)
            for family in c['families']:
                cols=feature_cols(family,c['available'])
                for name in config_grid:
                    # OPTUNA V3.3: deterministic reduced FIT/DEV matrices; real FINAL remains full.
                    # Long-horizon DEV uses true h-year labels if non-leaking historical data exist.
                    # Subsampling only searches hyperparameters; scores for choosing the
                    # winning family/model are independently recomputed on FULL DEV.
                    dev_long,dev_design=(horizon_development_panel(d,h,sp)
                        if c['forecast_type'].startswith('RECURSIVE') else (dev,'DIRECT_H_DEV'))
                    full_validation=dev_long if dev_long is not None else dev
                    recursive_dev=bool(c['forecast_type'].startswith('RECURSIVE') and dev_long is not None)
                    source_fit=(fit[fit.target_year.le(int(full_validation.information_cutoff.min()))]
                        if recursive_dev else fit)
                    if len(source_fit)<30 or len(full_validation)<20:
                        limitations.append({'subject':f'OPTUNA_h{h}_{protocol}_{family}_{name}',
                                            'status':'NOT_FEASIBLE', 'reason':'Insufficient non-leaking FIT/DEV'})
                        for i in range(OPTUNA_TRIALS[name]):tick(('OPTUNA',h,protocol,family,name,i,0),'NOT_FEASIBLE')
                        tick(('FINAL',h,protocol,family,name,0,11),'NOT_FEASIBLE')
                        continue
                    salt=h*227+len(protocol)*13+len(family)*17
                    fast_fit=ml1_stratified_search_sample(source_fit,OPTUNA_FIT_MAX_ROWS,salt)
                    fast_dev=ml1_stratified_search_sample(full_validation,OPTUNA_DEV_MAX_ROWS,salt+999)
                    if recovery_of_late_export:
                        trial_cache,not_recovered=recover_optuna_trial_values_v331(
                            ('OPTUNA',h,protocol,family,name),OPTUNA_TRIALS[name],store)
                        for i in range(OPTUNA_TRIALS[name]):
                            trial_job=('OPTUNA',h,protocol,family,name,i,0)
                            if i in trial_cache:
                                record=trial_cache[i]
                                dev_stats.append({'horizon':h,'protocol':protocol,'family':family,'model':name,
                                    'trial_id':i,'seed':11,'MAE_DEV_SEARCH':float(record['mae']),
                                    'dev_design':record['dev_design'],'n_dev':record['n_dev'],
                                    'n_fit':record['n_fit'],
                                    'parameters':json.dumps(record['parameters'],sort_keys=True),
                                    'source':'VERIFIED_CHECKPOINT'})
                                counts['resumed']+=1;tick(trial_job)
                            else:
                                # Exact reason (pruned vs budget cap) wasn't persisted in V3.3.
                                # Do not mislabel it as successfully evaluated or retrain it.
                                tick(trial_job,'RECOVERED_UNRECORDED')
                        job=('FINAL',h,protocol,family,name,0,11)
                        if not trial_cache:
                            limitations.append({'subject':str(job),'status':'NOT_FEASIBLE',
                                'reason':'No successful Optuna trial stored in original V3.3; recovery does not fabricate a model'})
                            tick(job,'NOT_FEASIBLE');continue
                        selected_trial_id=min(trial_cache,key=lambda i:(float(trial_cache[i]['mae']),i))
                        chosen=trial_cache[selected_trial_id]
                        param=chosen['parameters'];best_mae_search=float(chosen['mae'])
                        completed=list(trial_cache)
                        optuna_time_ledger.append({'horizon':h,'protocol':protocol,'family':family,
                            'model':name,'trial_budget':OPTUNA_TRIALS[name],
                            'trials_attempted':len(trial_cache),
                            'search_elapsed_s':np.nan,'search_time_source':'EARLIER_RUN_NOT_REPLAYED',
                            'n_fit_search':len(fast_fit),'n_fit_full':len(source_fit),
                            'n_dev_search':len(fast_dev),'n_dev_full':len(full_validation),
                            'recursive_dev':recursive_dev,'dev_design':dev_design,
                            'not_recovered_count':len(not_recovered)})
                    else:
                        sampler=optuna.samplers.TPESampler(seed=V3_OPTUNA_SAMPLER_SEED+salt,n_startup_trials=3)
                        study=optuna.create_study(direction='minimize',sampler=sampler,
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=3,n_warmup_steps=0))
                        study_start=time.monotonic()
                        def objective(trial):
                            params=ml1_optuna_parameters(trial,name)
                            job=('OPTUNA',h,protocol,family,name,trial.number,0)
                            progress.set_description(f'OPTUNA h{h} {protocol} {family}/{name}')
                            def calculate():
                                # Stage 1: small FIT subset. A trial can be cut off before
                                # expensive full-search FIT after comparable DEV scoring.
                                if OPTUNA_PRUNE_FAST_FIT and name in ('catboost','lightgbm','histgb') and len(fast_fit)>120:
                                    pilot=ml1_stratified_search_sample(fast_fit,max(90,int(len(fast_fit)*.45)),salt+313)
                                    interim,_=ml1_fit_dev_objective(pilot,fast_dev,cols,name,params,11,recursive_dev)
                                    trial.report(interim,step=0)
                                    if trial.should_prune():
                                        raise optuna.TrialPruned('early prune using fixed pilot FIT and DEV; no TEST')
                                mae,notes=ml1_fit_dev_objective(fast_fit,fast_dev,cols,name,params,11,recursive_dev)
                                if notes:warning_counts.update(notes)
                                trial.report(mae,step=1)
                                if trial.should_prune():
                                    raise optuna.TrialPruned('median prune after sampled FIT DEV')
                                return {'mae':float(mae),'per_seed':[float(mae)],
                                    'parameters':params,'dev_design':dev_design,
                                    'n_dev':len(fast_dev),'n_fit':len(fast_fit),
                                    'dev_total':len(full_validation),'fit_total':len(source_fit),
                                    'optimisation_sample_only':True}
                            record=guarded(job,calculate)
                            if record is None:raise optuna.TrialPruned('Real failure logged; fail-fast monitors repeats')
                            dev_stats.append({'horizon':h,'protocol':protocol,'family':family,'model':name,
                                  'trial_id':trial.number,'seed':11,'MAE_DEV_SEARCH':record['mae'],
                                  'dev_design':record['dev_design'],'n_dev':record['n_dev'],
                                  'n_fit':record['n_fit'],'parameters':json.dumps(params,sort_keys=True)})
                            trial.set_user_attr('dev_design',dev_design)
                            return record['mae']
                        # Each configuration always gets at least one valid trial; time budget
                        # stops only additional trials, never drops family or FINAL evaluation.
                        n_trials=(OPTUNA_TRIALS[name] if time.monotonic()-optuna_started<OPTUNA_GLOBAL_BUDGET_S
                                  else OPTUNA_MIN_TRIALS_PER_STUDY)
                        study.optimize(objective,n_trials=n_trials,n_jobs=1,show_progress_bar=False,catch=())
                        elapsed=time.monotonic()-study_start
                        for unused in range(n_trials,OPTUNA_TRIALS[name]):
                            job_skip=('OPTUNA',h,protocol,family,name,unused,0)
                            limitations.append({'subject':str(job_skip),'status':'SEARCH_BUDGET_STOP',
                                                'reason':'Research search budget; FULL final estimator still runs'})
                            tick(job_skip,'NOT_FEASIBLE')
                        optuna_time_ledger.append({'horizon':h,'protocol':protocol,'family':family,
                            'model':name,'trial_budget':OPTUNA_TRIALS[name],
                            'trials_attempted':n_trials,'search_elapsed_s':elapsed,
                            'n_fit_search':len(fast_fit),'n_fit_full':len(source_fit),
                            'n_dev_search':len(fast_dev),'n_dev_full':len(full_validation),
                            'recursive_dev':recursive_dev,'dev_design':dev_design})
                        job=('FINAL',h,protocol,family,name,0,11)
                        completed=[t for t in study.trials if t.state==optuna.trial.TrialState.COMPLETE]
                        if not completed:
                            limitations.append({'subject':str(job),'status':'ERROR',
                                                'reason':'No valid Optuna trial; see technical.log'})
                            tick(job,'NOT_FEASIBLE');continue
                        param=study.best_params
                        selected_trial_id=int(study.best_trial.number)
                        best_mae_search=float(study.best_value)
                    # Full DEV evaluation is MANDATORY before selecting final model/family.
                    # This refits the chosen hyperparameters on the original full FIT,
                    # preserving equal comparison of all families with shared DEV rows.
                    best_mae,notes=ml1_fit_dev_objective(source_fit,full_validation,cols,name,param,11,recursive_dev)
                    if notes:warning_counts.update(notes)
                    dev_selection[(h,protocol,family,name)]={'parameters':param,
                        'dev_mae':best_mae,'dev_mae_search':best_mae_search,
                        'selected_trial_id':int(selected_trial_id),
                        'dev_design':dev_design,'n_dev':len(full_validation),
                        'n_optuna_trials':(OPTUNA_TRIALS[name] if recovery_of_late_export else len(study.trials)),
                        'n_optuna_completed':(len(trial_cache) if recovery_of_late_export else len(completed))}
                    def compute_final():
                        if c['forecast_type'].startswith('RECURSIVE'):
                            _,predictor,notes=fit_predict(train,dev,cols,name,param,seed=11)
                            hat=forecast_recursively(predictor,test,cols)
                            name_rec='recursive_'+name
                        else:
                            hat,_,notes=fit_predict(train,test,cols,name,param,seed=11)
                            name_rec=name
                        if notes:warning_counts.update(notes)
                        return record_predictions(test,hat,name_rec,family,protocol,h,11,int(train.target_year.max()))
                    res=guarded(job,compute_final)
                    if res is not None:
                        predictions.append(res)
                        metrics.append({'horizon':h,'protocol':protocol,'feature_set':family,
                            'model':'recursive_'+name if c['forecast_type'].startswith('RECURSIVE') else name,
                            'training_type':c['forecast_type'],'seed':11,
                            'MAE_DEV':best_mae,**score(res.target_rate,res.population_exposure,res.prediction)})
            for baseline,guess in baselines(train,test).items():
                job=('BASELINE',h,protocol,'A_history',baseline,0,0)
                q=guarded(job,lambda:record_predictions(test,guess,baseline,'A_history',protocol,h,0,int(train.target_year.max())))
                if q is not None:
                    predictions.append(q)
                    metrics.append({'horizon':h,'protocol':protocol,'feature_set':'A_history','model':baseline,
                                    'seed':0,'MAE_DEV':np.nan,**score(q.target_rate,q.population_exposure,q.prediction)})
        if not predictions:raise RuntimeError('No predictions on holdout')
        mdf=pd.DataFrame(metrics)
        allpred=pd.concat(predictions,ignore_index=True)
        # Choose stochastic model by DEV only. 50 seeds for each available horizon; fixed preflight denominator.
        for h,protocol in seed_contexts:
            stoch_names=['catboost','lightgbm','histgb']
            if contexts[h,protocol]['forecast_type'].startswith('RECURSIVE'):
                stoch_names=['recursive_'+x for x in stoch_names]
            ref=mdf[(mdf.horizon==h)&mdf.protocol.eq(protocol)&mdf.model.isin(stoch_names)&mdf.seed.eq(11)]
            win=ref.sort_values(['MAE_DEV','model','feature_set']).head(1)
            for seed in FINAL_SEEDS:
                job=('SEED',h,protocol,'DEV_BEST_STOCHASTIC',0,0,seed)
                if win.empty:
                    limitations.append({'subject':str(job),'status':'NOT_FEASIBLE','reason':'No successful stochastic candidate selected by DEV'})
                    tick(job,'NOT_FEASIBLE');continue
                x=win.iloc[0];name,fs=x.model,x.feature_set
                c=contexts[h,protocol];sp=c['split'];params=dev_selection[h,protocol,fs,name.removeprefix('recursive_')]['parameters']
                feats=feature_cols(fs,c['available'])
                def seed_run():
                    fit_name=name.removeprefix('recursive_')
                    if c['forecast_type'].startswith('RECURSIVE'):
                        _,predictor,_=fit_predict(sp['train'],sp['dev'],feats,fit_name,params,seed=seed)
                        hat=forecast_recursively(predictor,sp['test'],feats)
                    else:
                        hat,_,_=fit_predict(sp['train'],sp['test'],feats,fit_name,params,seed=seed)
                    return record_predictions(sp['test'],hat,name,fs,protocol,h,seed,int(sp['train'].target_year.max()))
                result=guarded(job,seed_run)
                if result is not None:
                    allpred=pd.concat([allpred,result],ignore_index=True)
                    completed_seeds.append({'horizon':h,'protocol':protocol,'model':name,'family':fs,'seed':seed})
                    metrics.append({'horizon':h,'protocol':protocol,'feature_set':fs,'model':name,'seed':seed,
                        'MAE_DEV':float(x.MAE_DEV),**score(result.target_rate,result.population_exposure,result.prediction)})
        mdf=pd.DataFrame(metrics)
        # Regression completed; train/test prevalence and label availability for each ancillary target.
        secondary_scores=[];secondary_predictions=[];classification=[];classification_preds=[];calibration=[]
        for (h,protocol),c in sorted(contexts.items()):
            sp=c['split'];feat=feature_cols('A_history',c['available'])
            for outcome in ('target_net','target_cum_net'):
                for name in ('last_known_net','ridge','catboost'):
                    job=('SECONDARY',h,protocol,outcome,name,0,11)
                    if c['forecast_type'].startswith('RECURSIVE'):
                        tr=sp['train'].dropna(subset=['target_net','net_lag0']).copy()
                        dv=sp['dev'].dropna(subset=['target_net','net_lag0']).copy()
                        te=sp['test'].dropna(subset=[outcome,'net_lag0']).copy()
                        net_features=list(dict.fromkeys(FEATURE_BASE+['net_lag0','net_lag1','net_lag2','net_mean3']))
                        if min(len(tr),len(dv),len(te))<20:
                            limitations.append({'subject':str(job),'status':'NOT_FEASIBLE',
                              'reason':f'No verified signed flow train={len(tr)}, dev={len(dv)}, test={len(te)}'})
                            tick(job,'NOT_FEASIBLE');continue
                        def compute_net_recurrence():
                            if name=='last_known_net':
                                prediction=te.net_lag0.to_numpy(float)*(h if outcome=='target_cum_net' else 1)
                            else:
                                params={'alpha':30.} if name=='ridge' else {'iterations':240,'depth':5,'learning_rate':.045,'l2_leaf_reg':6}
                                _,model,_=fit_predict(tr,dv,net_features,name,params,seed=11,target='target_net')
                                simulated=forecast_net_recursively(model,te,net_features)
                                col='prediction_cum_net' if outcome=='target_cum_net' else 'prediction_net'
                                prediction=simulated[col].to_numpy(float)
                            return pd.DataFrame({'row_id':te.row_id,'truth':te[outcome],'prediction':prediction})
                        res=guarded(job,compute_net_recurrence)
                        if res is not None:
                            testscore=score(res.truth,te.population_exposure,res.prediction)
                            secondary_scores.append({'horizon':h,'protocol':protocol,'target':outcome,
                                'model':'recursive_'+name,'units':'persons','MAE_persons':testscore['MAE'],
                                'RMSE_persons':testscore['RMSE'],'test_n':len(te),
                                'assumption':'Future population frozen to last observed population; no future counts used'})
                            res['horizon']=h;res['protocol']=protocol;res['outcome']=outcome;res['model']='recursive_'+name
                            secondary_predictions.append(res)
                        continue
                    tr=sp['train'].dropna(subset=[outcome,'net_lag0']);te=sp['test'].dropna(subset=[outcome,'net_lag0'])
                    # Last training target year must be no later than publication cutoff.
                    if len(tr)<80 or len(te)<20:
                        limitations.append({'subject':str(job),'status':'NOT_FEASIBLE','reason':f'train={len(tr)} test={len(te)}'});tick(job,'NOT_FEASIBLE');continue
                    cols=list(dict.fromkeys(feat+['net_lag0','net_lag1','net_lag2','net_mean3']))
                    def predict_net():
                        if name=='last_known_net':
                            p=te.net_lag0.to_numpy(float)*(h if outcome=='target_cum_net' else 1)
                        else:
                            pars={'alpha':30.} if name=='ridge' else {'iterations':240,'depth':5,'learning_rate':.045,'l2_leaf_reg':6}
                            p,_,_=fit_predict(tr,te,cols,name,pars,seed=11,target=outcome)
                        return pd.DataFrame({'row_id':te.row_id,'truth':te[outcome],'prediction':p})
                    r=guarded(job,predict_net)
                    if r is not None:
                        err=score(r.truth,te.population_exposure,r.prediction)
                        secondary_scores.append({'horizon':h,'protocol':protocol,'target':outcome,'model':name,
                                'units':'persons','MAE_persons':err['MAE'],'RMSE_persons':err['RMSE'],'test_n':len(te)})
                        r['horizon']=h;r['protocol']=protocol;r['outcome']=outcome;r['model']=name
                        secondary_predictions.append(r)
            for target in ('target_sign','practical_outflow','stable_outflow_3y'):
                for name in ('frequency','logistic','catboost_cls'):
                    job=('CLASSIFY',h,protocol,target,name,0,11)
                    if c['forecast_type'].startswith('RECURSIVE'):
                        if name!='frequency':
                            limitations.append({'subject':str(job),'status':'NOT_FEASIBLE',
                              'reason':'No honest calibrated h-year classification training window; output available only as a deterministic flow-simulation rule'})
                            tick(job,'NOT_FEASIBLE');continue
                        tr=sp['train'].dropna(subset=['target_net','net_lag0']).copy()
                        dv=sp['dev'].dropna(subset=['target_net','net_lag0']).copy()
                        te=sp['test'].dropna(subset=[target,'net_lag0']).copy()
                        net_feats=list(dict.fromkeys(FEATURE_BASE+['net_lag0','net_lag1','net_lag2','net_mean3']))
                        if min(len(tr),len(dv),len(te))<20:
                            limitations.append({'subject':str(job),'status':'NOT_FEASIBLE',
                              'reason':'Cannot derive tested labels from verified annual net series'})
                            tick(job,'NOT_FEASIBLE');continue
                        def simulation_class():
                            _,mdl,_=fit_predict(tr,dv,net_feats,'ridge',{'alpha':30.},seed=11,target='target_net')
                            sim=forecast_net_recursively(mdl,te,net_feats)
                            col={'target_sign':'prediction_sign','practical_outflow':'prediction_outflow',
                                 'stable_outflow_3y':'prediction_stable3'}[target]
                            # Guard against pandas Series index alignment: row_id merge is authoritative.
                            return align_and_validate_binary(te,sim,target,col)
                        r=guarded(job,simulation_class)
                        if r is not None:
                            r=validate_binary_predictions(r,require_all_labels=True)
                            if len(r):
                                from sklearn.metrics import brier_score_loss,roc_auc_score,average_precision_score
                                classification.append({'horizon':h,'protocol':protocol,'target':target,
                                    'model':'recursive_flow_rule','n_train':len(tr),'n_test':len(r),
                                    'positive_rate':float(r.truth.mean()),
                                    'brier':brier_score_loss(r.truth,r.probability),
                                    'PR_AUC':average_precision_score(r.truth,r.probability) if r.truth.nunique()==2 else np.nan,
                                    'ROC_AUC':roc_auc_score(r.truth,r.probability) if r.truth.nunique()==2 else np.nan,
                                    'warning':'0/1 deterministic rule, NOT a calibrated probability'})
                                r['horizon']=h;r['protocol']=protocol;r['target']=target;r['model']='recursive_flow_rule'
                                classification_preds.append(r)
                        continue
                    tr=sp['train'].dropna(subset=[target]).copy()
                    dv=sp['dev'].dropna(subset=[target]).copy()
                    te=sp['test'].dropna(subset=[target]).copy()
                    if target=='stable_outflow_3y':
                        tr=tr[tr.origin_year.add(3).le(sp['cutoff'])].copy()
                        dv=dv[dv.origin_year.add(3).le(sp['cutoff'])].copy()
                    if len(tr)<80 or len(te)<20 or tr[target].nunique()<2 or min(tr[target].value_counts())<5:
                        limitations.append({'subject':str(job),'status':'NOT_FEASIBLE',
                            'reason':f'insufficient train/test positive-negative labels: train={len(tr)}, test={len(te)}'});tick(job,'NOT_FEASIBLE');continue
                    def fit_class():
                        if name=='frequency':prob=np.full(len(te),float(tr[target].mean()))
                        else:
                            from sklearn.pipeline import Pipeline
                            from sklearn.linear_model import LogisticRegression
                            from catboost import CatBoostClassifier
                            prep,columns=numeric_design(tr,feat)
                            estimator=(LogisticRegression(max_iter=1100,class_weight='balanced') if name=='logistic' else
                                CatBoostClassifier(iterations=230,depth=5,learning_rate=.05,random_seed=11,
                                    verbose=False,allow_writing_files=False,thread_count=2))
                            Xtr=prep.fit_transform(tr[columns].astype({'region_id':str,'mun_type':str}))
                            estimator.fit(Xtr,tr[target].astype(int))
                            Xte=prep.transform(te[columns].astype({'region_id':str,'mun_type':str}))
                            prob=estimator.predict_proba(Xte)[:,1]
                        return validate_binary_predictions(pd.DataFrame({
                            'row_id':te.row_id.to_numpy(),'truth':te[target].to_numpy(float),
                            'probability':np.asarray(prob,float)}),require_all_labels=True)
                    r=guarded(job,fit_class)
                    if r is not None:
                        r=validate_binary_predictions(r,require_all_labels=True)
                        from sklearn.metrics import roc_auc_score,average_precision_score,brier_score_loss
                        unique=r.truth.nunique()
                        classification.append({'horizon':h,'protocol':protocol,'target':target,'model':name,
                            'n_train':len(tr),'n_test':len(te),'positive_rate':r.truth.mean(),
                            'brier':brier_score_loss(r.truth,r.probability),
                            'PR_AUC':average_precision_score(r.truth,r.probability) if unique==2 else np.nan,
                            'ROC_AUC':roc_auc_score(r.truth,r.probability) if unique==2 else np.nan})
                        r['horizon']=h;r['protocol']=protocol;r['target']=target;r['model']=name
                        classification_preds.append(r)
                        r['bin']=pd.cut(r.probability,np.linspace(0,1,11),include_lowest=True).astype(str)
                        part=r.groupby('bin',observed=True).agg(n=('truth','size'),actual_rate=('truth','mean'),mean_probability=('probability','mean')).reset_index()
                        part['bin']=part['bin'].astype(str)
                        part['horizon']=h;part['protocol']=protocol;part['target']=target;part['model']=name
                        calibration.append(part)
        # Split-conformal and XAI: model locked using DEV, test not touched during choice.
        intervals=[];interval_predictions=[];imp=[];shap_all=[];ale_all=[];interaction_all=[]
        for (h,protocol),c in sorted(contexts.items()):
            sp=c['split']
            candidates=mdf[(mdf.horizon==h)&mdf.protocol.eq(protocol)&mdf.seed.eq(11)&mdf.MAE_DEV.notna()]
            winner=candidates.sort_values(['MAE_DEV','model','feature_set']).head(1)
            for stage in ('INTERVALS','XAI'):
                job=(stage,h,protocol,0,0,0,0)
                recursive_context=c['forecast_type'].startswith('RECURSIVE')
                if recursive_context and stage=='INTERVALS':
                    hdev,how=horizon_development_panel(d,h,sp)
                    if hdev is None:
                        limitations.append({'subject':str(job),'status':'NOT_FEASIBLE',
                          'reason':'No independent horizon-specific calibration labels before test cutoff; 1-step interval forbidden'})
                        tick(job,'NOT_FEASIBLE');continue
                if winner.empty:
                    limitations.append({'subject':str(job),'status':'NOT_FEASIBLE','reason':'No DEV-selected predictor'})
                    tick(job,'NOT_FEASIBLE');continue
                x=winner.iloc[0];model,family=x.model,x.feature_set
                features=feature_cols(family,c['available']);base_model=ml1_base_model_name(model)
                params=dev_selection[h,protocol,family,base_model]['parameters']
                if stage=='INTERVALS':
                    try:
                        pred_dev,pl,_=fit_predict(sp['fit'],sp['dev'],features,base_model,params,11)
                        if recursive_context:
                            calibrate,_=horizon_development_panel(d,h,sp)
                            pred_dev=forecast_recursively(pl,calibrate,features)
                            pred_te=forecast_recursively(pl,sp['test'],features)
                            cal_truth=calibrate.target_rate
                        else:
                            teX=sp['test'][features+['region_id','mun_type']].astype({'region_id':str,'mun_type':str})
                            pred_te=pl.predict(teX)
                            cal_truth=sp['dev'].target_rate
                        low,up,q=conformal_predictions(cal_truth,pred_dev,pred_te)
                        truth=sp['test'].target_rate.to_numpy(float)
                        sample=sp['test'][['row_id','horizon','region_id','mun_type','population_exposure']].copy()
                        sample['low']=low;sample['high']=up;sample['actual']=truth
                        sample['covered']=((truth>=low)&(truth<=up)).astype(int)
                        sample['protocol']=protocol;sample['model']=model
                        interval_predictions.append(sample)
                        intervals.append({'horizon':h,'protocol':protocol,'model':model,'feature_set':family,
                            'nominal_coverage':.90,'empirical_coverage':float(sample.covered.mean()),
                            'mean_width_pp1000':float((up-low).mean()),'q_calibration':q,
                            'dev_n':len(cal_truth),'test_n':len(sp['test'])})
                        tick(job)
                    except Exception as e:
                        problems.append({'job':str(job),'status':'ERROR','reason':str(e)[:300]});tick(job,'ERROR');logger.error(traceback.format_exc())
                else:
                    # Explain every feasible protocol; mark recursive interpretation as ONE_STEP_SURROGATE.
                    try:
                        explain_rows=sp['dev'] if recursive_context else sp['test']
                        hat,pl,_=fit_predict(sp['train'],explain_rows,features,base_model,params,11)
                        a,b,c1,e,n=interpret_model_v2(pl,sp['train'],explain_rows,features)
                        if recursive_context:
                            for table in (a,b,c1,e):
                                if len(table):table['scope']='ONE_STEP_SURROGATE_NOT_H_YEAR_CAUSALITY'
                        for z,collection in [(a,imp),(b,shap_all),(c1,ale_all),(e,interaction_all)]:
                            if len(z):
                                z['horizon']=h;z['protocol']=protocol;z['model']=model;z['feature_set']=family
                                collection.append(z)
                        limitations.extend(n);tick(job)
                    except Exception as e:
                        problems.append({'job':str(job),'status':'ERROR','reason':str(e)[:300]});tick(job,'ERROR');logger.error(traceback.format_exc())
        def concat(frames):return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
        secondary_df=pd.DataFrame(secondary_scores);secondary_pred=concat(secondary_predictions)
        cls_df=pd.DataFrame(classification);cls_pred=concat(classification_preds);cal_df=concat(calibration)
        interval_df=pd.DataFrame(intervals);interval_pred=concat(interval_predictions)
        impdf=concat(imp);shapdf=concat(shap_all);aledf=concat(ale_all);interdf=concat(interaction_all)
        job=('ANALYSIS','all',0,0,0,0,0)
        representative=allpred[allpred.seed.isin([0,11])].copy()
        # Hard full-key parity on every valid candidate, no differential coverage.
        for (h,protocol,model,family),q in representative.groupby(['horizon','protocol','model','feature_set']):
            expected=set(contexts[h,protocol]['split']['test'].row_id)
            if set(q.row_id)!=expected or q.row_id.duplicated().any():raise ValueError(f'Prediction keys mismatch: {h}/{protocol}/{model}/{family}')
        hypothesis=[];tests=[];family_choices={}
        for (h,protocol),ctx in sorted(contexts.items()):
            part=mdf[(mdf.horizon==h)&mdf.protocol.eq(protocol)&mdf.seed.eq(11)]
            choice={}
            for family in ctx['families']:
                x=part[part.feature_set.eq(family)].sort_values(['MAE_DEV','model']).head(1)
                if len(x):
                    winner=x.iloc[0]
                    f=representative[(representative.horizon==h)&representative.protocol.eq(protocol)&
                                     representative.model.eq(winner.model)&representative.feature_set.eq(family)]
                    choice[family]=(winner.model,f)
                    family_choices[h,protocol,family]=winner.model
            for hyp,bf,cf in [('H1','A_history','B_economy'),('H2','B_economy','C_housing_services'),('H3','C_housing_services','D_spatial')]:
                required={'H1':{'wage','employment','unemployment','regional_median_wage','regional_employment','regional_wage','regional_unemployment'},
                    'H2':{'housing_new','housing_individual','school_count','student_count','preschool_count','preschool_capacity'},
                    'H3':{'neighbor_rate'}}[hyp]
                needed_family={'H1':'B_economy','H2':'C_housing_services','H3':'D_spatial'}[hyp]
                if not required.intersection(ctx['available'].get(needed_family,[])):
                    hypothesis.append({'hypothesis':hyp,'horizon':h,'protocol':protocol,'status':'NOT_FEASIBLE',
                                       'reason':'Prerequisite substantive indicator(s) not validated: '+','.join(sorted(required))})
                    continue
                if bf not in choice or cf not in choice:
                    hypothesis.append({'hypothesis':hyp,'horizon':h,'protocol':protocol,'status':'NOT_FEASIBLE',
                                       'reason':f'No scientifically distinct {bf} versus {cf}'})
                    continue
                bn,b=choice[bf];cn,cand=choice[cf]
                x=b.assign(model='reference_ablation');y=cand.assign(model='candidate_ablation')
                result=paired_bootstrap(pd.concat([x,y],ignore_index=True),'candidate_ablation','reference_ablation')
                hypothesis.append({'hypothesis':hyp,'horizon':h,'protocol':protocol,
                    'baseline':bf+'/'+bn,'alternative':cf+'/'+cn,**result})
            base=representative[(representative.horizon==h)&representative.protocol.eq(protocol)&
                                representative.model.eq('last_known')]
            for fs,(name,cand) in choice.items():
                comparison=paired_bootstrap(pd.concat([base,cand],ignore_index=True),name,'last_known')
                tests.append({'horizon':h,'protocol':protocol,'comparison':f'{fs}/{name} vs last_known',**comparison})
        hdf=pd.DataFrame(hypothesis);testdf=pd.DataFrame(tests)
        if len(hdf) and 'p_boot' in hdf:
            hdf['p_holm']=holm(pd.to_numeric(hdf.p_boot,errors='coerce'))
            hdf['decision']=np.where(hdf.status.eq('OK'),
       np.where((hdf.p_holm<.05)&(hdf.CI_low>0),'SUPPORTED','NOT_SUPPORTED'),'NOT_TESTED')
        if len(testdf) and 'p_boot' in testdf:
            testdf['p_fdr']=benjamini_hochberg(pd.to_numeric(testdf.p_boot,errors='coerce'))
        representative['abs_error']=(representative.prediction-representative.target_rate).abs()
        by_year=representative.groupby(['horizon','protocol','target_year','model','feature_set'],as_index=False).agg(
            MAE=('abs_error','mean'),n=('row_id','size'))
        by_type=representative.groupby(['horizon','protocol','mun_type','model','feature_set'],dropna=False,as_index=False).agg(
            MAE=('abs_error','mean'),n=('row_id','size'),population=('population_exposure','sum'))
        by_region=representative.groupby(['horizon','protocol','region_id','model','feature_set'],as_index=False).agg(
            MAE=('abs_error','mean'),n=('row_id','size'))
        # H4 heterogeneity of improvement by objectively observed municipality type.
        h4=[]
        for (h,protocol),ctx in sorted(contexts.items()):
            available_families=[f for f in ('E_all','D_spatial','C_housing_services','B_economy') if (h,protocol,f) in family_choices]
            primary_family=available_families[0] if available_families else None
            choices=family_choices.get((h,protocol,primary_family)) if primary_family else None
            if not choices or (h,protocol,'A_history') not in family_choices:continue
            a=representative[(representative.horizon==h)&representative.protocol.eq(protocol)&representative.feature_set.eq('A_history')&
                             representative.model.eq(family_choices[h,protocol,'A_history'])][['row_id','mun_type','region_id','target_rate','prediction']]
            b=representative[(representative.horizon==h)&representative.protocol.eq(protocol)&representative.feature_set.eq(primary_family)&
                             representative.model.eq(choices)][['row_id','prediction']].rename(columns={'prediction':'candidate'})
            z=a.merge(b,on='row_id',validate='1:1');z['advantage']=(z.target_rate-z.prediction).abs()-(z.target_rate-z.candidate).abs()
            for kind,sub in z.groupby('mun_type',dropna=False):
                if len(sub)>=30:
                    h4.append({'hypothesis':'H4','horizon':h,'protocol':protocol,'type':str(kind),
                               'n':len(sub),'regions':sub.region_id.nunique(),'mean_improvement':sub.advantage.mean(),
                               'status':'EXPLORATORY'})
        h4df=pd.DataFrame(h4)
        supported=hdf[(hdf.get('decision',pd.Series(index=hdf.index,dtype=str))=='SUPPORTED')] if len(hdf) else pd.DataFrame()
        h5=[]
        for hyp in ('H1','H2','H3'):
            v=supported[supported.hypothesis.eq(hyp)] if len(supported) else pd.DataFrame()
            unique_protocols=set(v.protocol) if len(v) else set()
            evaluated=set(hdf[(hdf.hypothesis==hyp)&(hdf.status=='OK')].protocol) if len(hdf) else set()
            valid_transfer=('ROLLING_TIME' in evaluated and 'CROSSED' in evaluated)
            h5_status=('NOT_FEASIBLE' if not valid_transfer else
                       'SUPPORTED_TRANSFER' if ('ROLLING_TIME' in unique_protocols and 'CROSSED' in unique_protocols)
                       else 'NOT_CONFIRMED')
            h5.append({'hypothesis':'H5','subhypothesis':hyp,'status':h5_status,
                'supporting_protocols':','.join(sorted(unique_protocols)),
                'reason':'Different evaluation regimes may still share observations; not independent causal replications'})
        h5df=pd.DataFrame(h5)
        # Correlation stability, publication-lag sensitivity, explicit non-identified outputs.
        sensitivity=[]
        for lag in (1,2,3):
            for h in HORIZONS:
                alt,qa_alt=gate_panel(panel_for_h(d,h,lag=lag),h)
                if qa_alt['status']!='PASS':
                    sensitivity.append({'lag_years':lag,'horizon':h,'status':'NOT_FEASIBLE'});continue
                s,issues=plan_splits(alt)
                if 'ROLLING_TIME' not in s:
                    sensitivity.append({'lag_years':lag,'horizon':h,'status':'NOT_FEASIBLE_SPLIT'});continue
                test=s['ROLLING_TIME']['test']
                sensitivity.append({'lag_years':lag,'horizon':h,'status':'OK',
                    'n_test':len(test),'MAE_last_known':score(test.target_rate,test.population_exposure,test.lag0)['MAE']})
        sensdf=pd.DataFrame(sensitivity)
        if warning_counts:
            for msg,count in warning_counts.most_common(15):
                logger.warning('Model warning (%d occurrences): %s',count,msg)
            if any('converge' in msg.lower() or 'iterations reached' in msg.lower() for msg in warning_counts):
                limitations.append({'subject':'MODEL_CONVERGENCE','status':'WARN',
                    'reason':'Some optimization runs emitted convergence warnings; inspect technical.log'})
        if recovery_of_late_export and counts['recovered_unrecorded']:
            limitations.append({'subject':'OPTUNA_RECOVERY','status':'LIMITATION',
                'reason':f"{counts['recovered_unrecorded']} pruned/budget-skipped trials had no checkpoint in V3.3; only verified completed trials used for model selection"})
        summary_status='COMPUTE_COMPLETE_WITH_LIMITATIONS' if limitations or problems else 'FULL_LOCAL_COMPUTE_COMPLETE'
        tick(job)
        # Protect against accidental reuse of metadata identifiers during long runs.
        assert isinstance(source_asset_d01,dict) and isinstance(source_asset_d01.get('sha256'),str), 'Lost immutable D01 source metadata'
        # Export in two bound products plus report/manifest, with no re-publication of raw confidential sources.
        job=('EXPORT','all',0,0,0,0,0)
        chosen_specs=pd.DataFrame([v['spec'] for v in selected])
        factor_coverage,factor_corr,factor_lineage=(factor_diagnostics_v3(territory_panel[0],selected)
                                        if territory_panel else (pd.DataFrame(),pd.DataFrame(),pd.DataFrame()))
        optuna_report=build_optuna_report_v331(dev_stats,dev_selection)
        if len(dev_selection)!=int(optuna_report['selected_optuna'].sum()):
            raise ValueError('Optuna report selected-trial count differs from saved DEV selections')
        if len(dev_selection) and not np.isfinite(optuna_report.loc[optuna_report['selected_optuna'],'MAE_DEV_FULL']).all():
            raise ValueError('Selected Optuna models require finite FULL-DEV scores')
        (root/'OPTUNA_SELECTION_QA_V3_3_1.json').write_text(json.dumps({
            'version':NOTEBOOK_PATCH,'schema':'MAE_DEV_SEARCH != MAE_DEV_FULL',
            'completed_trials':len(optuna_report),
            'selected_models':len(dev_selection),
            'selected_trial_rows':int(optuna_report['selected_optuna'].sum()),
            'resumed_after_late_crash':recovery_of_late_export,
            'unrecorded_pruned_or_budget_trials':counts['recovered_unrecorded']
            },ensure_ascii=False,indent=2),encoding='utf-8')
        scenario_coverage=pd.DataFrame([{'horizon':h,'protocol':protocol,'forecast_type':ctx['forecast_type'],
             'families':','.join(ctx['families']),
             'dev_design':'ONE_STEP_PROXY_NOT_DIRECT_H' if (ctx['forecast_type'].startswith('RECURSIVE') and
                       horizon_development_panel(d,h,ctx['split'])[0] is None) else 'HORIZON_OR_DIRECT_DEV'}
             for (h,protocol),ctx in contexts.items()])
        profiles=profiles_from_d01(d)
        issues_frame=pd.DataFrame(limitations+problems+family_audit+spatial_audit)
        gatesframe=pd.DataFrame(panel_info)
        report=(f'# ML-1 V3: миграционная привлекательность муниципалитетов\n\n'
            f'## Фактически выполненный анализ\nИсходный источник D01: {source_asset_d01["sha256"]}. '
            f'Наблюдений {qa["observations"]}, муниципалитетов {qa["territories"]}, региональных групп {qa["regions"]}; '
            f'годы {min(qa["years"])}–{max(qa["years"])}. Коэффициент приведён в человек на 1000 жителей '
            f'с масштабом исходного ряда ×{qa["rate_multiplier"]}, проверенным на {qa["rate_inference"]["support"]} парах.\n\n'
            f'## Реальные модели и данные\nУспешных метрик регрессии: {len(mdf)}; модельных прогнозов: {len(allpred)}; '
            f'50-seed завершений: {len(completed_seeds)} из {50*len(seed_contexts)} запланированных. '
            f'Внешних факторных индикаторов: {len(selected)}, из них региональных: {sum(x.get("level")=="region" for x in selected)}. Пространственных наборов: '
            f'{sum("D_spatial" in c["families"] for c in contexts.values())}. '
            f'Ошибок: {len(problems)}, ограничений: {len(limitations)}.\n\n'
            f'## Оптимизация\nOptuna/TPE, n_trials={OPTUNA_TRIALS}. Вся настройка только на FIT/DEV; ' +
            f'для рекурсивных горизонтов статус многолетнего DEV фиксируется в таблице DEV_подбор.\n\n' +
            f'## Гипотезы\nПодтверждающие H1–H3 сравнения выполнены на одинаковых test keys, p Holm. '
            f'Поддержано: {int((hdf.get("decision",pd.Series(dtype=str))=="SUPPORTED").sum())}. '
            f'H4 описывает неоднородность, H5 проверяет переносимость между режимами при условной независимости. '
            f'Неисполнимые гипотезы остаются явно размеченными.\n\n'
            f'## Границы достоверности\nТекущая выгрузка — ретроспективный срез; отсутствие подлинных '
            f'исторических vintages ограничивает строгий backtest. Граница территории и тип уровня проверяются '
            f'по доступным кодам и метаданным, полная юридическая преемственность требует архивной географии. '
            f'Значимость и SHAP — прогнозные связи, не эффект миграционной политики. '
            f'Отчёт не предполагает автоматически подтверждённые гипотезы без фактических тестов.\n')
        for limit in limitations[:80]:
            report+=f'\n- {limit.get("status")}: {limit.get("subject")} — {limit.get("reason", "")}'
        # Empirical report, directly derived from saved test results; no assumed positive findings.
        report+='\n\n## Сравнение на закрытых тестах (человек на 1000 жителей)\n'
        for (hh,proto),data in sorted(contexts.items()):
            summary=mdf[(mdf.horizon.eq(hh))&(mdf.protocol.eq(proto))&(mdf.seed.isin([0,11]))]
            baseline=summary[summary.model.eq('last_known')]
            finalists=summary[summary.MAE_DEV.notna()].sort_values(['MAE_DEV','model','feature_set'])
            if len(baseline) and len(finalists):
                ref=float(baseline.MAE.iloc[0]);winner=finalists.iloc[0]
                improvement=(ref-float(winner.MAE))/ref*100 if ref else np.nan
                report+=(f'\n- h={hh}, {proto}: последний известный {ref:.4f}; '
                         f'зафиксированная по DEV {winner.feature_set}/{winner.model}: {winner.MAE:.4f}; '
                         f'изменение ошибки {improvement:+.2f}%. DEV={winner.MAE_DEV:.4f}.')
        report+='\n\n## Гипотезы и множественные проверки\n'
        if len(hdf):
            for hyp,part in hdf.groupby('hypothesis'):
                hypothesis_status_counts=part.status.fillna('TESTED').value_counts().to_dict()
                supported=int((part.get('decision',pd.Series(index=part.index)).eq('SUPPORTED')).sum())
                report+=f'\n- {hyp}: протоколов {len(part)}, статусы {hypothesis_status_counts}; статистически поддержано: {supported}.'
        report+='\n\n## Факторные источники (не подменять региональный показатель муниципальным)\n'
        if selected:
            for feature in selected:
                spec=feature['spec']
                report+=(f'\n- {spec["feature"]}: {spec["indicator_name"]}; код {spec["indicator_code"]}; '
                         f'единица «{spec["indicator_unit"]}»; уровень {spec["scope"]}; '
                         f'пересекающиеся годы {spec["years_overlap"]}; '
                         f'качество: {spec["status"]}, конфликтных пар {spec["n_conflict"]}.')
        else:report+='\n- Нет внешних показателей, удовлетворивших контракту качества; H1–H3 не опровергнуты, а не проверены.'
        report+='\n\n## Оптимизация и интерпретация\n'
        report+=(f'\n- Optuna/TPE: {len(dev_stats)} завершенных trial, точные параметры и DEV-мера в '
                 'DEV_подбор; итоговые метрики закрытого TEST не используются для выбора модели.')
        report+=(f'\n- Интерпретация: permutation/grouped permutation, SHAP при технической поддержке, '
                 'ALE, PDP и взаимодействия. Все значения обозначают прогностические ассоциации, а не '
                 'причинный эффект экономической или миграционной политики.')
        report+='\n- Пространственная география при соответствии — только прокси по реальным поселениям; '
        report+='исторический административный контур и доступность этого справочника на прошлую дату не установлены.'
        report+='\n- Для рекурсивных горизонтов экзогенные переменные заморожены на информационном срезе origin; '
        report+='при недостатке ранних меток многолетний DEV заменён открыто обозначенной одногодичной прокси.'
        report+='\n\n## Практическая применимость\nРезультаты описывают прогнозную неопределённость и '
        report+='не допускают автоматической политики изменения миграции. Отрицательный выигрыш относительно '
        report+='baseline, отсутствие пригодной выборки и неустойчивые значения SHAP не удаляются из отчёта.'
        root.joinpath('REPORT.md').write_text(report,encoding='utf-8')
        imagefiles=study_figures(mdf,representative,impdf,interval_df,aledf,by_type,
                                pd.DataFrame(family_audit),interdf,root/'_figures')
        extra_plot=make_factor_correlation_plot(factor_corr,root/'_figures')
        if extra_plot is not None and len(imagefiles)<10:imagefiles.append(extra_plot)
        # V4 — фиксированный набор из 4 книг и одного архива рисунков; Parquet остаётся
        # внутренним checkpoint, но не рассыпается на десятки пользовательских файлов.
        results_tables={
            'Паспорт':pd.DataFrame([{'Показатель':'Версия','Значение':VERSION},
                {'Показатель':'Источник_SHA256','Значение':source_asset_d01['sha256']},
                {'Показатель':'Наблюдений','Значение':len(d)},
                {'Показатель':'Муниципалитетов','Значение':qa['territories']},
                {'Показатель':'Единица','Значение':'человек на 1000 жителей'},
                {'Показатель':'Статус','Значение':summary_status}]),
            'Гипотезы_H1_H3':hdf,'H4_типология':h4df,'H5_переносимость':h5df,
            'Модельные_метрики':mdf,'Стат_тесты':testdf,'По_годам':by_year,
            'По_регионам':by_region,'По_типам':by_type,'Профили':profiles,
            'Устойчивость':sensdf,'Сиды':pd.DataFrame(completed_seeds),
            'Прогнозные_интервалы':interval_df,'Классификация':cls_df,
            'Калибровка':cal_df,'Потоки':secondary_df,
            'Ограничения':issues_frame,
        }
        predictions_tables={
            **{f'Прогноз_h{hh}':allpred.loc[allpred.horizon.eq(hh),
                 [k for k in ['row_id','territory_id','region_id','mun_type','origin_year',
                     'target_year','information_cutoff','max_feature_year','horizon',
                     'target_rate','prediction','model','feature_set','protocol','seed',
                     'max_label_year_fit','population_exposure'] if k in allpred.columns]]
                 for hh in HORIZONS},
            'Абс_потоки':secondary_pred,
            'Классиф_прогнозы':cls_pred,'Интервальные_прогнозы':interval_pred,
        }
        data_qa_tables={
            'Цели_приёмка':v4_goals_matrix(hdf,h4df,h5df,selected,spatial_audit,
                completed_seeds,seed_contexts,problems,allpred,contexts,extra={
          'metrics':mdf,'optuna':optuna_report,'importance':impdf,'shap':shapdf,
          'ale':aledf,'intervals':interval_df,'classification':cls_df,
          'absolute_flows':secondary_df,'figures':imagefiles}),
            'Данные_QA':gatesframe,'Источники':pd.DataFrame(input_hashes),
            'Каталог_факторов':catalog,'Отбор_факторов':indicator_audit,
            'Выбранные_коды':chosen_specs,'Срезы_протоколов':pd.DataFrame(split_audit),
            'Факторные_группы':pd.DataFrame(family_audit),
            'Пространство_QA':pd.DataFrame(spatial_audit),
            'Факторы_покрытие':factor_coverage,
            'Факторы_корреляции':factor_corr,'Происхождение':factor_lineage,
            'Прогнозный_дизайн':scenario_coverage,
        }
        model_qa_tables={
            'Optuna_DEV':optuna_report,'Optuna_время':pd.DataFrame(optuna_time_ledger),
            'Важность_XAI':impdf,'SHAP':shapdf,'ALE':aledf,
            'Взаимодействия':interdf,'Сходимость_предупр':v4_warning_table(warning_counts),
            'Ошибки_ограничения':issues_frame,
            'Разбиения':pd.DataFrame(split_audit),
            'Протоколы_тестов':v4_tests_ledger(root),
        }
        if data_qa_tables['Цели_приёмка']['Статус'].eq('FAIL').any():
            failures=data_qa_tables['Цели_приёмка'].loc[data_qa_tables['Цели_приёмка']['Статус'].eq('FAIL'),'Цель'].tolist()
            raise RuntimeError('V4 scientific acceptance GATE failed: '+repr(failures))
        output_meta=v4_export_bundle(root,results_tables,predictions_tables,
               data_qa_tables,model_qa_tables,imagefiles,report)
        if 'completed' not in counts or not isinstance(counts['completed'],int):
            raise RuntimeError('V4 counter shadowing guard FAILED before VERIFY')
        tick(job)
        job=('VERIFY','all',0,0,0,0,0)
        manifest={'version':VERSION,'notebook_patch':NOTEBOOK_PATCH,'mode':mode,'optuna_trials':OPTUNA_TRIALS,'status':summary_status,
            'scientific_acceptance':'NOT_ASSESSED_PENDING_REVIEW','created_at':utc(),'source_meta':source_manifest,
            'data_qa':qa,'input_assets':input_hashes,'gates':panel_info,
            'indicator_audit':{'catalog_rows':len(catalog),'selected':len(selected)},
            'n_contexts':len(contexts),'search_runtime':optuna_time_ledger,
            'search_fit_max_rows':OPTUNA_FIT_MAX_ROWS,'search_dev_max_rows':OPTUNA_DEV_MAX_ROWS,
            'n_model_metrics':len(mdf),'n_optuna_trials':len(optuna_report),'n_prediction_rows':len(allpred),
            'n_models':int(mdf[['horizon','protocol','feature_set','model']].drop_duplicates().shape[0]),
            'seed_done':completed_seeds,'seed_planned':50*len(seed_contexts),
            'completed_jobs':completed_jobs,'failed_jobs':problems,'limitations':limitations,
            'workplan_jobs':len(work),'workplan_sha256':plan_hash,'work_completed':counts,
            'requested_mode':requested_mode,'checkpoint_recovery_audit':checkpoint_audit,
            'future_mutation_canary':'PASS','publication_lag_years':SOURCE_LAG_YEARS,'recursive_exogenous_scenario':'FROZEN_ORIGIN_CUTOFF',
            'publication_lag_sensitivity':[1,2,3],
            'target_unit':'persons per 1000 residents','python':platform.python_version(),
            'pandas':pd.__version__,'elapsed_seconds':time.monotonic()-started,
            'outputs':output_meta,
            'publication':{'github':'PENDING_POST_VERIFY','release':'PENDING_POST_VERIFY'}}
        write_json(root/'run_manifest.json',manifest)
        verified=verify_local(root)
        tick(job);progress.close()
        if recovery_of_late_export:
            failed_marker=root/'FAILED_RUN.json'
            if failed_marker.is_file():
                original=root/'FAILED_RUN_RECOVERED.json'
                # A previous failed marker is retained in evidence; never silently erased.
                failed_marker.replace(original)
                manifest['recovered_failure_marker']=str(original.name)
        manifest['local_verification']=verified;manifest['work_completed']=counts
        write_json(root/'run_manifest.json',manifest)
        print('ML-1 V4',summary_status,'| tested contexts:',len(contexts),'| rates:',len(mdf),
              '| completed stochastic seeds:',len(completed_seeds),'| errors:',len(problems))
        print('V4 compact package:',list(output_meta),'| SHA256:',verified['sha256_verified'])
        return manifest
    except Exception as exc:
        progress.close()
        write_json(root/'FAILED_RUN.json',{'status':'COMPUTE_INCOMPLETE','timestamp':utc(),
          'failed_exception':repr(exc),'traceback_tail':traceback.format_exc()[-4000:],
          'workplan_sha256':plan_hash,'jobs_completed':counts,'output_dir':str(root)})
        try:
            if len(list(store.glob('*.meta.json')))>100:
                backup=root/'ML1_RECOVERY_CHECKPOINTS.zip'
                temp=backup.with_suffix('.temporary.zip')
                with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
                    for p in sorted(store.iterdir()):
                        if p.is_file() and not '.temporary.' in p.name:
                            z.write(p,arcname='checkpoint/'+plan_hash+'/'+p.name)
                    for name in ('checkpoint_plan.json','FAILED_RUN.json','technical.log'):
                        p=root/name
                        if p.is_file():z.write(p,arcname=name)
                with zipfile.ZipFile(temp) as z:
                    assert z.testzip() is None
                temp.replace(backup)
                print('CHECKPOINT BACKUP READY:',backup,'SHA256:',sha256_file(backup))
                if globals().get('AUTO_DOWNLOAD',False):
                    try:
                        from google.colab import files
                        files.download(str(backup))
                    except Exception as download_error:
                        print('Recovery backup can be downloaded manually:',repr(download_error))
        except Exception as zip_error:
            print('Checkpoint backup failed; original exception preserved:',repr(zip_error))
        raise


def smoke_suite_v2():
    # Synthetic fixtures test algorithmic invariants ONLY. Never called FULL research data.
    from sklearn.metrics import mean_absolute_error
    r=[]
    for i in range(32):
        for year in range(2010,2024):
            net=10*(i-6)+year-2010
            pop=3000+i*100
            r.append({'oktmo':str(10000000+i),'year':year,'region':'R'+str(i//4),
                      'municipality':'M'+str(i),'mun_type':'Муниципальный район',
                      'population':pop,'average_population':pop,
                      'migration':net,'migration_rate':net/pop})
    d,qa=clean_d01(pd.DataFrame(r))
    assert qa['rate_multiplier']==1000
    assert d.net_verified.mean()>.99
    assert abs(d.iloc[0].migration_rate - (d.iloc[0].migration_net/d.iloc[0].population_exposure*1000))<1e-5
    assert future_canary(d)
    assert panel_for_h(d,5).target_cum_net.notna().any()
    # duplicated municipality-year rows must be rejected, never expanded.
    try:clean_d01(pd.concat([pd.DataFrame(r),pd.DataFrame(r).iloc[:1]],ignore_index=True))
    except ValueError:pass
    else:raise AssertionError('Duplicate JOIN canary failed')
    p=panel_for_h(d,1)
    assert p.max_feature_year.le(p.information_cutoff).all()
    assert p.row_id.is_unique
    assert not p.stable_outflow_3y.dropna().empty
    g,audit=gate_panel(p,1);assert g.shape[0]>100
    folds,flags=plan_splits(g);assert 'ROLLING_TIME' in folds
    # baseline, grouping and bootstrap
    a=p.dropna(subset=['target_rate','lag0'])
    assert np.isfinite(score(a.target_rate,a.population_exposure,a.lag0)['MAE'])
    assert len(profiles_from_d01(d))==32
    # Regression canary for the actual error: nonmatching pandas indexes must never inject NaN.
    left=pd.DataFrame({'row_id':['a','b'],'stable_outflow_3y':[1.,0.]},index=[104,207])
    right=pd.DataFrame({'row_id':['b','a'],'prediction_stable3':[0.,1.]},index=[0,1])
    merged=align_and_validate_binary(left,right,'stable_outflow_3y','prediction_stable3')
    assert merged.row_id.tolist()==['a','b'] and merged.truth.tolist()==[1.,0.]
    assert merged.probability.tolist()==[1.,0.]
    try:validate_binary_predictions(pd.DataFrame({'row_id':['a'],'truth':[1.], 'probability':[np.nan]}))
    except ValueError:pass
    else:raise AssertionError('Nonfinite probability must not reach Brier metric')
    try:align_and_validate_binary(left,right.iloc[:1],'stable_outflow_3y','prediction_stable3')
    except ValueError:pass
    else:raise AssertionError('Unmatched test row must stop classification')
    print('SMOKE PASS: 13 checks including row_id alignment and NaN probability canary')
    return {'status':'SMOKE_PASS','checks':13,'synthetic_fixture_only':True,
            'version':VERSION,'patch':NOTEBOOK_PATCH}


## 7. Независимые тесты до полного расчёта
**V4.4:** критическая регрессия экспортного контракта: 4 Excel + графический ZIP, SHA-256, чтение листов, проверка архива, порча файла и запрет принятия манифеста V2/V3. Этот тест выполняется *до* FULL.


In [ ]:

# Изолированные unit + integration tests на миниатюрных СИНТЕТИЧЕСКИХ фикстурах.
# Эти строки НИКОГДА не смешиваются с фактической научной панелью.
def run_preflight_tests_v4():
    import tempfile
    from openpyxl import load_workbook
    import pyarrow.parquet as pq
    tests=[]
    def check(label, f):
        try:
            f(); tests.append({'test':label,'status':'PASS'})
        except Exception as e:
            tests.append({'test':label,'status':'FAIL','error':repr(e)[:240]})
            raise AssertionError(f'PRE-FLIGHT FAIL: {label}: {e}') from e
    check('V2.2 scientific invariants (13)',lambda:smoke_suite_v2())
    n=41
    example=pd.DataFrame({'row_id':['T'+str(i) for i in range(n)],
        'truth':np.linspace(-.04,.04,n),'prediction':np.linspace(-.039,.041,n),
        'probability':np.linspace(0,1,n)})
    example['bin']=pd.cut(example['probability'],np.linspace(0,1,11),include_lowest=True)
    example['mixed']=[{'n':i,'group':[i,i+1]} if i%3==0 else ('A',i) if i%3==1 else None
                      for i in range(n)]
    example['period']=pd.period_range('2010',periods=n,freq='Y')
    example['categorical']=pd.Series(['urban','rural']*(n//2)+['urban'],dtype='category')
    def test_normalization():
        q,log=normalize_parquet_schema(example,'regression_fixture')
        assert len(q)==n and q.row_id.tolist()==example.row_id.tolist()
        assert q.bin.dtype.name=='string' and q.period.dtype.name=='string'
        assert q.mixed.isna().sum()==example.mixed.isna().sum()
        assert np.allclose(q.truth,example.truth)
        assert any(s['reason']=='categorical_to_string' for s in log)
    check('Interval / Period / Categorical / nested object conversions',test_normalization)
    def test_evidence():
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);z=root/'ML1_EVIDENCE.zip'
            # Parquet table names and scientific keys mirror FULL output.
            m=export_evidence(z,{'predictions':example,'splits':example[['row_id','truth']],
                    'calibration':example[['row_id','bin','probability']],
                    'nested_qa':example[['row_id','mixed','period']]},[],
                    '# Methodology test',{'version':'v2.3-fixture','fixture_only':True})
            with zipfile.ZipFile(z) as arc:
                assert arc.testzip() is None
                from io import BytesIO
                r=pd.read_parquet(BytesIO(arc.read('predictions.parquet')))
                assert len(r)==n and not (r.truth-r.prediction).isna().any()
                assert r.bin.iloc[0]==str(example.bin.iloc[0])
                assert r.probability.equals(example.probability)
                assert 'serialization_qa.parquet' in arc.namelist()
                assert pd.read_parquet(BytesIO(arc.read('serialization_qa.parquet'))).shape[0]>=3
                assert pq.ParquetFile(BytesIO(arc.read('predictions.parquet'))).metadata.num_rows==n
            assert sha256_file(z)==m['sha256']
    check('Evidence ZIP: ALL Parquet write/read/CRC/SHA/row counts',test_evidence)
    def test_excel():
        with tempfile.TemporaryDirectory() as td:
            name=Path(td)/'results.xlsx'
            meta=export_excel(name,{'Тест_данных':example[['row_id','truth','bin']]})
            wb=load_workbook(name,read_only=True)
            assert wb.sheetnames==['Тест_данных']
            assert v4_excel_actual_rows(wb['Тест_данных'],n+1)==n+1
            wb.close()
            assert sha256_file(name)==meta['sha256']
    check('Excel readback 42 rows and SHA',test_excel)
    def test_checkpoint():
        # FULL is always computed anew; test isolated cache on a small numeric fixture.
        with tempfile.TemporaryDirectory() as td:
            j=('TEST',1,'PARQUET')
            first,was_resumed=checkpoint_job(j,td,lambda:example[['row_id','truth']],mode='FULL')
            second,restored=checkpoint_job(j,td,lambda:(_ for _ in ()).throw(RuntimeError('bad resume')),
                                           mode='RESUME')
            assert not was_resumed and restored and first.equals(second)
    check('Checkpoint SHA / restoration (not used by FULL)',test_checkpoint)
    def test_end_to_end():
        # V4 contract: four Excel workbooks and one figures archive, NOT V2/V3 artifacts.
        with tempfile.TemporaryDirectory() as tmp:
            root=Path(tmp)
            from matplotlib import pyplot as plt
            fig,ax=plt.subplots(figsize=(3.2,1.8));ax.plot([1,2,3],[1,1.5,1.2]);ax.set_title('SYNTHETIC TEST ONLY')
            figure=root/'fixture_plot.png';fig.savefig(figure,dpi=90,bbox_inches='tight');plt.close(fig)
            results={'Паспорт':pd.DataFrame({'Показатель':['Тест'],'Значение':['SYNTHETIC']}),
                     'Метрики':example[['row_id','truth','prediction']]}
            predictions={'Прогнозы':example[['row_id','truth','prediction','probability']]}
            dataqa={'Структура':pd.DataFrame({'status':['PASS'],'rows':[n]})}
            modelqa={'Контракт':pd.DataFrame({'contract':['V4_5_FILES'],'status':['PASS']})}
            outputs=v4_export_bundle(root,results,predictions,dataqa,modelqa,[figure],
                                     'Synthetic fixture only — not empirical results')
            assert set(outputs)==set(V4_OUTPUT_FILES) and len(outputs)==5
            write_json(root/'run_manifest.json',{'version':VERSION,'outputs':outputs,'n_prediction_rows':n})
            result=verify_local(root)
            assert result['status']=='PASS_LOCAL_ONLY' and result['sha256_verified']
            assert len(result['files'])==5
            from openpyxl import load_workbook
            wb=load_workbook(root/'ML1_V4_RESULTS.xlsx',read_only=True)
            assert set(wb.sheetnames)=={'Паспорт','Метрики'} and v4_excel_actual_rows(wb['Метрики'],n+1)==n+1
            wb.close()
            with zipfile.ZipFile(root/'ML1_V4_FIGURES.zip') as arc:
                assert arc.testzip() is None
                assert 'figures/fixture_plot.png' in arc.namelist()
                figure_meta=json.loads(arc.read('MANIFEST.json'))
                assert figure_meta['sha256']['fixture_plot.png']==sha256_file(figure)
            package=assemble_v4_delivery(root)
            with zipfile.ZipFile(package) as arc:
                assert arc.testzip() is None and set(arc.namelist())==set(V4_OUTPUT_FILES)
                for name in V4_OUTPUT_FILES:
                    assert hashlib.sha256(arc.read(name)).hexdigest()==outputs[name]['sha256']
            # Negative canaries: missing artifact, wrong hash, V3 manifest cannot pass V4.
            victim=root/'ML1_V4_DATA_QA.xlsx'; bytes0=victim.read_bytes();victim.write_bytes(bytes0+b'altered')
            try:verify_local(root)
            except ValueError as exc:assert 'SHA mismatch' in str(exc)
            else:raise AssertionError('Tampered XLSX was accepted')
            victim.write_bytes(bytes0)
            write_json(root/'run_manifest.json',{'outputs':{'ML1_MIGRATION_RESULTS.xlsx':outputs['ML1_V4_RESULTS.xlsx']},'n_prediction_rows':n})
            try:verify_local(root)
            except ValueError as exc:assert 'V4 output file set mismatch' in str(exc)
            else:raise AssertionError('Legacy V3 output manifest was accepted')
    check('V4: 4 Excel + graphs ZIP + manifest + SHA + delivery + corruption canaries',test_end_to_end)
    assert len(tests)==6 and all(r['status']=='PASS' for r in tests)
    print('V4 PRE-FLIGHT PASS: 6 independent groups, V4 five-file contract, 13 scientific smoke invariants.')
    return tests

# Fail immediately if a required function is not loaded in Run all sequence.
_V4_REQUIRED_BEFORE_PREFLIGHT = (
    'sha256_file','smoke_suite_v2','normalize_parquet_schema','export_evidence',
    'export_excel','checkpoint_job','v4_export_bundle','v4_workbook',
    'verify_local','assemble_v4_delivery')
_V4_MISSING_BEFORE_PREFLIGHT = [name for name in _V4_REQUIRED_BEFORE_PREFLIGHT
                                if not callable(globals().get(name))]
if _V4_MISSING_BEFORE_PREFLIGHT:
    raise RuntimeError('V4.3 RUN ALL ORDER FAILURE: missing functions '+repr(_V4_MISSING_BEFORE_PREFLIGHT))
print('V4.3 RUN ALL ORDER: PASS |',len(_V4_REQUIRED_BEFORE_PREFLIGHT),'functions ready')
PRE_FLIGHT_RESULTS=run_preflight_tests_v4()
write_json(Path(RESULT_DIR)/'PRE_FLIGHT_TESTS.json',{
    'version':VERSION,'status':'PASS','scientific_smoke_checks':13,
    'other_test_groups':5,'results':PRE_FLIGHT_RESULTS,'timestamp':utc()})


In [ ]:
# ML-1 V3: отдельная научно-инженерная регрессия (до загрузки реальных 50+ млн строк).

def run_v3_separate_tests():
    import tempfile
    result=[]
    def check(name,fn):
        fn();result.append({'test':name,'status':'PASS'})
    def duplicates():
        x=pd.DataFrame({'territory_id':['01','01','02','02','03','03'],
            'year':[2020]*6,'value':[4.,4.,7.,8.,5.,5.]})
        y,m=_v3_collapse_series(x,['territory_id','year'],tolerance=.40)
        assert len(y)==2 and m['n_conflicted_keys']==1
        assert y.territory_id.tolist()==['01','03']
        b,_=_v3_collapse_series(x,['territory_id','year'],tolerance=.02)
        assert b.empty  # genuine conflict beyond threshold cannot be aggregated
    check('Factor JOIN: exact duplicates vs disaggregated conflicting values',duplicates)
    def no_lookahead():
        p=pd.DataFrame({'territory_id':['01']*3,'region_id':['A']*3,
            'origin_year':[2020,2021,2022],'information_cutoff':[2018,2019,2020]})
        s={'spec':{'feature':'regional_wage','family':'B_economy','scope':'region',
                   'indicator_code':'Y477170013','source_sha256':'test'},'level':'region',
           'source_frame':pd.DataFrame({'region_id':['A','A'],'year':[2020,2021],
                                        'value':[100.,100000.]})}
        q,features,audit=attach_factors_v2(p,[s]);assert q.regional_wage.isna().iloc[:2].all()
        assert float(q.regional_wage.iloc[2])==100.  # 2021 never enters 2022-origin with 2020 cutoff
        assert len(q)==len(p)
    check('Region covariates: type/territory + publication cutoff',no_lookahead)
    def constrained_region():
        x=pd.DataFrame({'territory_id':['1','1'],'region_id':['A','B'],'year':[2020,2021]})
        assert x.groupby('territory_id').region_id.nunique().gt(1).any()
        # D01 clean_d01 must reject such conflicting keys in the full pipeline
    check('Territory identity collision guard',constrained_region)
    def optuna_checks():
        assert optuna_parameters_smoke().startswith('PASS')
    check('Optuna TPE trial configuration, six estimator families',optuna_checks)
    def recursion_frozen():
        class IdentityPredictor:
            def predict(self,X):
                assert X['regional_wage'].eq(7).all()
                return X['lag0'].astype(float).to_numpy()
        a=pd.DataFrame({'region_id':['R'],'mun_type':['район'],
             'information_cutoff':[2016],'target_year':[2021],'origin_year':[2018],
             'regional_wage':[7.], 'population_log':[10.],
             'lag0':[1.], 'lag1':[1.],'lag2':[1.], 'lag3':[1.], 'lag4':[1.],
             'mean3':[1.],'mean5':[1.],'drift3':[0.]})
        q=forecast_recursively(IdentityPredictor(),a,FEATURE_BASE+['regional_wage'])
        assert len(q)==1 and np.isfinite(q).all() and abs(float(q[0])-1.)<1e-8
        b=a.copy();b['regional_wage']=7.
        assert np.array_equal(q,forecast_recursively(IdentityPredictor(),b,FEATURE_BASE+['regional_wage']))
    check('5-year recursive forecast: frozen exogenous attributes',recursion_frozen)
    def xai_pipeline():
        rng=np.random.default_rng(22)
        from sklearn.ensemble import HistGradientBoostingRegressor
        n=90
        frame=pd.DataFrame({'lag0':rng.normal(size=n),'origin_year':[2018]*n,
            'region_id':['region_a']*n,'mun_type':['округ']*n})
        y=1.5*frame.lag0.to_numpy()+rng.normal(scale=.2,size=n)
        frame['target_rate']=y
        est=HistGradientBoostingRegressor(max_iter=40,max_leaf_nodes=7,random_state=4)
        prep,cols=numeric_design(frame,['lag0','origin_year']);X=frame[cols].astype({'region_id':str,'mun_type':str})
        est.fit(prep.fit_transform(X),y)
        model=FittedTabularRegressor(prep,est,cols)
        # Existing SHAP could be unavailable for some estimator backends: permutation/ALE must survive.
        a,b,c,d,notes=interpret_model_v2(model,frame,frame,['lag0','origin_year'],max_rows=60)
        assert len(a)>0 and a.importance_MAE.notna().any()
        assert 'grouped permutation' in set(a.method)
        assert not d.empty
    check('Model-specific XAI: permutation/grouped, ALE/PDP/ICE',xai_pipeline)
    def factor_csv_and_parquet():
        import duckdb
        with tempfile.TemporaryDirectory() as tmp:
            file=Path(tmp)/'test.parquet'
            frame=pd.DataFrame({'oktmo':['00000001','00000001','00000002'],
                'mun_level':['Муниципальное образование верхнего уровня']*3,
                'region_name':['Область А']*3,'municipality':['район а']*3,
                'year':[2018,2018,2019],'indicator_code':['Y48010001']*3,
                'indicator_name':['Введено в действие жилых домов']*3,
                'indicator_unit':['Квадратный метр общей площади']*3,
                'indicator_value':['7','7','9'],'mosh':[None]*3,'gilsem':[None]*3})
            frame.to_parquet(file,index=False)
            item={'key_column':'oktmo','level_column':'mun_level','indicator_code':'Y48010001',
                'indicator_name':'Введено в действие жилых домов','indicator_unit':'Квадратный метр общей площади',
                'observation_level':'Муниципальное образование верхнего уровня'}
            got,_=_v3_read_series(file,item,'municipality')
            assert len(got)==3
            got['territory_id']=got.source_id.astype(str)
            good,audit=_v3_collapse_series(got,['territory_id','year'])
            assert len(good)==2 and audit['status']=='DEDUP_PASS'
    check('DuckDB×PyArrow: exact catalog code and duplicate Parquet',factor_csv_and_parquet)
    print('ML-1 V3 extra preflight:',len(result),'/',len(result),'PASS')
    return result

V3_SEPARATE_TESTS=run_v3_separate_tests()
write_json(Path(RESULT_DIR)/'PRE_FLIGHT_TESTS_V3.json',{
   'version':VERSION,'status':'PASS','checks':V3_SEPARATE_TESTS,'timestamp':utc()})


In [ ]:
# Отдельные контрактные тесты критического исправления V3.1.
# Выполняются ПЕРЕД 5 000+ задачами и не читают исходные миграционные данные.
def run_v31_checkpoint_optuna_tests():
    import tempfile, optuna
    from pathlib import Path
    with tempfile.TemporaryDirectory() as root:
        root=Path(root)
        job=('OPTUNA',1,'ROLLING_TIME','A_history','ridge',0,0)
        value={'mae':1.275,'per_seed':[1.275],'parameters':{'alpha':1.0},
               'dev_design':'ONE_YEAR_DEV','n_dev':30}
        val,was_resumed=checkpoint_job(job,root,lambda:value,mode='FULL')
        assert not was_resumed and isinstance(val,dict) and val==value
        val2,was_resumed=checkpoint_job(job,root,
            lambda: (_ for _ in ()).throw(RuntimeError('Recalculated unexpectedly')),mode='RESUME')
        assert was_resumed and val2==value
        # Не должны пропускать неверный тип или повреждённый JSON checkpoint.
        try:
            checkpoint_job(('OPTUNA',1,'x','y','z',0,0),root,lambda:3.5,mode='FULL')
            raise AssertionError('Scalar checkpoint was accepted')
        except TypeError as e:
            assert 'Unsupported checkpoint' in str(e)
        import pandas as pd
        frame=pd.DataFrame({'row_id':['001','002'],'prediction':[3.5,4.0]})
        a,res=checkpoint_job(('FINAL',1,'ROLLING_TIME','A_history','ridge',0,11),
            root,lambda:frame,mode='FULL')
        assert not res and a.equals(frame)
        b,res=checkpoint_job(('FINAL',1,'ROLLING_TIME','A_history','ridge',0,11),
            root,lambda:None,mode='RESUME')
        assert res and b.equals(frame)
        # Проверить, что Optuna действительно получает скаляр из dict checkpoint
        study=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=42))
        def objective(trial):
            x=trial.suggest_float('alpha',.05,2.)
            out,_=checkpoint_job(('OPTUNA',1,'TEST','A_history','ridge',trial.number,0),
                root,lambda:{'mae':float((x-.7)**2)},mode='FULL')
            assert isinstance(out,dict)
            trial.report(out['mae'],step=0)
            return out['mae']
        study.optimize(objective,n_trials=4,catch=())
        assert len(study.trials)==4
        assert all(t.state==optuna.trial.TrialState.COMPLETE for t in study.trials)
        assert all(t.value is not None for t in study.trials)
    return 'PASS: dict/Parquet checkpoints + resume SHA + Optuna objectives 4/4'

V31_CHECKPOINT_TEST_RESULT=run_v31_checkpoint_optuna_tests()
print('V3.1 OPTUNA CHECKPOINT PREFLIGHT:',V31_CHECKPOINT_TEST_RESULT)


In [ ]:

# V3.2: отдельные целевые тесты ранее наблюдавшегося падения h3/B_economy
# и сквозная проверка h3/h5 × train/fit/dev/test на несовпадающих row_id.
def run_v32_recursive_factor_tests():
    import tempfile
    audits=[]
    def verify(h):
        n=8
        origins=np.arange(2011,2011+n)
        keys={
            'territory_id':['00000777']*n,
            'origin_year':origins,
            'information_cutoff':origins-2,
        }
        origin_panel=pd.DataFrame(keys)
        origin_panel['row_id']=[f'00000777:{year}:h{h}' for year in origins]
        origin_panel['municipal_sme_support']=np.arange(n,dtype=float)+101.
        origin_panel['regional_wage']=np.arange(n,dtype=float)+700.
        # h1 row IDs MUST differ from h3/h5; these are NEVER join keys.
        training_one_year=origin_panel.drop(columns=['municipal_sme_support','regional_wage']).copy()
        training_one_year['row_id']=[f'00000777:{year}:h1' for year in origins]
        long_test=origin_panel.drop(columns=['municipal_sme_support','regional_wage']).copy()
        for frame in (training_one_year,long_test):
            frame['region_id']='Z';frame['mun_type']='район'
            frame['target_year']=frame.origin_year+(1 if frame is training_one_year else h)
            frame['max_feature_year']=frame.information_cutoff
            for k in range(5):frame[f'lag{k}']=1.25
            frame['mean3']=1.25;frame['mean5']=1.25;frame['drift3']=0.
            frame['population_log']=9.
        split={'train':training_one_year.iloc[:4].copy(),
               'fit':training_one_year.iloc[:3].copy(),
               'dev':training_one_year.iloc[3:5].copy(),
               'test':long_test.iloc[5:].copy()}
        assert 'municipal_sme_support' not in split['test']
        joined,qa=bridge_recursive_factors(split,origin_panel,{
            'B_economy':['municipal_sme_support','regional_wage']})
        assert len(qa)==4 and all(x['status']=='BRIDGE_PASS' for x in qa)
        for phase,block in joined.items():
            assert {'municipal_sme_support','regional_wage'}.issubset(block)
            assert block['municipal_sme_support'].notna().all()
            assert (block.max_feature_year<=block.information_cutoff).all()
            expected=block.origin_year.to_numpy()-2011+101.
            assert np.allclose(block.municipal_sme_support.to_numpy(),expected)
        class HistoricalOnly:
            def predict(self,frame):
                assert frame['municipal_sme_support'].notna().all()
                return frame['lag0'].to_numpy(float)
        cols=FEATURE_BASE+['municipal_sme_support','regional_wage']
        out=forecast_recursively(HistoricalOnly(),joined['test'],cols)
        assert len(out)==len(joined['test']) and np.allclose(out,1.25)
        context={'forecast_type':'RECURSIVE_ENDOGENOUS_1STEP',
                 'families':['A_history','B_economy'],
                 'available':{'B_economy':['municipal_sme_support','regional_wage']},
                 'split':joined}
        gate=preflight_recursive_contexts({(h,'CROSSED'):context})
        assert len(gate)==2 and all(entry['status']=='PASS' for entry in gate)
        missing_test=joined['test'].drop(columns=['municipal_sme_support'])
        try:
            preflight_recursive_contexts({(h,'CROSSED'):{**context,
                'split':{**joined,'test':missing_test}}})
            raise AssertionError('PRE-OPTUNA gate failed to detect absent TEST covariate')
        except ValueError as e:
            assert 'missing columns' in str(e)
        # Guard against accidental injection of a future-year covariate.
        corrupted=origin_panel.copy();corrupted.loc[5,'municipal_sme_support']+=1e7
        already=joined['test'].copy()
        try:
            bridge_recursive_factors({**joined,'test':already},corrupted,{
                'B_economy':['municipal_sme_support','regional_wage']})
            raise AssertionError('Origin-covariate mismatch not caught')
        except ValueError as e:
            assert 'information conflict' in str(e)
        # A missing REAL FEATURE SOURCE (not just NaNs) must fail before Optuna.
        try:
            bridge_recursive_factors({'train':joined['train'],'fit':joined['fit'],
                'dev':joined['dev'],'test':joined['test']},origin_panel.drop(columns=['municipal_sme_support']),
                {'B_economy':['municipal_sme_support']})
            raise AssertionError('Missing source feature not caught')
        except ValueError as e:
            assert 'source lacks validated columns' in str(e)
        return {'horizon':h,'status':'PASS','stages':4,
                'tested':'missing test covariate, h-specific row_id, no future data, conflicts'}
    for h in (3,5):audits.append(verify(h))
    print('V3.2 recursive exogenous factor bridge:',len(audits),'/',len(audits),'PASS')
    return audits


# Предобучение шести РЕАЛЬНЫХ estimator implementations × h3/h5 на маленькой
# детерминированной панели. Будущие исходные данные не привлекаются.
def run_v32_estimator_recursion_tests():
    rng=np.random.default_rng(20261022)
    outcomes=[]
    small_parameters={
        'ridge':{'alpha':1.},
        'elasticnet':{'alpha':.02,'l1_ratio':.5},
        'huber':{'epsilon':1.35,'alpha':.001},
        'histgb':{'max_iter':80,'max_leaf_nodes':9,'learning_rate':.05},
        'catboost':{'iterations':60,'depth':4,'learning_rate':.05,'l2_leaf_reg':5},
        'lightgbm':{'n_estimators':80,'num_leaves':11,'learning_rate':.05,'reg_lambda':1.,
                    'colsample_bytree':.8,'subsample':.8,'subsample_freq':1},
    }
    for horizon in (3,5):
        rows=[]
        for territory in range(18):
            for origin in range(2010,2020):
                r={'territory_id':f'{territory:08d}',
                   'origin_year':origin,'information_cutoff':origin-SOURCE_LAG_YEARS,
                   'region_id':f'R{territory%5}','mun_type':'округ',
                   'row_id':f'{territory:08d}:{origin}:h{horizon}',
                   'target_year':origin+horizon,'max_feature_year':origin-SOURCE_LAG_YEARS,
                   'population_log':np.log(2000+territory*31)}
                for lag in range(5):r[f'lag{lag}']=rng.normal(1,2)
                r['mean3']=np.mean([r[f'lag{k}'] for k in range(3)])
                r['mean5']=np.mean([r[f'lag{k}'] for k in range(5)])
                r['drift3']=(r['lag0']-r['lag2'])/2
                r['target_rate']=1.+.6*r['lag0']+.002*territory+rng.normal(0,.3)
                r['municipal_sme_support']=20+territory+origin*.01
                rows.append(r)
        panel=pd.DataFrame(rows)
        split={}
        for phase,part in [('train',panel[panel.origin_year.le(2016)]),
                           ('fit',panel[panel.origin_year.le(2015)]),
                           ('dev',panel[panel.origin_year.eq(2016)]),
                           ('test',panel[panel.origin_year.ge(2017)])]:
            part=part.copy().drop(columns=['municipal_sme_support'])
            if phase!='test':
                part['row_id']=part.row_id.str.replace(f':h{horizon}',':h1',regex=False)
            split[phase]=part
        split,_=bridge_recursive_factors(split,panel,{'B_economy':['municipal_sme_support']})
        av={'B_economy':['municipal_sme_support']}
        assert len(preflight_recursive_contexts({(horizon,'CROSSED'):{
            'split':split,'forecast_type':'RECURSIVE_ENDOGENOUS_1STEP',
            'families':['A_history','B_economy'],'available':av}}))==2
        cols=feature_cols('B_economy',av)
        for model_name,params in small_parameters.items():
            _,estimator,_=fit_predict(split['train'],split['dev'],cols,model_name,params,seed=11)
            predicted=forecast_recursively(estimator,split['test'],cols)
            assert len(predicted)==len(split['test']) and np.isfinite(predicted).all(),(horizon,model_name)
            outcomes.append({'horizon':horizon,'model':model_name,'status':'PASS',
                             'n_test':len(predicted),'external_features':1})
    print('V3.2 real-estimator smoke:',len(outcomes),'/',12,'PASS')
    return outcomes

V32_ESTIMATOR_TESTS=run_v32_estimator_recursion_tests()

V32_RECURSIVE_TESTS=run_v32_recursive_factor_tests()
write_json(Path(RESULT_DIR)/'PRE_FLIGHT_TESTS_V3_2.json',{
    'version':VERSION,'status':'PASS','tests':V32_RECURSIVE_TESTS,
    'estimators':V32_ESTIMATOR_TESTS,'timestamp':utc()})


## 8. Сохранение источника и публикация в отдельную ветку GitHub

Эксперимент сохраняет воспроизводимый notebook source snapshot (все исследовательские ячейки и научное описание; без выходов выполнения и без вспомогательной ячейки снимка). Его SHA-256 проверяется после коммита в GitHub. В GitHub Release передаются только рассчитанные Excel и Evidence ZIP, опубликованные файлы скачиваются обратно для SHA-256 и проверки структуры. В случае сбоя публикации локальные файлы остаются доступными со статусом `PUBLISH_INCOMPLETE`.

In [ ]:

# Publication is executed ONLY after successful computation and independent local checks.
# One release stores large binary outputs; small reports are committed into own GitHub segment.
# No tokens, credentials, or authorization URLs are printed, logged, or exported.
import base64, tempfile

def _gh_json(session,method,url,**kwargs):
    r=session.request(method,url,timeout=(30,180),**kwargs)
    if r.status_code>=400:
        raise RuntimeError(f'GitHub {method} HTTP {r.status_code} on authorized repository endpoint')
    return r.json() if r.content else {}

def ensure_publication_branch(session,branch=PUBLISH_BRANCH):
    from urllib.parse import quote
    path=GITHUB_API+'/git/ref/heads/'+quote(branch,safe='')
    response=session.get(path,timeout=(30,180))
    if response.status_code==200:return response.json()['object']['sha']
    if response.status_code!=404:raise RuntimeError(f'GitHub branch read HTTP {response.status_code}')
    head=_gh_json(session,'GET',GITHUB_API+'/git/ref/heads/main')['object']['sha']
    _gh_json(session,'POST',GITHUB_API+'/git/refs',json={'ref':'refs/heads/'+branch,'sha':head})
    return head

def put_small_git_file(session,branch,path,contents,message):
    obj={'message':message,'content':base64.b64encode(contents.encode('utf-8')).decode(),
         'branch':branch}
    prior=session.get(GITHUB_API+'/contents/'+path,params={'ref':branch},timeout=(30,180))
    if prior.status_code==200:obj['sha']=prior.json()['sha']
    elif prior.status_code!=404:raise RuntimeError('GitHub contents lookup HTTP '+str(prior.status_code))
    result=_gh_json(session,'PUT',GITHUB_API+'/contents/'+path,json=obj)
    return result['commit']['sha']

def _release_asset_bytes(session,asset_id,destination):
    url=GITHUB_API+'/releases/assets/'+str(asset_id)
    response=session.get(url,headers={'Accept':'application/octet-stream'},
                         timeout=(30,900),stream=True)
    response.raise_for_status()
    if 'application/json' in response.headers.get('Content-Type','').lower():
        response.close()
        asset=_gh_json(session,'GET',url)
        download=asset['browser_download_url']
        response=session.get(download,headers={'Accept':'application/octet-stream'},
                             timeout=(30,900),stream=True)
        response.raise_for_status()
        if 'json' in response.headers.get('Content-Type','').lower():
            response.close()
            raise RuntimeError('Release API returned JSON rather than artifact bytes')
    with open(destination,'wb') as f:
        for chunk in response.iter_content(4*1024**2):
            if chunk:f.write(chunk)
    response.close()

def _publish_release_file(session,upload_endpoint,asset_name,file_path):
    path=Path(file_path)
    if path.stat().st_size>1_900_000_000:
        raise ValueError('Release asset exceeds safe 1.9GB; split required before publication')
    with path.open('rb') as inp:
        response=session.post(upload_endpoint,params={'name':asset_name},
             data=inp,headers={'Content-Type':'application/octet-stream',
                               'Content-Length':str(path.stat().st_size)},
             timeout=(60,1200))
    if response.status_code not in (200,201):
        raise RuntimeError(f'GitHub Release asset HTTP {response.status_code}')
    return response.json()



def publish_ml1_v4(output_dir,branch=PUBLISH_BRANCH,tag_prefix=RELEASE_TAG_PREFIX):
    """Publish after local verification; verify every downloaded output and Git blob."""
    root=Path(output_dir);local=verify_local(root)
    manifest=json.loads((root/'run_manifest.json').read_text(encoding='utf-8'))
    source=root/'ML1_MIGRATION_V4_SOURCE.ipynb'
    if not source.is_file():raise FileNotFoundError('Complete V4 source not found')
    import nbformat
    nbformat.validate(nbformat.read(source,as_version=4))
    session=github_session(token_from_colab());ensure_publication_branch(session,branch)
    date=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    tag=f'{tag_prefix}-{date}-{manifest["workplan_sha256"][:8]}'
    # Only 3 small Git files; original private Parquet remain untouched.
    report=(root/'REPORT.md').read_text(encoding='utf-8')
    md=(f'# ML-1 V4 — выпущенные материалы\n\nСтатус вычислений: {manifest["status"]}.\n'
        f'Исходные базы не перевыпускаются. Научная приёмка независимая.\n\n'
        f'Release: https://github.com/{REPO}/releases/tag/{tag}\n\n'+
        '\n'.join(f'- {name}: SHA-256 {v["sha256"]}' for name,v in manifest['outputs'].items()))
    public_manifest={k:v for k,v in manifest.items() if k not in ('completed_jobs','failed_jobs','limitations')}
    public_manifest['publication']={'status':'PREPARED','tag':tag,'branch':branch}
    commit_inputs=[('ML1_MIGRATION.ipynb',source.read_text(encoding='utf-8')),
                   ('REPORT.md',report+'\n\n'+md),
                   ('run_manifest.json',json.dumps(public_manifest,ensure_ascii=False,default=str,indent=2))]
    prefix='stage2/ml01_migration/'
    for name,contents in commit_inputs:
        put_small_git_file(session,branch,prefix+name,contents,'ML-1 V4 verified '+name)
    head=_gh_json(session,'GET',GITHUB_API+'/git/ref/heads/'+branch)['object']['sha']
    for name,contents in commit_inputs:
        remote=github_get(session,GITHUB_API+'/contents/'+prefix+name,params={'ref':branch}).json()
        if hashlib.sha256(base64.b64decode(remote['content'])).digest()!=hashlib.sha256(contents.encode('utf-8')).digest():
            raise ValueError('GitHub source SHA mismatch: '+name)
    release=_gh_json(session,'POST',GITHUB_API+'/releases',json={
        'tag_name':tag,'target_commitish':head,'name':'ML-1 V4 · '+date,
        'body':md,'draft':True,'prerelease':False})
    assets=[]
    for name in V4_OUTPUT_FILES:
        obj=_publish_release_file(session,release['upload_url'].split('{',1)[0],name,root/name)
        with tempfile.TemporaryDirectory() as tmp:
            restored=Path(tmp)/name
            _release_asset_bytes(session,obj['id'],restored)
            if sha256_file(restored)!=manifest['outputs'][name]['sha256']:
                raise ValueError('Release asset SHA mismatch: '+name)
            if name.endswith('.xlsx'):
                from openpyxl import load_workbook
                book=load_workbook(restored,read_only=True)
                assert book.sheetnames;book.close()
            else:
                with zipfile.ZipFile(restored) as z:assert z.testzip() is None
        assets.append({'name':name,'size':obj.get('size'),'sha256':manifest['outputs'][name]['sha256']})
    try:
        _gh_json(session,'POST',GITHUB_API+'/git/refs',json={'ref':'refs/tags/'+tag,'sha':head})
    except RuntimeError as e:
        if 'HTTP 422' not in str(e):raise
        _gh_json(session,'PATCH',GITHUB_API+'/git/refs/tags/'+tag,json={'sha':head,'force':True})
    if _gh_json(session,'GET',GITHUB_API+'/git/ref/tags/'+tag)['object']['sha']!=head:
        raise ValueError('Release tag points to wrong commit')
    published=_gh_json(session,'PATCH',GITHUB_API+'/releases/'+str(release['id']),json={'draft':False})
    if published.get('draft',True):raise ValueError('Release still draft')
    if _gh_json(session,'GET',GITHUB_API+'/git/ref/heads/'+branch)['object']['sha']!=head:
        raise ValueError('Release branch changed after pinning')
    return {'status':'PASS_PUBLISHED_VERIFIED','branch':branch,'tag':tag,
            'release_url':published.get('html_url',f'https://github.com/{REPO}/releases/tag/{tag}'),
            'head_sha':head,'assets':assets,'remote_source_verified':True}


In [ ]:
# V4 | проверки публикационного контракта без обращений к GitHub
import inspect
assert set(V4_OUTPUT_FILES)=={'ML1_V4_RESULTS.xlsx','ML1_V4_PREDICTIONS.xlsx',
    'ML1_V4_DATA_QA.xlsx','ML1_V4_MODEL_QA.xlsx','ML1_V4_FIGURES.zip'}
assert {'verify_local','sha256_file','put_small_git_file','_release_asset_bytes'}.issubset(set(publish_ml1_v4.__code__.co_names))
print('V4 publication contract: PASS | 5 binary assets + source/report/manifest')

## 9. Снимок автономного кода (подготовка GitHub-публикации)

In [ ]:
# V3.3.1: regression test of final Optuna report AND recovered checkpoint, before loading D01.
V331_RECOVERY_QA=run_v331_late_export_preflight()
assert V331_RECOVERY_QA['status']=='PASS'
import inspect as _inspect
try:
    _research_source=_inspect.getsource(run_study_v2)
except (OSError,TypeError):
    _research_source=None  # Some Colab kernels don't retain notebook source in linecache.
if _research_source is not None:
    assert 'if recovery_of_late_export:' in _research_source
    assert 'len(trial_cache) if recovery_of_late_export else len(completed)' in _research_source
    V331_RECOVERY_QA['checks']+=2
write_json(Path(RESULT_DIR)/'V3_3_1_RECOVERY_TESTS.json',V331_RECOVERY_QA)
print('ML-1 V3.3.1 FINAL-EXPORT RECOVERY CONTRACT:',V331_RECOVERY_QA['checks'],'checks PASS')

V333_RESUME_QA=run_v333_resume_tests()
assert V333_RESUME_QA['status']=='PASS'
write_json(Path(RESULT_DIR)/'V3_3_3_RESUME_TESTS.json',V333_RESUME_QA)
print('ML-1 V3.3.3 AUTO / STRICT RESUME:',V333_RESUME_QA['tests'],'checks PASS')

# Cross-runtime ZIP extraction is not performed unless RECOVERY_BACKUP_ZIP is explicitly provided.


In [ ]:
# V4.4 | предварительная проверка экспорта, научных целей, XLSX, рисунков, SHA и счётчиков
import tempfile, ast, inspect
import numpy as np, pandas as pd
from pathlib import Path

def run_v4_output_preflight():
    tests=[]
    def run(name,fn):
        try:fn();tests.append({'test':name,'status':'PASS'})
        except Exception as e:raise RuntimeError('V4 PREFLIGHT FAILURE '+name+' / '+repr(e)) from e
    def sample():
        with tempfile.TemporaryDirectory() as tmp:
            p=Path(tmp);df=pd.DataFrame({'row_id':['01','02'],'horizon':[1,3],'truth':[2.,-1.],
                 'prediction':[1.2,-.8]})
            data={'Паспорт':pd.DataFrame({'field':['V4'],'value':['test']}),'Метрики':df}
            out=v4_export_bundle(p,data,{'Прогнозы':df},data,data,[], 'synthetic QA')
            assert len(out)==5 and all((p/k).is_file() for k in out)
            v={'outputs':out,'n_prediction_rows':2}
            (p/'run_manifest.json').write_text(json.dumps(v),encoding='utf-8')
            assert verify_local(p)['sha256_verified']
            with zipfile.ZipFile(p/'ML1_V4_FIGURES.zip') as z:assert z.testzip() is None
            assert set(zipfile.ZipFile(assemble_v4_delivery(p)).namelist())==set(V4_OUTPUT_FILES)
    run('4 Excel + 1 figures archive + SHA + readback + delivery',sample)
    def engine_regression():
        import builtins
        df=pd.DataFrame({'id':['001','002'],'value':[1.5,2.5],
                         'text':['=1+1','-danger']})
        with tempfile.TemporaryDirectory() as tmp:
            one=Path(tmp)/'engine_a.xlsx';two=Path(tmp)/'engine_b.xlsx'
            report_a=v4_workbook(one,{'Экспорт':df})
            report_b=v4_workbook_openpyxl(two,{'Экспорт':df})
            from openpyxl import load_workbook
            for p,report in ((one,report_a),(two,report_b)):
                wb=load_workbook(p,read_only=True,data_only=False)
                try:
                    assert v4_excel_actual_rows(wb['Экспорт'],3)==3
                    rows=list(wb['Экспорт'].values)
                    assert rows[1][0]=='001' and rows[1][2]=='=1+1'
                    assert wb['Экспорт']['C2'].data_type!='f', 'Excel formula injection'
                finally:wb.close()
                assert sha256_file(p)==report['sha256']
            streamed=load_workbook(two,read_only=True,data_only=False)
            try:
                assert streamed['Экспорт'].max_row in (None,3)
                assert v4_excel_actual_rows(streamed['Экспорт'],3)==3
            finally:streamed.close()
    run('XLSX two engines + write-only fallback + formula canary',engine_regression)
    def source_goals():
        import dis
        names=set(run_study_v2.__code__.co_names)
        for k in ('bridge_recursive_factors','build_optuna_report_v331','v4_goals_matrix','v4_export_bundle','verify_local'):
            assert k in names,k
        writes=[x for x in dis.get_instructions(run_study_v2) if x.opname.startswith('STORE_') and x.argval=='counts']
        assert len(writes)==1, 'counter variable shadowing'
        assert 'V4 counter shadowing guard FAILED before VERIFY' in run_study_v2.__code__.co_consts
    run('H1-H5 & Optuna & 50 seeds & XAI & no counter shadowing',source_goals)
    def v4_only_preflight():
        import types
        parent=run_preflight_tests_v4.__code__
        child=next(c for c in parent.co_consts if isinstance(c,types.CodeType) and c.co_name=='test_end_to_end')
        assert {'v4_export_bundle','assemble_v4_delivery','verify_local'}.issubset(set(child.co_names))
        assert set(V4_OUTPUT_FILES)=={'ML1_V4_RESULTS.xlsx','ML1_V4_PREDICTIONS.xlsx','ML1_V4_DATA_QA.xlsx','ML1_V4_MODEL_QA.xlsx','ML1_V4_FIGURES.zip'}
        assert 'ML1_MIGRATION_RESULTS.xlsx' not in set(child.co_names)
    run('Preflight V4 five-file schema and E2E exporter calls',v4_only_preflight)

    def no_false_eligibility():
        sample=pd.DataFrame({'row_id':['a'],'horizon':[1],'protocol':['ROLLING_TIME'],
            'max_feature_year':[2019],'information_cutoff':[2018],'max_label_year_fit':[2018]})
        ctx={(1,'ROLLING_TIME'):1}
        r=v4_goals_matrix(pd.DataFrame(),pd.DataFrame(),pd.DataFrame(),[],[],[],[(1,'ROLLING_TIME')],[],sample,ctx)
        assert r.loc[r['Цель']=='Прогноз без утечки','Статус'].iloc[0]=='FAIL'
        assert r.loc[r['Цель']=='H3','Статус'].iloc[0]=='NOT_FEASIBLE'
    run('Leakage catches future and H3 cannot be invented',no_false_eligibility)
    def publisher_mock():
        import base64 as _b64, shutil, types
        with tempfile.TemporaryDirectory() as tmp:
            p=Path(tmp)
            simple={'Паспорт':pd.DataFrame({'x':[1,2],'y':['a','b']})}
            outs=v4_export_bundle(p,simple,simple,simple,simple,[], 'sample')
            (p/'run_manifest.json').write_text(json.dumps({'outputs':outs,'n_prediction_rows':2,
                'workplan_sha256':'abcddcba0123456789','status':'FULL_LOCAL_COMPUTE_COMPLETE'}))
            (p/'REPORT.md').write_text('Научный mock, не реальные данные',encoding='utf-8')
            import nbformat as _nf
            _nf.write(_nf.v4.new_notebook(cells=[_nf.v4.new_markdown_cell('V4 fixture')]),
                p/'ML1_MIGRATION_V4_SOURCE.ipynb')
            saved={}; g=globals();state={'files':{},'assets':{},'head':'abcdef012345','tag_sha':None}
            def fake_put(sess,branch,path,contents,message):state['files'][path]=contents;return state['head']
            def fake_get(sess,url,params=None):
                path=url.split('/contents/',1)[1];value=state['files'][path]
                return types.SimpleNamespace(json=lambda:{'content':_b64.b64encode(value.encode()).decode()})
            def fake_api(sess,method,url,**kwargs):
                if '/git/ref/heads/' in url:return {'object':{'sha':state['head']}}
                if method=='POST' and '/releases' in url:return {'id':1,'upload_url':'https://mock.upload/{?name}'}
                if '/git/refs/tags/' in url or '/git/ref/tags/' in url:
                    if method in ('POST','PATCH'):state['tag_sha']=kwargs['json']['sha']
                    return {'object':{'sha':state['tag_sha']}}
                if method=='POST' and '/git/refs' in url:
                    state['tag_sha']=kwargs['json']['sha'];return {'object':{'sha':state['tag_sha']}}
                if method=='PATCH' and '/releases/' in url:return {'draft':False,'html_url':'https://github.com/mock/release'}
                raise AssertionError('Unknown mock GitHub request: '+method+' '+url)
            def fake_upload(sess,endpoint,name,path):state['assets'][name]=Path(path).read_bytes();return {'id':name,'size':len(state['assets'][name])}
            def fake_download(sess,asset_id,path):Path(path).write_bytes(state['assets'][asset_id])
            swaps={'github_session':lambda token:object(),'token_from_colab':lambda:'MOCK',
                   'ensure_publication_branch':lambda session,branch:state['head'],
                   'put_small_git_file':fake_put,'github_get':fake_get,
                   '_gh_json':fake_api,'_publish_release_file':fake_upload,
                   '_release_asset_bytes':fake_download}
            try:
                for key,v in swaps.items():saved[key]=g[key];g[key]=v
                result=publish_ml1_v4(p)
                assert result['status']=='PASS_PUBLISHED_VERIFIED'
                assert len(result['assets'])==5 and len(state['files'])==3
                assert state['tag_sha']==state['head']
            finally:
                for key,v in saved.items():g[key]=v
    run('GitHub mock: 5 assets + 3 code files + tag + SHA',publisher_mock)
    result={'version':'V4','status':'PASS','tests':tests}
    write_json(Path(RESULT_DIR)/'V4_OUTPUT_PREFLIGHT.json',result)
    return result
V4_OUTPUT_QA=run_v4_output_preflight()
print('V4 integrated preflight:',len(V4_OUTPUT_QA['tests']),'groups PASS')

In [ ]:
# V4 | полный исходный ноутбук для GitHub, включая эту ячейку
import nbformat,base64,zlib
_SELF_TEMPLATE = "# V4 | полный исходный ноутбук для GitHub, включая эту ячейку\nimport nbformat,base64,zlib\n_SELF_TEMPLATE = <<SELF>>\n_NOTEBOOK_PAYLOAD_B64 = <<PAYLOAD>>\n_nb_src=nbformat.reads(zlib.decompress(base64.b64decode(_NOTEBOOK_PAYLOAD_B64)).decode('utf-8'),as_version=4)\n_src_cell=nbformat.v4.new_code_cell(_SELF_TEMPLATE.replace('<<PAYLOAD>>',repr(_NOTEBOOK_PAYLOAD_B64),1).replace('<<SELF>>',repr(_SELF_TEMPLATE),1))\n_src_cell.id='ml1-v4-self-snapshot'\n_nb_src.cells.insert(34,_src_cell)\nnbformat.validate(_nb_src)\n_src_out=Path(RESULT_DIR)/'ML1_MIGRATION_V4_SOURCE.ipynb';_src_out.parent.mkdir(parents=True,exist_ok=True)\nnbformat.write(_nb_src,_src_out)\nassert len(_nb_src.cells)==39\nprint('V4 full reproducible source:',len(_nb_src.cells),'cells; SHA',sha256_file(_src_out)[:16])\n"
_NOTEBOOK_PAYLOAD_B64 = '...'
_nb_src=nbformat.reads(zlib.decompress(base64.b64decode(_NOTEBOOK_PAYLOAD_B64)).decode('utf-8'),as_version=4)
_src_cell=nbformat.v4.new_code_cell(_SELF_TEMPLATE.replace('<<PAYLOAD>>',repr(_NOTEBOOK_PAYLOAD_B64),1).replace('<<SELF>>',repr(_SELF_TEMPLATE),1))
_src_cell.id='ml1-v4-self-snapshot'
_nb_src.cells.insert(34,_src_cell)
nbformat.validate(_nb_src)
_src_out=Path(RESULT_DIR)/'ML1_MIGRATION_V4_SOURCE.ipynb';_src_out.parent.mkdir(parents=True,exist_ok=True)
nbformat.write(_nb_src,_src_out)
assert len(_nb_src.cells)==39
print('V4 full reproducible source:',len(_nb_src.cells),'cells; SHA',sha256_file(_src_out)[:16])


## 10. Исполнение FULL, публикация, автоматическая загрузка результата

In [ ]:

# V4.3: AUTO проверяет checkpoint по SHA и плану; при отсутствии объявляет FULL
if RECOVERY_BACKUP_ZIP:
    print('Importing user-supplied recovery checkpoint ZIP:',ml1_import_checkpoint_backup(RECOVERY_BACKUP_ZIP,RESULT_DIR))
print(f'Requested mode={MODE}; results={RESULT_DIR}; GitHub publishing={PUBLISH_GITHUB}')
RESULT=run_study_v2(mode=MODE,output_dir=RESULT_DIR,cache_dir=CACHE_DIR,
                    compute_profile=COMPUTE_PROFILE,overrides=INDICATOR_OVERRIDES,
                    enable_spatial=ENABLE_SPATIAL)
if MODE in ('AUTO','FULL','RESUME'):
    VERIFIED=verify_local(RESULT_DIR)
    assert VERIFIED['sha256_verified']
    print('FULL вычисления завершены; независимая локальная проверка:',VERIFIED)
    saved_manifest=Path(RESULT_DIR)/'run_manifest.json'
    journal=json.loads(saved_manifest.read_text(encoding='utf-8'))
    journal['preflight_tests']={'status':'PASS','groups':len(PRE_FLIGHT_RESULTS),'smoke_checks':13,'recursive_factor_tests':len(V32_RECURSIVE_TESTS),'recursive_estimator_tests':len(V32_ESTIMATOR_TESTS),
                                 'acceleration_tests':len(V33_ACCELERATION_TESTS),'resume_safety_tests':V333_RESUME_QA['tests']}
    write_json(saved_manifest,journal)
    if PUBLISH_GITHUB:
        try:
            required_ledgers=ml1_required_publication_ledgers()
            missing_ledgers=[name for _,name in required_ledgers if not (Path(RESULT_DIR)/name).is_file()]
            if missing_ledgers:
                raise FileNotFoundError('Scientific publication preflight requires: '+repr(missing_ledgers))
            PUBLICATION=publish_ml1_v4(RESULT_DIR)
            print('GitHub:',PUBLICATION['status'],PUBLICATION['release_url'])
        except Exception as exc:
            # Never fake published data; retain computed local products for the user.
            PUBLICATION={'status':'PUBLISH_INCOMPLETE','error_type':type(exc).__name__,
                         'explanation':str(exc)[:300]}
            mf=Path(RESULT_DIR)/'run_manifest.json'
            document=json.loads(mf.read_text(encoding='utf-8'))
            document['publication']=PUBLICATION
            write_json(mf,document)
            print('GitHub PUBLISH_INCOMPLETE; локальные данные сохранены:',PUBLICATION)
    DOWNLOAD_ARCHIVE=assemble_v4_delivery(RESULT_DIR)
    print('Архив:',DOWNLOAD_ARCHIVE, 'SHA256:',sha256_file(DOWNLOAD_ARCHIVE))
    if AUTO_DOWNLOAD:
        try:
            from google.colab import files
            files.download(str(DOWNLOAD_ARCHIVE))
            _REVIEW_ARCHIVE_DOWNLOADED=True
        except ImportError:
            print('Файл можно скачать вручную из каталога:',DOWNLOAD_ARCHIVE)
else:
    print(RESULT)


## 11. Повторное скачивание/публикация без обучения

In [ ]:

# ОТДЕЛЬНО: повторить публикацию / скачать архив БЕЗ переобучения моделей.
# Ячейка полезна, если доступ к GitHub был восстановлен после полного расчёта.
from pathlib import Path
if (Path(RESULT_DIR)/'run_manifest.json').is_file():
    print('Проверка локальных результатов:',verify_local(RESULT_DIR)['status'])
    manifest=json.loads((Path(RESULT_DIR)/'run_manifest.json').read_text(encoding='utf-8'))
    if RETRY_PUBLISH and PUBLISH_GITHUB and manifest.get('publication',{}).get('status')!='PASS_PUBLISHED_VERIFIED':
        pub=publish_ml1_v4(RESULT_DIR)
        print('GitHub:',pub['release_url'])
    location=assemble_v4_delivery(RESULT_DIR)
    print('Готовый архив:',location, 'SHA256:',sha256_file(location))
    if AUTO_DOWNLOAD and not globals().get('_REVIEW_ARCHIVE_DOWNLOADED',False):
        from google.colab import files
        files.download(str(location))
else:
    print('Исследовательские результаты не сформированы: выполнить ячейку полного эксперимента.')
